In [2]:
# =============================================================================
# CELL 1 — Environment, reproducibility, configuration
# Run in a FRESH kernel: the thread settings must be applied before numpy loads.
# =============================================================================
import os
# One BLAS/OpenMP thread per process: parallelism comes from running many
# pipelines at once, and single-threaded maths keeps results reproducible.
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = "1"

import sys
import json
import time
import pickle
import hashlib
import platform
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import shap
import ConfigSpace
import autosklearn
from joblib import Parallel, delayed
from joblib.externals.loky import get_reusable_executor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# -----------------------------------------------------------------------------
# Versions (fail loudly rather than silently mis-measure)
# -----------------------------------------------------------------------------
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__, "auto-sklearn": autosklearn.__version__,
            "ConfigSpace": ConfigSpace.__version__, "shap": shap.__version__,
            "platform": platform.platform()}
for k, v in VERSIONS.items():
    print(f"{k:>13}: {v}")

assert (3, 7) <= sys.version_info[:2] <= (3, 10), "auto-sklearn 0.15 supports Python 3.7–3.10"
assert autosklearn.__version__.startswith("0.15"), "expected auto-sklearn 0.15.x"
assert sklearn.__version__.startswith("0.24"), "auto-sklearn 0.15 pins scikit-learn 0.24.x"
assert hasattr(shap, "PermutationExplainer") and hasattr(shap.maskers, "Independent"), \
    "item 1 needs shap.PermutationExplainer and shap.maskers.Independent"

# -----------------------------------------------------------------------------
# Configuration (frozen)
# -----------------------------------------------------------------------------
@dataclass(frozen=True)
class Config:
    dataset_ids: tuple = (37, 31, 44, 4534, 1461)
    master_seed: int = 20260923          # unchanged: saved Phase-2 results depend on it

    # Original study's split (rebuilt and verified in Cell 2)
    orig_test_size: float = 0.20
    orig_random_state: int = 42

    # Item 1: fixed background B (train rows, class-stratified) and explanation set E
    n_background: int = 100
    n_explain: int = 200                  # or every test row if the test set is smaller
    n_permutations: int = 10              # max_evals = n_permutations * (2d + 1), d = raw features
    pred_chunk: int = 2048                # predict in chunks (row-wise, identical results)

    # Item 4: replicate levels
    n_train_seeds: int = 3
    n_explainer_reps: int = 5

    # Item 2: distances
    topk: tuple = (3, 5)
    rbo_p: float = 0.9

    def max_evals(self, n_features: int) -> int:
        return self.n_permutations * (2 * n_features + 1)


CFG = Config()


def derive_seed(*keys) -> int:
    """Named, order-independent seeds (identical to the previous notebook)."""
    payload = json.dumps([CFG.master_seed, *map(str, keys)]).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little") % (2**31 - 1)

np.random.seed(derive_seed("global-guard"))

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
WORKDIR       = Path.cwd()
DATA_DIR      = WORKDIR / "datasets"
ORIG_FILTERED = WORKDIR / "results" / "filtered_pipelines"   # original study's filtered pipelines
PHASE2_CACHE  = WORKDIR / "results" / "phase1"               # saved factorial results (Phase 2)
OUT           = WORKDIR / "results" / "clean"                 # everything this notebook writes
DIRS = {name: OUT / name for name in ("data", "shap", "metrics", "figures", "tmp")}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

for p, what in ((DATA_DIR, "datasets"), (ORIG_FILTERED, "original filtered pipelines"),
                (PHASE2_CACHE, "saved Phase-2 results")):
    assert p.exists(), f"missing {what}: {p}"
    print(f"found {what}: {p}")

# -----------------------------------------------------------------------------
# Machine helpers
# -----------------------------------------------------------------------------
def mem_available_gb() -> float:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemAvailable:"):
                return int(line.split()[1]) / 1024**2
    raise RuntimeError("cannot read /proc/meminfo")


def workers_for(mem_limit_mb: int) -> int:
    """Worker count that keeps total memory well inside what is available."""
    return max(1, min((os.cpu_count() or 2) - 1,
                      int(0.8 * mem_available_gb() / (mem_limit_mb / 1024 + 0.5))))


def fresh_pool():
    """Shut down joblib's reusable pool before a new parallel run (avoids the resize hang)."""
    get_reusable_executor().shutdown(wait=True)

print(f"\ncpu_count = {os.cpu_count()}, available RAM = {mem_available_gb():.0f} GB")
print(f"seed check: derive_seed('split', 37) = {derive_seed('split', 37)}  (old notebook: same value)")

with open(OUT / "environment.json", "w") as f:
    json.dump({"versions": VERSIONS, "config": asdict(CFG)}, f, indent=2)
print(f"\nwriting to {OUT}")

       python: 3.8.20
        numpy: 1.22.4
       pandas: 2.0.3
        scipy: 1.7.3
 scikit-learn: 0.24.2
 auto-sklearn: 0.15.0
  ConfigSpace: 0.4.21
         shap: 0.41.0
     platform: Linux-6.8.0-1061-nvidia-x86_64-with-glibc2.35
found datasets: /home/dbanerjee/example_for_debjeet/datasets
found original filtered pipelines: /home/dbanerjee/example_for_debjeet/results/filtered_pipelines
found saved Phase-2 results: /home/dbanerjee/example_for_debjeet/results/phase1

cpu_count = 384, available RAM = 1427 GB
seed check: derive_seed('split', 37) = 105697778  (old notebook: same value)

writing to /home/dbanerjee/example_for_debjeet/results/clean


In [6]:
# =============================================================================
# CELL 2 — Raw data, the original study's split and encoding, and verification
#          of every filtered pipeline against its recorded accuracy
# =============================================================================
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

CLASSIFIER_NAMES = ["adaboost", "bernoulli_nb", "decision_tree", "extra_trees", "gaussian_nb",
                    "gradient_boosting", "k_nearest_neighbors", "lda", "liblinear_svc",
                    "libsvm_svc", "mlp", "multinomial_nb", "passive_aggressive", "qda",
                    "random_forest", "sgd"]
# Event class used by the saved Phase-2 results (unchanged from the previous notebook)
EVENT_LABEL = {37: "tested_positive", 31: "bad", 44: 1, 4534: -1, 1461: 2}
DATASET_NAME = {37: "diabetes", 31: "credit-g", 44: "spambase",
                4534: "PhishingWebsites", 1461: "bank-marketing"}

# -----------------------------------------------------------------------------
# 1. Raw data (categorical columns -> pandas category codes, as the original code)
# -----------------------------------------------------------------------------
@dataclass
class Data:
    did: int
    name: str
    X: np.ndarray            # raw features: numeric as float, categoricals as category codes
    feature_names: list
    cat_mask: np.ndarray
    y_orig: np.ndarray       # original study's target codes (what the filtered models predict)
    y_event: np.ndarray      # event coding used by the saved Phase-2 results
    orig_labels: list        # original code i -> label

    @property
    def d(self): return self.X.shape[1]

    @property
    def n(self): return self.X.shape[0]


def load_raw(did) -> Data:
    folder = DATA_DIR / str(did)
    Xdf = pd.read_csv(folder / "features.csv")
    ydf = pd.read_csv(folder / "targets.csv")
    with open(folder / "categorical_indicator.json") as f:
        cat_ind = json.load(f)
    assert isinstance(cat_ind, dict) and set(cat_ind) == set(Xdf.columns)
    assert not Xdf.isna().any().any(), "missing values: original code drops rows; handle explicitly"
    cols = list(Xdf.columns)
    cat_mask = np.array([cat_ind[c] is True for c in cols])
    X = np.empty(Xdf.shape)
    for j, c in enumerate(cols):
        X[:, j] = (Xdf[c].astype("category").cat.codes.to_numpy() if cat_mask[j]
                   else pd.to_numeric(Xdf[c]).to_numpy())
    t = ydf.iloc[:, 0]
    return Data(did=did, name=DATASET_NAME[did], X=X.astype(np.float64), feature_names=cols,
                cat_mask=cat_mask,
                y_orig=t.astype("category").cat.codes.to_numpy().astype(np.int8),
                y_event=(t == EVENT_LABEL[did]).to_numpy().astype(np.int8),
                orig_labels=list(t.astype("category").cat.categories))


DATA = {did: load_raw(did) for did in CFG.dataset_ids}

# -----------------------------------------------------------------------------
# 2. Original split and encoding (exact replica of prepare_data_for_SHAP_importance)
# -----------------------------------------------------------------------------
@dataclass
class Original:
    did: int
    train_rows: np.ndarray
    test_rows: np.ndarray
    encoder: object           # OneHotEncoder fitted as in the original code (None if no categoricals)
    cat_idx: list             # raw column indices of the categorical block, in original order
    num_idx: list             # raw column indices of the numeric block, in original order
    encoded_columns: list     # the original models' input column names (their feat_type keys)


def encode_original(o: Original, X_raw: np.ndarray) -> np.ndarray:
    """Raw features -> the original one-hot layout [categorical one-hot block, numeric block]."""
    parts = []
    if o.encoder is not None:
        parts.append(o.encoder.transform(X_raw[:, o.cat_idx].astype(np.int64)).toarray())
    parts.append(X_raw[:, o.num_idx])
    return np.hstack(parts).astype(np.float64)


def model_input(o: Original, X_raw: np.ndarray) -> pd.DataFrame:
    """Exactly what the original models take: the one-hot layout as a DataFrame with
    the original encoded column names (their feat_type is keyed by column name).
    This step is part of each pipeline (item 1: raw features in, preprocessing inside)."""
    return pd.DataFrame(encode_original(o, X_raw), columns=o.encoded_columns)


def build_original(did) -> Original:
    folder = DATA_DIR / str(did)
    X = pd.read_csv(folder / "features.csv")
    y = pd.read_csv(folder / "targets.csv")
    with open(folder / "categorical_indicator.json") as f:
        cat_ind = json.load(f)
    cat_cols = [k for k, v in cat_ind.items() if v is True]
    cont_cols = [k for k, v in cat_ind.items() if v is not True]
    for c in cat_cols:
        X[c] = X[c].astype("category").cat.codes
    tcol = y.columns[0]
    y[tcol] = y[tcol].astype("category").cat.codes
    X_cat = X.drop(cont_cols, axis=1)
    if X_cat.shape[1] > 0:
        enc = OneHotEncoder().fit(X_cat)
        X_enc = pd.DataFrame(enc.transform(X_cat).toarray(),
                             columns=enc.get_feature_names(X_cat.columns))
        X_for = pd.concat([X_enc, X.drop(cat_cols, axis=1)], axis=1)
    else:
        enc, X_for = None, X
    y = np.ravel(y)
    X_tr, X_te, _, _ = train_test_split(X_for, y, test_size=CFG.orig_test_size,
                                        random_state=CFG.orig_random_state)
    names = DATA[did].feature_names
    o = Original(did=did, train_rows=X_tr.index.to_numpy(), test_rows=X_te.index.to_numpy(),
                 encoder=enc, cat_idx=[names.index(c) for c in X_cat.columns],
                 num_idx=[names.index(c) for c in X.drop(cat_cols, axis=1).columns],
                 encoded_columns=list(X_for.columns))
    # model_input must reproduce the original encoded matrix exactly: values AND column names
    rebuilt = model_input(o, DATA[did].X)
    assert list(rebuilt.columns) == list(X_for.columns), f"{did}: column names differ"
    assert np.array_equal(rebuilt.to_numpy(), X_for.to_numpy(dtype=np.float64)), \
        f"{did}: encoding does not reproduce the original values"
    return o


ORIG = {did: build_original(did) for did in CFG.dataset_ids}
print("Original split and encoding (rebuilt and checked row for row, names and values):")
for did, o in ORIG.items():
    D = DATA[did]
    print(f"  {did:>5} {D.name:<17} train {len(o.train_rows):>6}  test {len(o.test_rows):>6}  "
          f"encoded width {len(o.encoded_columns):>3}  label codes {dict(enumerate(D.orig_labels))}")

# -----------------------------------------------------------------------------
# 3. Seed-function continuity: the old notebook's split must come out identical
# -----------------------------------------------------------------------------
for did in CFG.dataset_ids:
    old = np.load(PHASE2_CACHE / "data" / f"rowsets_{did}.npz", allow_pickle=True)
    tr, _ = train_test_split(np.arange(DATA[did].n), test_size=0.30, stratify=DATA[did].y_event,
                             random_state=derive_seed("split", did))
    assert np.array_equal(np.sort(tr), old["train_idx"]), f"{did}: seed function changed"
print("Seed continuity: old notebook's splits reproduced exactly -> saved Phase-2 results valid")

# -----------------------------------------------------------------------------
# 4. Verify every filtered model: load, predict the rebuilt test set, compare accuracy
# -----------------------------------------------------------------------------
VERIFY_COLS = ["path", "status", "type", "acc", "proba_shape",
               "cfg_classifier", "cfg_preprocessor", "cfg_balancing", "cfg_rescaling"]


def parse_cell(cell):
    clf = max((c for c in CLASSIFIER_NAMES if cell.startswith(c + "_")), key=len, default=None)
    return (clf, cell[len(clf) + 1:]) if clf else (None, cell)


def _verify_model(path, X_test, y_test):
    import pickle, warnings
    import numpy as np
    warnings.filterwarnings("ignore")
    try:
        with open(path, "rb") as f:
            m = pickle.load(f)
        pred = np.asarray(m.predict(X_test)).ravel()
        proba = m.predict_proba(X_test)
        out = dict(path=path, status="ok", type=type(m).__name__,
                   acc=float((pred == y_test).mean()), proba_shape=str(proba.shape))
        if hasattr(m, "config"):
            cfg = m.config.get_dictionary()
            out.update(cfg_classifier=cfg.get("classifier:__choice__"),
                       cfg_preprocessor=cfg.get("feature_preprocessor:__choice__"),
                       cfg_balancing=cfg.get("balancing:strategy"),
                       cfg_rescaling=next((v for k, v in cfg.items()
                                           if k.endswith("rescaling:__choice__")), None))
        return out
    except Exception as e:
        return dict(path=path, status=f"{type(e).__name__}: {str(e)[:150]}")


FILTERED = {}
fresh_pool()
for did in CFG.dataset_ids:
    with open(ORIG_FILTERED / str(did) / "filtered_pipelines.json") as f:
        F = pd.DataFrame(json.load(f))
    F["cell"] = F.model_path.apply(lambda p: Path(p).parent.name)
    F[["classifier", "preprocessor"]] = F.cell.apply(lambda c: pd.Series(parse_cell(c)))
    F["exists"] = F.model_path.apply(lambda p: (WORKDIR / p).exists())
    o = ORIG[did]
    X_test = model_input(o, DATA[did].X[o.test_rows])
    y_test = DATA[did].y_orig[o.test_rows]
    res = Parallel(n_jobs=min(os.cpu_count() - 1, 200))(
        delayed(_verify_model)(str(WORKDIR / p), X_test, y_test) for p in F.model_path)
    R = pd.DataFrame(res).reindex(columns=VERIFY_COLS)
    F = pd.concat([F.reset_index(drop=True), R.drop(columns="path")], axis=1)
    F["acc_match"] = (F.status == "ok") & (np.abs(F.acc - F.accuracy) < 1e-9)
    F["cfg_agrees"] = (F.cfg_classifier == F.classifier) & (F.cfg_preprocessor == F.preprocessor)
    F["did"] = did
    FILTERED[did] = F

ALLF = pd.concat(FILTERED.values(), ignore_index=True)
print("\nVerification of the original filtered pipelines:")
print(ALLF.groupby("did").agg(n=("model_path", "size"), file_exists=("exists", "sum"),
                              loaded=("status", lambda s: (s == "ok").sum()),
                              accuracy_reproduced=("acc_match", "sum"),
                              config_matches_folder=("cfg_agrees", "sum")).to_string())
print("\nmodel types:", ALLF[ALLF.status == "ok"].type.value_counts().to_dict())

bad = ALLF[~ALLF.acc_match]
if len(bad):
    print(f"\n{len(bad)} models do NOT reproduce their recorded accuracy (first 15):")
    print(bad[["did", "classifier", "preprocessor", "status", "accuracy", "acc"]].head(15).to_string())
    print("\ndistinct failure messages:")
    print(bad.status.value_counts().head(10).to_string())

print("\nFiltered pipelines per classifier x dataset:")
print(ALLF.pivot_table(index="classifier", columns="did", values="model_path",
                       aggfunc="count", fill_value=0).to_string())
print("\nFeature preprocessors x dataset:")
print(ALLF.pivot_table(index="preprocessor", columns="did", values="model_path",
                       aggfunc="count", fill_value=0).to_string())
print("\nBalancing (from each model's own config):")
print(ALLF.pivot_table(index="cfg_balancing", columns="did", values="model_path",
                       aggfunc="count", fill_value=0).to_string())
print("\nRescaling inside the data preprocessor (uncontrolled in the original study, item 6):")
print(ALLF.pivot_table(index="cfg_rescaling", columns="did", values="model_path",
                       aggfunc="count", fill_value=0).to_string())

ALLF.to_csv(DIRS["data"] / "original_filtered_verified.csv", index=False)
with open(DIRS["data"] / "original_split.pkl", "wb") as f:
    pickle.dump({did: dict(train_rows=o.train_rows, test_rows=o.test_rows,
                           cat_idx=o.cat_idx, num_idx=o.num_idx,
                           encoded_columns=o.encoded_columns) for did, o in ORIG.items()}, f)
print(f"\nSaved to {DIRS['data']}")

Original split and encoding (rebuilt and checked row for row, names and values):
     37 diabetes          train    614  test    154  encoded width   8  label codes {0: 'tested_negative', 1: 'tested_positive'}
     31 credit-g          train    800  test    200  encoded width  61  label codes {0: 'bad', 1: 'good'}
     44 spambase          train   3680  test    921  encoded width  57  label codes {0: 0, 1: 1}
   4534 PhishingWebsites  train   8844  test   2211  encoded width  68  label codes {0: -1, 1: 1}
   1461 bank-marketing    train  36168  test   9043  encoded width  51  label codes {0: 1, 1: 2}
Seed continuity: old notebook's splits reproduced exactly -> saved Phase-2 results valid

Verification of the original filtered pipelines:
         n  file_exists  loaded  accuracy_reproduced  config_matches_folder
did                                                                        
31    1608         1608    1608                 1588                   1608
37    8610         8610  

In [4]:
did = CFG.dataset_ids[0]
with open(ORIG_FILTERED / str(did) / "filtered_pipelines.json") as f:
    F0 = pd.DataFrame(json.load(f))
p = WORKDIR / F0.model_path.iloc[0]
print("path:", p, "| exists:", p.exists())
o = ORIG[did]
print(_verify_model(str(p), encode_original(o, DATA[did].X[o.test_rows]), DATA[did].y_orig[o.test_rows]))

path: /home/dbanerjee/example_for_debjeet/results/pipeline_models/37/adaboost_extra_trees_preproc_for_classification/42.1041.0.0.model | exists: True
{'path': '/home/dbanerjee/example_for_debjeet/results/pipeline_models/37/adaboost_extra_trees_preproc_for_classification/42.1041.0.0.model', 'status': 'ValueError: Specifying the columns using strings is only supported for pandas DataFrames'}


In [5]:
# Diagnose the original models' expected input (read-only)
import pickle

def peek(did):
    with open(ORIG_FILTERED / str(did) / "filtered_pipelines.json") as f:
        F0 = pd.DataFrame(json.load(f))
    with open(WORKDIR / F0.model_path.iloc[0], "rb") as f:
        m = pickle.load(f)
    ft = getattr(m, "feat_type", None)
    print(f"\n{did} {DATA[did].name}: {type(m).__name__}, feat_type entries: {None if ft is None else len(ft)}")
    if ft:
        print("  first keys/values:", list(ft.items())[:6])
        print("  value counts:", pd.Series(list(ft.values())).value_counts().to_dict())
    o, D = ORIG[did], DATA[did]
    y = D.y_orig[o.test_rows]
    candidates = {
        "one-hot encoded (SHAP-prep layout)": pd.DataFrame(encode_original(o, D.X[o.test_rows]),
                                                           columns=o.encoded_columns),
        "raw columns, category codes": pd.DataFrame(D.X[o.test_rows], columns=D.feature_names),
    }
    for name, Xc in candidates.items():
        try:
            acc = (np.asarray(m.predict(Xc)).ravel() == y).mean()
            print(f"  {name:<36} accuracy {acc:.4f}   (recorded {F0.accuracy.iloc[0]:.4f})")
        except Exception as e:
            print(f"  {name:<36} {type(e).__name__}: {str(e)[:110]}")

for did in CFG.dataset_ids:
    peek(did)
    


37 diabetes: SimpleClassificationPipeline, feat_type entries: 8
  first keys/values: [('preg', 'numerical'), ('plas', 'numerical'), ('pres', 'numerical'), ('skin', 'numerical'), ('insu', 'numerical'), ('mass', 'numerical')]
  value counts: {'numerical': 8}
  one-hot encoded (SHAP-prep layout)   accuracy 0.7273   (recorded 0.7273)
  raw columns, category codes          accuracy 0.7273   (recorded 0.7273)

31 credit-g: SimpleClassificationPipeline, feat_type entries: 61
  first keys/values: [('checking_status_0', 'numerical'), ('checking_status_1', 'numerical'), ('checking_status_2', 'numerical'), ('checking_status_3', 'numerical'), ('credit_history_0', 'numerical'), ('credit_history_1', 'numerical')]
  value counts: {'numerical': 61}
  one-hot encoded (SHAP-prep layout)   accuracy 0.8000   (recorded 0.8000)
  raw columns, category codes          ValueError: X has 20 features, but ColumnTransformer is expecting 61 features as input.

44 spambase: SimpleClassificationPipeline, feat_type 

In [8]:
# =============================================================================
# CELL 3 — Load the existing SHAP results (option C: reuse, deviations documented)
#   * one importance vector per filtered pipeline, in raw-feature order
#   * test and correct the feature-label order of the original remapping
# =============================================================================
SHAP_DIR = WORKDIR / "results" / "shap_values"

def block_order(did):
    """Order in which the original code assembles the summed SHAP vector:
    [categorical features (original block order), numeric features]."""
    o, names = ORIG[did], DATA[did].feature_names
    return [names[i] for i in o.cat_idx] + [names[i] for i in o.num_idx]


def saved_to_raw(did, pairs, labels_shifted: bool):
    """Saved [(label, value)] -> vector in raw feature order.
    labels_shifted=True: value stored under label names[i] belongs to block feature i."""
    names, saved = DATA[did].feature_names, dict(pairs)
    assert set(saved) == set(names), f"{did}: saved labels differ from feature names"
    m = np.empty(len(names))
    if labels_shifted:
        blk = block_order(did)
        for i, lab in enumerate(names):
            m[names.index(blk[i])] = saved[lab]
    else:
        for lab, v in saved.items():
            m[names.index(lab)] = v
    return m

# -----------------------------------------------------------------------------
# 1. Load every saved result, matched to the filtered pipeline by model path
# -----------------------------------------------------------------------------
t0 = time.time()
SAVED = {}
for did in CFG.dataset_ids:
    recs = {}
    for p in (SHAP_DIR / str(did)).glob("*.json"):
        with open(p) as f:
            j = json.load(f)
        recs.setdefault(j["configPath"], []).append(j["origFeature_abs_mean_shap"])
    SAVED[did] = recs
print(f"loaded saved SHAP files in {time.time() - t0:.0f}s")

rows = []
for did in CFG.dataset_ids:
    F, recs = FILTERED[did], SAVED[did]
    matched = F.model_path.isin(recs)
    rows.append(dict(did=did, pipelines=len(F), with_shap=int(matched.sum()),
                     missing=int((~matched).sum()),
                     duplicates=sum(len(v) > 1 for v in recs.values()),
                     orphan_files=len(set(recs) - set(F.model_path)),
                     order_differs=block_order(did) != DATA[did].feature_names))
print(pd.DataFrame(rows).set_index("did").to_string())

# -----------------------------------------------------------------------------
# 2. Test the label order: recompute one pipeline with the ORIGINAL setup exactly
# -----------------------------------------------------------------------------
def original_kernel_shap(model, did, seed=0):
    """Replica of get_abs_SHAP_feature_importance: KernelExplainer on predict_proba,
    background shap.sample(X_train, 100) [random_state=0], E = shap.sample(X_test, 100,
    random_state=42), nsamples='auto', categorical dummies summed, |.| over both classes.
    Returns the vector in BLOCK order."""
    o, D = ORIG[did], DATA[did]
    Xtr = model_input(o, D.X[o.train_rows])
    Xte = model_input(o, D.X[o.test_rows])
    bg = shap.sample(Xtr, 100)
    E = shap.sample(Xte, 100, random_state=42)
    np.random.seed(seed)
    ex = shap.KernelExplainer(model.predict_proba, bg, keep_index=True, keep_index_ordered=True)
    sv = ex.shap_values(E, nsamples="auto", silent=True)
    n_cat = [len(c) for c in o.encoder.categories_] if o.encoder is not None else []
    bounds = np.cumsum([0] + n_cat)
    per_class = []
    for cs in sv:
        cs = np.asarray(cs)
        summed = [cs[:, bounds[k]:bounds[k + 1]].sum(1) for k in range(len(n_cat))]
        per_class.append(np.column_stack(summed + [cs[:, bounds[-1]:]]) if summed else cs)
    return np.abs(np.vstack(per_class)).sum(0) / len(E)


def shares(v):
    return np.asarray(v, float) / np.sum(v)


LABEL_SHIFT = {}
for did in CFG.dataset_ids:
    if block_order(did) == DATA[did].feature_names:
        LABEL_SHIFT[did] = False                         # orders coincide: nothing to correct
        continue
    F = FILTERED[did]
    cand = F[F.classifier.isin(["lda", "liblinear_svc", "sgd", "gaussian_nb"]) & F.model_path.isin(SAVED[did])]
    r = (cand[cand.preprocessor == "no_preprocessing"] if (cand.preprocessor == "no_preprocessing").any()
         else cand).iloc[0]
    with open(WORKDIR / r.model_path, "rb") as f:
        model = pickle.load(f)
    t1 = time.time()
    new_block = original_kernel_shap(model, did)
    names, blk = DATA[did].feature_names, block_order(did)
    saved = dict(SAVED[did][r.model_path][0])
    as_shifted = np.array([saved[names[i]] for i in range(len(names))])   # hypothesis: labels shifted
    as_labelled = np.array([saved[blk[i]] for i in range(len(names))])    # hypothesis: labels correct
    tv_shift = 0.5 * np.abs(shares(new_block) - shares(as_shifted)).sum()
    tv_label = 0.5 * np.abs(shares(new_block) - shares(as_labelled)).sum()
    LABEL_SHIFT[did] = bool(tv_shift < tv_label)
    print(f"\n{did} {DATA[did].name}: test pipeline {r.classifier} + {r.preprocessor} "
          f"(recomputed in {time.time() - t1:.0f}s)")
    print(f"  TV(recomputed, saved read as SHIFTED)  = {tv_shift:.4f}")
    print(f"  TV(recomputed, saved read as LABELLED) = {tv_label:.4f}")
    print(f"  -> labels are {'SHIFTED (original code mislabels features)' if LABEL_SHIFT[did] else 'correct'}")

# -----------------------------------------------------------------------------
# 3. Build the importance matrix per dataset (raw-feature order, labels corrected)
# -----------------------------------------------------------------------------
POP = {}
for did in CFG.dataset_ids:
    F = FILTERED[did].copy()
    F["has_shap"] = F.model_path.isin(SAVED[did])
    F = F[F.has_shap].reset_index(drop=True)
    M = np.vstack([saved_to_raw(did, SAVED[did][p][0], LABEL_SHIFT[did]) for p in F.model_path])
    assert np.isfinite(M).all() and (M >= 0).all(), f"{did}: invalid importance values"
    F["degenerate"] = M.sum(1) <= 0
    POP[did] = dict(meta=F, M=M)
    np.save(DIRS["data"] / f"orig_M_{did}.npy", M)
    F.to_csv(DIRS["data"] / f"orig_meta_{did}.csv", index=False)
print("\nImportance matrices (pipelines x raw features):",
      {did: P["M"].shape for did, P in POP.items()})
print("degenerate (all-zero) pipelines:", {did: int(P["meta"].degenerate.sum()) for did, P in POP.items()})

# -----------------------------------------------------------------------------
# 4. Deviations of the reused measurement from item 1 (for the report)
# -----------------------------------------------------------------------------
DEVIATIONS = pd.DataFrame([
    ("explainer", "PermutationExplainer or KernelExplainer, same for all", "KernelExplainer for all", "meets"),
    ("output", "P(y=1) via predict_proba", "predict_proba, |phi| summed over both classes (= 2|phi_1|, shares identical)", "meets"),
    ("masker/background", "Independent(B), ~100 train rows, stratified by class",
     "shap.sample(X_train, 100), same rows for all pipelines, NOT stratified", "minor deviation"),
    ("explanation set", "fixed 200-500 held-out rows", "fixed 100 test rows (shap.sample, random_state=42)", "deviation"),
    ("budget", "one max_evals per dataset", "nsamples='auto' (2M+2048) per dataset", "meets"),
    ("features", "raw features end-to-end, categoricals whole",
     "one-hot columns explained separately, then summed (credit-g, Phishing, bank)", "deviation"),
    ("explainer seed", "replicates with controlled seeds (item 4)", "unseeded KernelExplainer draws", "deviation"),
    ("feature labels", "-", f"original code mislabels features when categoricals and numerics interleave; "
     f"corrected here for {[d for d, s in LABEL_SHIFT.items() if s]}", "corrected"),
], columns=["aspect", "item 1 requires", "reused measurement", "status"])
DEVIATIONS.to_csv(DIRS["metrics"] / "item1_deviations_reused_shap.csv", index=False)
print("\n" + DEVIATIONS.to_string(index=False))

loaded saved SHAP files in 6s
      pipelines  with_shap  missing  duplicates  orphan_files  order_differs
did                                                                         
37         8610       8610        0           0             0          False
31         1608       1608        0           0             0           True
44         4307       4307        0           0             0          False
4534       3681       3680        1           0             0          False
1461       6637       6637        0           0             0           True

31 credit-g: test pipeline liblinear_svc + no_preprocessing (recomputed in 73s)
  TV(recomputed, saved read as SHIFTED)  = 0.0367
  TV(recomputed, saved read as LABELLED) = 0.3266
  -> labels are SHIFTED (original code mislabels features)

1461 bank-marketing: test pipeline sgd + no_preprocessing (recomputed in 99s)
  TV(recomputed, saved read as SHIFTED)  = 0.0251
  TV(recomputed, saved read as LABELLED) = 0.4538
  -> labels 

In [9]:
# =============================================================================
# CELL 4 — Measurement library: item 2 (shares, ranks, distances) and
#          item 3 (pairwise D, mean pairwise dispersion), with tests
# =============================================================================
from scipy.spatial.distance import cdist

DIST_NAMES = ["tv"] + [f"jac{k}" for k in CFG.topk] + ["rbo"]

# ---------------------------------------------------------------- item 2 --------
def normalise(m):
    """m_pj (mean |phi|) -> shares s_pj = m_pj / sum_k m_pk, ranking (1 = top).
    Ties broken by feature index. Constant pipelines are flagged, never fixed."""
    m = np.asarray(m, dtype=float)
    tot = m.sum()
    if not np.isfinite(tot) or tot <= 0:
        return dict(shares=np.full_like(m, np.nan), order=None, ranks=None, degenerate=True)
    s = m / tot
    order = np.lexsort((np.arange(len(s)), -s))
    ranks = np.empty(len(s), dtype=int)
    ranks[order] = np.arange(1, len(s) + 1)
    return dict(shares=s, order=order, ranks=ranks, degenerate=False)


def rbo_ext(order_p, order_q, p=CFG.rbo_p):
    """Extrapolated rank-biased overlap (Webber, Moffat & Zobel 2010, eq. 32)."""
    k = len(order_p)
    seen_p, seen_q, X, acc = set(), set(), 0, 0.0
    for d in range(1, k + 1):
        a, b = order_p[d - 1], order_q[d - 1]
        X += 1 if a == b else (a in seen_q) + (b in seen_p)
        seen_p.add(a); seen_q.add(b)
        acc += (X / d) * p ** d
    return (X / k) * p ** k + ((1 - p) / p) * acc


def explanation_distances(a, b):
    """All item-2 distances between two normalised explanations (0 = identical)."""
    out = {"tv": 0.5 * float(np.abs(a["shares"] - b["shares"]).sum())}
    for k in CFG.topk:
        A, B = set(a["order"][:k]), set(b["order"][:k])
        out[f"jac{k}"] = 1.0 - len(A & B) / len(A | B)
    out["rbo"] = 1.0 - rbo_ext(a["order"], b["order"])
    return out

# ---------------------------------------------------------------- item 3 --------
def pairwise_D(S, R, p=CFG.rbo_p):
    """S: shares (n x d), R: ranks (n x d). Vectorised item-2 distances for all pairs."""
    n, d = S.shape
    D = {"tv": 0.5 * cdist(S, S, metric="cityblock")}
    for k in CFG.topk:
        T = (R <= k).astype(np.float64)
        inter = T @ T.T
        D[f"jac{k}"] = 1.0 - inter / (2 * k - inter)
    acc = np.zeros((n, n))
    for depth in range(1, d + 1):
        I = (R <= depth).astype(np.float64)
        acc += (I @ I.T / depth) * p ** depth
    D["rbo"] = 1.0 - (p ** d + ((1 - p) / p) * acc)      # X_d / d = 1 for full, same-item lists
    for key in D:
        np.fill_diagonal(D[key], 0.0)
        np.clip(D[key], 0.0, 1.0, out=D[key])
    return D


def mean_pairwise(Dm, idx=None):
    """D_bar = 2 / (|P|(|P|-1)) * sum_{p<q} d(p,q), optionally on a subset of pipelines."""
    if idx is not None:
        Dm = Dm[np.ix_(idx, idx)]
    iu = np.triu_indices(Dm.shape[0], k=1)
    return float(Dm[iu].mean()) if len(iu[0]) else np.nan

# ---------------------------------------------------------------- tests ---------
rng = np.random.default_rng(derive_seed("library-tests"))
m = rng.random(8)
assert all(abs(v) < 1e-12 for v in explanation_distances(normalise(m), normalise(3.7 * m)).values())
assert all(abs(v) < 1e-12 for v in explanation_distances(normalise(m), normalise(m)).values())
assert np.isclose(explanation_distances(normalise(np.array([1., 1, 0, 0])),
                                        normalise(np.array([0., 0, 1, 1])))["tv"], 1.0)
assert abs(rbo_ext(np.array([0, 1, 2]), np.array([1, 0, 2]), 0.9) - 0.9) < 1e-12   # hand value
assert abs(rbo_ext(np.arange(5), np.arange(5), 0.9) - 1.0) < 1e-12
for d_feat in (8, 16, 20, 30, 57):
    toy = [normalise(rng.gamma(0.5, size=d_feat)) for _ in range(30)]
    D = pairwise_D(np.vstack([t["shares"] for t in toy]), np.vstack([t["ranks"] for t in toy]))
    worst = max(abs(D[k][i, j] - explanation_distances(toy[i], toy[j])[k])
                for i in range(30) for j in range(30) for k in DIST_NAMES)
    assert worst < 1e-12, f"vectorised D differs from definition (d={d_feat}): {worst}"
    n_ = len(toy)
    brute = 2 / (n_ * (n_ - 1)) * sum(D["tv"][i, j] for i in range(n_) for j in range(i + 1, n_))
    assert abs(brute - mean_pairwise(D["tv"])) < 1e-12
    perm = rng.permutation(n_)
    Dp = pairwise_D(np.vstack([toy[i]["shares"] for i in perm]), np.vstack([toy[i]["ranks"] for i in perm]))
    assert all(abs(mean_pairwise(Dp[k]) - mean_pairwise(D[k])) < 1e-12 for k in DIST_NAMES)
print("Library tests passed: scale invariance, identity, bounds, RBO hand value, "
      "vectorised D == definition, D_bar formula, relabelling invariance.")

# ---------------------------------------------------------------- apply ---------
for did, P in POP.items():
    keep = ~P["meta"].degenerate.to_numpy()
    N = [normalise(m) for m in P["M"][keep]]
    P["meta_ok"] = P["meta"][keep].reset_index(drop=True)
    P["S"] = np.vstack([x["shares"] for x in N])
    P["R"] = np.vstack([x["ranks"] for x in N])
    assert np.allclose(P["S"].sum(1), 1.0)
    print(f"  {did:>5} {DATA[did].name:<17} {keep.sum():>5} pipelines normalised "
          f"({(~keep).sum()} degenerate excluded), d = {P['S'].shape[1]}")

Library tests passed: scale invariance, identity, bounds, RBO hand value, vectorised D == definition, D_bar formula, relabelling invariance.
     37 diabetes           8610 pipelines normalised (0 degenerate excluded), d = 8
     31 credit-g           1598 pipelines normalised (10 degenerate excluded), d = 20
     44 spambase           4306 pipelines normalised (1 degenerate excluded), d = 57
   4534 PhishingWebsites   3680 pipelines normalised (0 degenerate excluded), d = 30
   1461 bank-marketing     6627 pipelines normalised (10 degenerate excluded), d = 16


In [10]:
# =============================================================================
# CELL 5 — Replicate jobs for item 4 (noise floor) and item 5 (sanity check),
#          under the REUSED measurement (exact replica of the original
#          KernelExplainer setup). Analysis follows in Cell 6.
# =============================================================================
import importlib
N_SUBSET     = 50            # third compute lever: replicates on ~50 pipelines per dataset
REP_MEM_MB   = 6144
REP_TIME_S   = 4 * 3600
REP_DIR = DIRS["shap"] / "replicates"
REP_DIR.mkdir(parents=True, exist_ok=True)

JOB_SRC = r'''
# phase1c_jobs.py -- refit an original configuration (or its matched RF baseline) with a
# given seed and explain it with the original KernelExplainer setup.
# Run as: python -m phase1c_jobs <args.pkl>
import os, sys, pickle, time, warnings, resource
import numpy as np


def _vmsize():
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmSize:"):
                return int(line.split()[1]) * 1024


class _Baseline:
    """sklearn RF behind the same interface the explainer calls (DataFrame in)."""
    def __init__(self, rf): self.rf = rf
    def predict_proba(self, X): return self.rf.predict_proba(np.asarray(X, dtype=np.float64))
    def predict(self, X): return self.rf.predict(np.asarray(X, dtype=np.float64))


def _refit(a):
    from autosklearn.pipeline.classification import SimpleClassificationPipeline
    with open(a["model_path"], "rb") as f:
        orig = pickle.load(f)
    kw = dict(feat_type=getattr(orig, "feat_type", None),
              dataset_properties=getattr(orig, "dataset_properties", None),
              include=getattr(orig, "include", None), exclude=getattr(orig, "exclude", None),
              random_state=a["seed_train"])
    try:
        pipe = SimpleClassificationPipeline(config=orig.config, **kw)
    except ValueError:
        pipe = SimpleClassificationPipeline(config=orig.config.get_dictionary(), **kw)
    pipe.fit(a["X_train"], a["y_train"])
    if a["kind"] == "asl":
        return pipe
    from sklearn.ensemble import RandomForestClassifier          # item-5 baseline
    est = pipe.named_steps["classifier"].choice.estimator
    params = est.get_params()
    params.update(n_jobs=1, warm_start=False, verbose=0, random_state=a["seed_base"])
    rf = RandomForestClassifier(**params).fit(np.asarray(a["X_train"], dtype=np.float64), a["y_train"])
    return _Baseline(rf)


def _kernel(model, a):
    """Replica of the original get_abs_SHAP_feature_importance; returns BLOCK-order vector."""
    import shap
    chunk = a["chunk"]

    def f(X):                                        # row-wise identical to predict_proba(X)
        n = X.shape[0]
        if n <= chunk:
            return model.predict_proba(X)
        get = (lambda i: X.iloc[i:i + chunk]) if hasattr(X, "iloc") else (lambda i: X[i:i + chunk])
        return np.vstack([model.predict_proba(get(i)) for i in range(0, n, chunk)])

    np.random.seed(a["seed_kernel"])
    ex = shap.KernelExplainer(f, a["bg"], keep_index=True, keep_index_ordered=True)
    sv = ex.shap_values(a["E"], nsamples="auto", silent=True)
    bounds = np.cumsum([0] + list(a["n_cat"]))
    per_class = []
    for cs in sv:
        cs = np.asarray(cs)
        summed = [cs[:, bounds[k]:bounds[k + 1]].sum(1) for k in range(len(a["n_cat"]))]
        per_class.append(np.column_stack(summed + [cs[:, bounds[-1]:]]) if summed else cs)
    return np.abs(np.vstack(per_class)).sum(0) / len(a["E"])


def job(a):
    warnings.filterwarnings("ignore")
    soft, hard = resource.getrlimit(resource.RLIMIT_AS)
    limit = _vmsize() + a["mem_limit_mb"] * 1024 ** 2
    if hard != resource.RLIM_INFINITY:
        limit = min(limit, hard)
    resource.setrlimit(resource.RLIMIT_AS, (limit, hard))
    out = dict(status="ok", err="")
    try:
        t0 = time.time()
        model = _refit(a)
        out["fit_secs"] = time.time() - t0
        out["pE"] = np.asarray(model.predict_proba(a["E"]))[:, 1]
        out["test_acc"] = float((np.asarray(model.predict(a["X_test"])).ravel() == a["y_test"]).mean())
        t1 = time.time()
        out["m_block"] = _kernel(model, a)
        out["explain_secs"] = time.time() - t1
    except MemoryError:
        out.update(status="memout")
    except Exception as e:
        out.update(status="error", err=f"{type(e).__name__}: {str(e)[:300]}")
    tmp = a["out_path"] + ".tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, a["out_path"])


def run_isolated(a, time_limit_s, workdir):
    import subprocess, tempfile
    fd, apath = tempfile.mkstemp(suffix=".pkl", dir=a["tmp_dir"])
    os.close(fd)
    with open(apath, "wb") as f:
        pickle.dump(a, f, protocol=4)
    t0 = time.time()
    status, err = None, ""
    try:
        r = subprocess.run([sys.executable, "-m", "phase1c_jobs", apath], cwd=workdir,
                           timeout=time_limit_s, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        if r.returncode != 0:
            status = "crashed"
            err = f"exit code {r.returncode}; " + r.stderr.decode(errors="ignore")[-300:]
    except subprocess.TimeoutExpired:
        status, err = "timeout", f"exceeded {time_limit_s}s"
    finally:
        os.remove(apath)
    if os.path.exists(a["out_path"]):
        return "done"
    np.savez(a["out_path"], status=status or "crashed", err=err, total_secs=time.time() - t0)
    return status or "crashed"


if __name__ == "__main__":
    with open(sys.argv[1], "rb") as f:
        job(pickle.load(f))
'''
(WORKDIR / "phase1c_jobs.py").write_text(JOB_SRC)
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
import phase1c_jobs
importlib.reload(phase1c_jobs)

# -----------------------------------------------------------------------------
# 1. Original measurement inputs per dataset (identical rows to the original run)
# -----------------------------------------------------------------------------
KIN = {}
for did in CFG.dataset_ids:
    o, D = ORIG[did], DATA[did]
    Xtr = model_input(o, D.X[o.train_rows])
    Xte = model_input(o, D.X[o.test_rows])
    KIN[did] = dict(X_train=Xtr, y_train=D.y_orig[o.train_rows],
                    X_test=Xte, y_test=D.y_orig[o.test_rows],
                    bg=shap.sample(Xtr, 100),                         # original: random_state=0
                    E=shap.sample(Xte, 100, random_state=42),         # original E
                    n_cat=[len(c) for c in o.encoder.categories_] if o.encoder is not None else [])

# -----------------------------------------------------------------------------
# 2. Subset for item 4 (stratified by classifier) + item-5 RF pipeline
# -----------------------------------------------------------------------------
def equal_allocation(avail: dict, n: int) -> dict:
    q = {k: 0 for k in avail}
    while n > 0:
        open_ = [k for k in sorted(avail) if q[k] < avail[k]]
        if not open_:
            break
        for k in open_:
            if n == 0:
                break
            q[k] += 1; n -= 1
    return q

SUBSET, ITEM5_PICK = {}, {}
for did in CFG.dataset_ids:
    meta = POP[did]["meta_ok"]
    rng = np.random.default_rng(derive_seed("delta0-subset-orig", did))
    quota = equal_allocation(meta.classifier.value_counts().to_dict(), N_SUBSET)
    chosen = []
    for c, q in quota.items():
        idx = meta.index[meta.classifier == c].to_numpy()
        chosen += list(rng.choice(idx, size=q, replace=False))
    rf_cand = meta[(meta.classifier == "random_forest") & (meta.preprocessor == "no_preprocessing")
                   & (meta.cfg_balancing == "none") & (meta.cfg_rescaling != "normalize")]
    if len(rf_cand):
        pick = int(rng.choice(rf_cand.index.to_numpy()))
        ITEM5_PICK[did] = meta.loc[pick, "model_path"]
        if pick not in chosen:
            chosen.append(pick)
    SUBSET[did] = meta.loc[sorted(chosen)].copy()

print("Item-4 subset per classifier x dataset:")
print(pd.concat(SUBSET.values()).pivot_table(index="classifier", columns="did", values="model_path",
                                             aggfunc="count", fill_value=0).to_string())
print("\nItem-5 RF pipeline per dataset:",
      {did: (Path(p).parent.name + "/" + Path(p).name if p else None)
       for did, p in {d: ITEM5_PICK.get(d) for d in CFG.dataset_ids}.items()})

# -----------------------------------------------------------------------------
# 3. Jobs: (pipeline, kind, training seed s, explainer draw r)
# -----------------------------------------------------------------------------
def key_of(path):
    return hashlib.sha1(path.encode()).hexdigest()[:12]

def rep_path(did, path, kind, s, r):
    d = REP_DIR / str(did)
    d.mkdir(exist_ok=True)
    return d / f"{kind}_{key_of(path)}_s{s}_r{r}.npz"

def rep_args(did, path, kind, s, r):
    k = KIN[did]
    return dict(model_path=str(WORKDIR / path), kind=kind,
                X_train=k["X_train"], y_train=k["y_train"], X_test=k["X_test"], y_test=k["y_test"],
                bg=k["bg"], E=k["E"], n_cat=k["n_cat"], chunk=CFG.pred_chunk,
                seed_train=derive_seed("orig-refit", did, path, s),
                seed_base=derive_seed("orig-base", did, s),
                seed_kernel=derive_seed("kernel", did, path, kind, s, r),
                mem_limit_mb=REP_MEM_MB, out_path=str(rep_path(did, path, kind, s, r)),
                tmp_dir=str(DIRS["tmp"]))

JOBS = []
for did in CFG.dataset_ids:
    for path in SUBSET[did].model_path:
        JOBS += [(did, path, "asl", s, r) for s in range(CFG.n_train_seeds)
                 for r in range(CFG.n_explainer_reps)]
    if did in ITEM5_PICK:
        JOBS += [(did, ITEM5_PICK[did], "base", s, r) for s in range(CFG.n_train_seeds)
                 for r in range(CFG.n_explainer_reps)]
todo = [j for j in JOBS if not rep_path(*j).exists()]
n_workers = workers_for(REP_MEM_MB)
print(f"\n{len(JOBS)} replicate jobs, {len(todo)} to run on {n_workers} workers "
      f"({REP_MEM_MB} MB, {REP_TIME_S // 3600} h limit per job)")

if todo:
    for p in DIRS["tmp"].glob("*"):
        p.unlink()
    fresh_pool()
    t0 = time.time()
    Parallel(n_jobs=n_workers, verbose=5)(
        delayed(phase1c_jobs.run_isolated)(rep_args(*j), REP_TIME_S, str(WORKDIR)) for j in todo)
    print(f"finished in {(time.time() - t0) / 60:.1f} min")

status = pd.Series([str(np.load(rep_path(*j))["status"]) for j in JOBS])
print("\njob status:", status.value_counts().to_dict())

Item-4 subset per classifier x dataset:
did                  31    37    44    1461  4534
classifier                                       
adaboost                5     4     5     4     4
bernoulli_nb            4     4     0     4     0
decision_tree           0     3     0     3     4
extra_trees             4     3     5     3     4
gaussian_nb             0     3     0     3     0
gradient_boosting       4     3     4     3     4
k_nearest_neighbors     0     3     4     3     4
lda                     4     3     4     3     4
liblinear_svc           4     3     4     3     4
libsvm_svc              4     3     4     3     4
mlp                     4     3     4     3     4
multinomial_nb          1     3     0     3     0
passive_aggressive      4     3     4     3     4
qda                     4     3     4     3     4
random_forest           5     3     5     4     4
sgd                     4     3     4     3     3

Item-5 RF pipeline per dataset: {37: None, 31: 'random_fore

[Parallel(n_jobs=171)]: Using backend LokyBackend with 171 concurrent workers.
[Parallel(n_jobs=171)]: Done 108 tasks      | elapsed:   40.9s
[Parallel(n_jobs=171)]: Done 306 tasks      | elapsed:  2.0min
[Parallel(n_jobs=171)]: Done 540 tasks      | elapsed:  3.4min
[Parallel(n_jobs=171)]: Done 810 tasks      | elapsed:  8.6min
[Parallel(n_jobs=171)]: Done 1116 tasks      | elapsed: 39.8min
[Parallel(n_jobs=171)]: Done 1458 tasks      | elapsed: 70.3min
[Parallel(n_jobs=171)]: Done 1836 tasks      | elapsed: 163.9min
[Parallel(n_jobs=171)]: Done 2250 tasks      | elapsed: 258.0min
[Parallel(n_jobs=171)]: Done 2700 tasks      | elapsed: 518.3min
[Parallel(n_jobs=171)]: Done 3186 tasks      | elapsed: 645.7min
[Parallel(n_jobs=171)]: Done 3870 out of 3870 | elapsed: 918.1min finished


finished in 918.1 min

job status: {'ok': 3630, 'timeout': 240}


In [12]:
# =============================================================================
# CELL 6 — Item 4 (noise floor of the reused measurement) and item 5 (sanity)
#   Reused measurement: original KernelExplainer setup, unseeded draws per pipeline
#   -> two pipelines in the saved results differ by training AND independent draws.
#   delta_0     = E d(rep(s, r), rep(s', r')),  s != s'   (independent draws)
#   delta_expl  = E d(rep(s, r), rep(s, r')),   r != r'   (same trained model)
#   delta_seed  = E d(mean_r rep(s, .), mean_r rep(s', .)), s != s'
# =============================================================================
from itertools import combinations

S_, R_ = CFG.n_train_seeds, CFG.n_explainer_reps

def block_to_raw(did, v):
    names, blk = DATA[did].feature_names, block_order(did)
    raw = np.empty(len(names))
    for i, name in enumerate(blk):
        raw[names.index(name)] = v[i]
    return raw


def load_reps(did, path, kind):
    """Returns M (S, R, d) in raw order, PE (S, R, |E|), test accuracies, statuses."""
    M, PE, acc, st = [], [], [], []
    for s in range(S_):
        Ms, Ps = [], []
        for r in range(R_):
            z = np.load(rep_path(did, path, kind, s, r))
            st.append(str(z["status"]))
            if str(z["status"]) == "ok":
                Ms.append(block_to_raw(did, z["m_block"])); Ps.append(z["pE"]); acc.append(float(z["test_acc"]))
            else:
                Ms.append(None); Ps.append(None)
        M.append(Ms); PE.append(Ps)
    return M, PE, acc, st


def noise_components(M):
    """M: S x R list of raw vectors (None = failed). Returns per-distance means."""
    N = [[normalise(m) if m is not None else None for m in row] for row in M]
    ok = lambda x: x is not None and not x["degenerate"]
    pairs = {
        "delta_0":    [(N[s][r], N[t][q]) for s, t in combinations(range(S_), 2)
                       for r in range(R_) for q in range(R_)],
        "delta_expl": [(N[s][r], N[s][q]) for s in range(S_) for r, q in combinations(range(R_), 2)],
    }
    seed_avg = []
    for s in range(S_):
        good = [m for m in M[s] if m is not None]
        seed_avg.append(normalise(np.mean(good, axis=0)) if good else None)
    pairs["delta_seed"] = [(seed_avg[s], seed_avg[t]) for s, t in combinations(range(S_), 2)]
    out = {}
    for name, pp in pairs.items():
        dd = [explanation_distances(a, b) for a, b in pp if ok(a) and ok(b)]
        for k in DIST_NAMES:
            out[(name, k)] = float(np.mean([d[k] for d in dd])) if dd else np.nan
    return out

# -----------------------------------------------------------------------------
# Item 4: per pipeline, then pooled
# -----------------------------------------------------------------------------
rows, bad_jobs = [], []
for did in CFG.dataset_ids:
    for r_ in SUBSET[did].itertuples():
        M, PE, acc, st = load_reps(did, r_.model_path, "asl")
        if any(x != "ok" for x in st):
            bad_jobs.append(dict(did=did, classifier=r_.classifier, preprocessor=r_.preprocessor,
                                 statuses=pd.Series(st).value_counts().to_dict()))
        for s in range(S_):                       # same seed -> same model in every job
            good = [p for p in PE[s] if p is not None]
            assert all(np.array_equal(good[0], p) for p in good[1:]), \
                f"{did} {r_.classifier}: seed {s} not reproducible across jobs"
        nc = noise_components(M)
        rows.append(dict(did=did, classifier=r_.classifier, preprocessor=r_.preprocessor,
                         model_path=r_.model_path, n_ok=sum(x == "ok" for x in st),
                         orig_acc=r_.accuracy, refit_acc=np.mean(acc) if acc else np.nan,
                         deterministic=all(np.array_equal(PE[0][0], PE[s][0])
                                           for s in range(S_) if PE[s][0] is not None),
                         **{f"{a}|{k}": v for (a, k), v in nc.items()}))
NOISE = pd.DataFrame(rows)
NOISE.to_csv(DIRS["metrics"] / "item4_noise_per_pipeline.csv", index=False)

print("Replicate jobs not ok:", len(bad_jobs))
if bad_jobs:
    print(pd.DataFrame(bad_jobs).to_string())

def pooled(df, comp):
    return df.groupby("did")[[f"{comp}|{k}" for k in DIST_NAMES]].mean().set_axis(DIST_NAMES, axis=1)

DELTA0 = pooled(NOISE, "delta_0")
print("\nItem 4 — delta_0 (different seeds + independent draws = how two pipelines in the "
      "reused results differ when nothing real differs):")
print(DELTA0.round(4).to_string())
print("\nexplainer part (same model, different draws):")
print(pooled(NOISE, "delta_expl").round(4).to_string())
print("\ntraining-seed part (draws averaged out):")
print(pooled(NOISE, "delta_seed").round(4).to_string())
print("\ndelta_0 (TV) per classifier x dataset:")
print(NOISE.pivot_table(index="classifier", columns="did", values="delta_0|tv",
                        aggfunc="mean").round(4).to_string())
print("\ndeterministic refits (all seeds give the same model):",
      NOISE.groupby("did").deterministic.mean().round(2).to_dict())
print("\nrefit vs original test accuracy (mean |difference|):",
      (NOISE.refit_acc - NOISE.orig_acc).abs().groupby(NOISE.did).mean().round(4).to_dict())
DELTA0.to_csv(DIRS["metrics"] / "delta0_reused_measurement.csv")

# -----------------------------------------------------------------------------
# Item 5: (a) explainer noise per trained model; (b) RF vs RF within delta_0
# -----------------------------------------------------------------------------
def item5(did):
    path = ITEM5_PICK[did]
    Ma, _, _, sa = load_reps(did, path, "asl")
    Mb, _, _, sb = load_reps(did, path, "base")
    assert all(x == "ok" for x in sa + sb), f"{did}: item-5 jobs incomplete"
    groups = [[normalise(m) for m in row] for row in Ma] + [[normalise(m) for m in row] for row in Mb]
    n = len(groups)                                   # 3 auto-sklearn + 3 baseline trained models
    # (a) explainer-only TV within each trained model
    expl = [np.mean([explanation_distances(g[r], g[q])["tv"] for r, q in combinations(range(R_), 2)])
            for g in groups]

    def stats(labels, k):
        cross, within = [], []
        for i, j in combinations(range(n), 2):
            d = [explanation_distances(a, b)[k] for a in groups[i] for b in groups[j]]
            (within if labels[i] == labels[j] else cross).extend(d)
        return np.mean(cross), np.mean(within)

    obs = np.array([0] * S_ + [1] * S_)
    out = {}
    for k in DIST_NAMES:
        c, w = stats(obs, k)
        null = []
        for grp in combinations(range(n), S_):
            lab = np.zeros(n, int); lab[list(grp)] = 1
            cc, ww = stats(lab, k)
            null.append(cc - ww)
        out[k] = dict(within=w, rf_vs_rf=c, ratio=c / w if w > 0 else (np.inf if c > 0 else 1.0),
                      perm_p=float(np.mean(np.array(null) >= (c - w) - 1e-15)),
                      delta0_pooled=DELTA0.loc[did, k])
    return np.array(expl), pd.DataFrame(out).T

print("\n" + "=" * 90)
ITEM5 = {}
for did in CFG.dataset_ids:
    if did not in ITEM5_PICK:
        print(f"\n{did} {DATA[did].name}: no random_forest + no_preprocessing + no-balancing "
              f"pipeline in the filtered set -> item 5 not available")
        continue
    expl, tab = item5(did)
    ITEM5[did] = tab
    print(f"\n{did} {DATA[did].name}  ({Path(ITEM5_PICK[did]).parent.name}/{Path(ITEM5_PICK[did]).name})")
    print("  (a) explainer-only TV per trained model: auto-sklearn "
          + ", ".join(f"{v:.4f}" for v in expl[:S_]) + " | baseline " + ", ".join(f"{v:.4f}" for v in expl[S_:]))
    print("  (b) RF vs RF under the reused measurement:")
    print(tab.round(4).to_string())
    verdict = "PASS" if tab.loc["tv", "ratio"] <= 2 else "FAIL: RF-vs-RF well above the noise floor"
    print(f"  verdict (TV ratio {tab.loc['tv', 'ratio']:.2f}): {verdict}")
if ITEM5:
    pd.concat(ITEM5, names=["did", "distance"]).to_csv(DIRS["metrics"] / "item5_reused_measurement.csv")

Replicate jobs not ok: 16
     did           classifier                 preprocessor         statuses
0     31             adaboost                   polynomial  {'timeout': 15}
1     44   passive_aggressive             nystroem_sampler  {'timeout': 15}
2     44  k_nearest_neighbors                   polynomial  {'timeout': 15}
3     44  k_nearest_neighbors       random_trees_embedding  {'timeout': 15}
4   4534  k_nearest_neighbors        feature_agglomeration  {'timeout': 15}
5   4534  k_nearest_neighbors             no_preprocessing  {'timeout': 15}
6   4534        liblinear_svc             nystroem_sampler  {'timeout': 15}
7   4534                  qda                   polynomial  {'timeout': 15}
8   4534                  qda                   polynomial  {'timeout': 15}
9   4534                  qda                   polynomial  {'timeout': 15}
10  4534                  lda                   kernel_pca  {'timeout': 15}
11  4534                  qda                   polynomial  {'

In [13]:
# =============================================================================
# CELL 7 — Item 3: pairwise distance matrix D, mean pairwise dispersion D_bar,
#          and mean distance within / between component groups (all in delta_0 units)
#          on all 24,821 original filtered pipelines (reused measurement)
# =============================================================================
FACTORS = ["classifier", "preprocessor", "cfg_balancing", "cfg_rescaling"]
D_DIR = DIRS["metrics"] / "D"
D_DIR.mkdir(exist_ok=True)


def within_between(Dm, labels):
    """Mean distance over pairs in the same group vs pairs in different groups."""
    codes, uniq = pd.factorize(labels)
    G = np.zeros((len(codes), len(uniq)))
    G[np.arange(len(codes)), codes] = 1.0
    block = G.T @ (Dm @ G)                      # g x g sums of D over group pairs
    n_g = G.sum(0)
    within_sum = np.trace(block) / 2            # each within pair counted twice
    within_n = float((n_g * (n_g - 1) / 2).sum())
    total_sum = Dm.sum() / 2
    n = len(codes)
    total_n = n * (n - 1) / 2
    between_n = total_n - within_n
    return (within_sum / within_n if within_n else np.nan,
            (total_sum - within_sum) / between_n if between_n else np.nan)


ITEM3_ROWS = []
t0 = time.time()
for did in CFG.dataset_ids:
    P = POP[did]
    meta = P["meta_ok"]
    D = pairwise_D(P["S"], P["R"])
    for k in DIST_NAMES:
        np.save(D_DIR / f"D_{did}_{k}.npy", D[k].astype(np.float32))
    meta[["model_path", "classifier", "preprocessor", "cfg_balancing", "cfg_rescaling"]] \
        .to_csv(D_DIR / f"D_{did}_index.csv", index=False)
    for k in DIST_NAMES:
        d0 = DELTA0.loc[did, k]
        row = dict(did=did, distance=k, n=len(meta), delta0=d0,
                   D_bar=mean_pairwise(D[k]))
        row["D_bar/delta0"] = row["D_bar"] / d0 if d0 > 0 else np.inf
        for f in FACTORS:
            w, b = within_between(D[k], meta[f].astype(str).to_numpy())
            row[f"{f}:within/delta0"] = w / d0 if d0 > 0 else np.inf
            row[f"{f}:between/delta0"] = b / d0 if d0 > 0 else np.inf
            row[f"{f}:between/within"] = b / w if w > 0 else np.inf
        ITEM3_ROWS.append(row)
    del D
    print(f"  {did} {DATA[did].name}: D computed for {len(meta)} pipelines ({time.time() - t0:.0f}s)")

ITEM3 = pd.DataFrame(ITEM3_ROWS).set_index(["did", "distance"])
ITEM3.to_csv(DIRS["metrics"] / "item3_dispersion.csv")

pd.set_option("display.width", 220)
print("\nItem 3 — mean pairwise dispersion D_bar, in units of delta_0:")
print(ITEM3[["n", "delta0", "D_bar", "D_bar/delta0"]].round(4).to_string())

for k in DIST_NAMES:
    print(f"\n[{k}] between-group / within-group mean distance (1.0 = component separates nothing):")
    print(ITEM3.xs(k, level="distance")[[f"{f}:between/within" for f in FACTORS]]
          .rename(columns=lambda c: c.split(":")[0]).round(3).to_string())

print("\n[tv] within- and between-group distances in delta_0 units:")
cols = [f"{f}:{w}/delta0" for f in FACTORS for w in ("within", "between")]
print(ITEM3.xs("tv", level="distance")[cols].round(2).to_string())

noisy = ITEM3[ITEM3.delta0 > 0.5].index.tolist()
print(f"\ndistances whose noise floor exceeds 0.5 (near-uninformative under the reused measurement): {noisy}")

  37 diabetes: D computed for 8610 pipelines (5s)
  31 credit-g: D computed for 1598 pipelines (6s)
  44 spambase: D computed for 4306 pipelines (10s)
  4534 PhishingWebsites: D computed for 3680 pipelines (13s)
  1461 bank-marketing: D computed for 6627 pipelines (17s)

Item 3 — mean pairwise dispersion D_bar, in units of delta_0:
                  n  delta0   D_bar  D_bar/delta0
did  distance                                    
37   tv        8610  0.0088  0.0538        6.1097
     jac3      8610  0.1527  0.4424        2.8976
     jac5      8610  0.1231  0.3316        2.6935
     rbo       8610  0.0620  0.1669        2.6895
31   tv        1598  0.0377  0.0392        1.0402
     jac3      1598  0.3354  0.3297        0.9831
     jac5      1598  0.2246  0.2094        0.9324
     rbo       1598  0.0874  0.0867        0.9924
44   tv        4306  0.0600  0.0598        0.9969
     jac3      4306  0.9583  0.9625        1.0044
     jac5      4306  0.9392  0.9434        1.0045
     rbo       4

In [14]:
# =============================================================================
# CELL 8 — Phase-1 decision point: does classifier dominance survive?
#   One-way distance-based R^2 per component (Anderson 2001 sums of squares on
#   squared distances), random-label baseline, permutation p (TV), verdict.
# =============================================================================
N_PERM = 199
ORIGINAL_HEADLINE = {"classifier": 0.18, "feature preprocessing": 0.0075}


def ss_within(D2, labels):
    codes, uniq = pd.factorize(labels)
    G = np.zeros((len(codes), len(uniq)))
    G[np.arange(len(codes)), codes] = 1.0
    sums = np.einsum("ig,ig->g", G, D2 @ G) / 2          # sum_{i<j in g} d^2
    return float((sums / G.sum(0)).sum())


def r2_oneway(D2, labels, ss_total):
    return 1.0 - ss_within(D2, labels) / ss_total


ROWS = []
rng = np.random.default_rng(derive_seed("phase1-permutations"))
for did in CFG.dataset_ids:
    meta = pd.read_csv(D_DIR / f"D_{did}_index.csv")
    n = len(meta)
    for k in DIST_NAMES:
        D2 = np.load(D_DIR / f"D_{did}_{k}.npy").astype(np.float64) ** 2
        ss_total = float(D2[np.triu_indices(n, 1)].sum() / n)
        resolvable = ITEM3.loc[(did, k), "D_bar/delta0"] >= 1.2
        for f in FACTORS:
            lab = meta[f].astype(str).to_numpy()
            g = len(np.unique(lab))
            r2 = r2_oneway(D2, lab, ss_total)
            base = (g - 1) / (n - 1)
            p = np.nan
            if k == "tv":
                null = [r2_oneway(D2, rng.permutation(lab), ss_total) for _ in range(N_PERM)]
                p = (1 + sum(x >= r2 for x in null)) / (N_PERM + 1)
            ROWS.append(dict(did=did, name=DATA[did].name, distance=k, factor=f, n_groups=g,
                             R2=r2, R2_random=base, R2_over_random=r2 / base, perm_p=p,
                             D_bar_over_delta0=ITEM3.loc[(did, k), "D_bar/delta0"],
                             resolvable=resolvable))
        del D2
    print(f"  {did} {DATA[did].name} done")

PH1 = pd.DataFrame(ROWS)
PH1.to_csv(DIRS["metrics"] / "phase1_decision_r2.csv", index=False)

for k in DIST_NAMES:
    print(f"\n[{k}] one-way R^2 per component (x random-label baseline in brackets)")
    t = PH1[PH1.distance == k].assign(
        cell=lambda x: x.R2.map("{:.4f}".format) + " (" + x.R2_over_random.map("{:.1f}x".format) + ")")
    print(t.pivot(index="did", columns="factor", values="cell")[FACTORS].to_string())

print("\n[tv] permutation p-values (199 label shuffles):")
print(PH1[PH1.distance == "tv"].pivot(index="did", columns="factor", values="perm_p")[FACTORS].round(3).to_string())

# -----------------------------------------------------------------------------
# Verdict (TV is the only distance with a usable noise floor on every dataset)
# -----------------------------------------------------------------------------
print("\n" + "=" * 90)
print("PHASE-1 VERDICT  (rule: where D_bar/delta_0 >= 1.2, classifier dominates if its R^2 is "
      "the largest and >= 2x the next-largest component)")
print(f"original headline for reference: {ORIGINAL_HEADLINE}")
print("=" * 90)
verdicts = []
for did in CFG.dataset_ids:
    t = PH1[(PH1.did == did) & (PH1.distance == "tv")].set_index("factor")
    ratio = t.D_bar_over_delta0.iloc[0]
    order = t.R2.sort_values(ascending=False)
    if ratio < 1.2:
        v = "NOT RESOLVABLE: dispersion at the noise floor -> no component effect can be claimed"
    elif order.index[0] == "classifier" and order.iloc[0] >= 2 * order.iloc[1]:
        v = "classifier DOMINATES"
    elif order.index[0] == "classifier":
        v = f"classifier largest but NOT dominant (next: {order.index[1]}, ratio {order.iloc[0] / order.iloc[1]:.2f})"
    else:
        v = f"classifier NOT largest (largest: {order.index[0]})"
    verdicts.append(dict(did=did, name=DATA[did].name, D_bar_over_delta0=round(ratio, 2),
                         **{f"R2_{f}": round(t.loc[f, "R2"], 4) for f in FACTORS}, verdict=v))
VERDICT = pd.DataFrame(verdicts).set_index("did")
VERDICT.to_csv(DIRS["metrics"] / "phase1_verdict.csv")
print(VERDICT.to_string())

  37 diabetes done
  31 credit-g done
  44 spambase done
  4534 PhishingWebsites done
  1461 bank-marketing done

[tv] one-way R^2 per component (x random-label baseline in brackets)
factor       classifier    preprocessor   cfg_balancing    cfg_rescaling
did                                                                     
31        0.0232 (3.1x)   0.0147 (2.0x)   0.0026 (4.1x)    0.0056 (1.5x)
37      0.2936 (168.5x)  0.1260 (90.4x)  0.0108 (93.4x)  0.1842 (264.3x)
44        0.0082 (3.2x)   0.0055 (2.0x)   0.0004 (1.5x)    0.0026 (1.9x)
1461      0.0063 (2.8x)   0.0027 (1.6x)   0.0005 (3.1x)    0.0023 (2.5x)
4534      0.0100 (3.1x)   0.0063 (2.3x)   0.0007 (2.6x)    0.0024 (1.5x)

[jac3] one-way R^2 per component (x random-label baseline in brackets)
factor      classifier    preprocessor   cfg_balancing    cfg_rescaling
did                                                                    
31       0.0280 (3.7x)   0.0143 (1.9x)   0.0010 (1.7x)    0.0050 (1.3x)
37      0.1417 (81

In [15]:
# =============================================================================
# CELL 9 — Phase 2: load the saved factorial (items 6, 7, 1-2 from the previous
#          notebook) and report coverage for items 8-9
#   Design and explanations: PermutationExplainer, run r = 0 shared by all
#   pipelines, 3 training seeds, rescaling pinned (standardize), LDA/QDA as-is.
# =============================================================================
P2_DATA  = PHASE2_CACHE / "data"
P2_METR  = PHASE2_CACHE / "metrics"
P2_EXPL  = PHASE2_CACHE / "shap" / "explain"

DESIGN = pd.read_csv(P2_DATA / "design.csv")
SEL = pd.read_csv(P2_METR / "explained_selection.csv")
CV2 = {did: pd.read_csv(P2_METR / f"cv_{did}.csv") for did in CFG.dataset_ids}
for did, cv in CV2.items():
    assert {"in_R_1se", "in_R_2se", "logloss", "accuracy"} <= set(cv.columns), f"{did}: CV file incomplete"

# -----------------------------------------------------------------------------
# Item 6 and 7 summary (from saved results)
# -----------------------------------------------------------------------------
cells = DESIGN.drop_duplicates(["did", "classifier", "preprocessor", "balancing"])
summary = []
for did in CFG.dataset_ids:
    c = cells[cells.did == did]
    cv = CV2[did]
    summary.append(dict(did=did, name=DATA[did].name,
                        cells_sampled=int((c.status == "sampled").sum()),
                        holes_forbidden=int((c.status == "forbidden").sum()),
                        holes_incompatible=int((c.status == "incompatible").sum()),
                        configs=int((DESIGN[DESIGN.did == did].status == "sampled").sum()),
                        cv_ok=int((cv.status == "ok").sum()), cv_failed=int((cv.status != "ok").sum()),
                        R_1se=int(cv.in_R_1se.sum()), R_2se=int(cv.in_R_2se.sum())))
print("Items 6-7 (saved):")
print(pd.DataFrame(summary).set_index("did").to_string())

# -----------------------------------------------------------------------------
# Explanations (run r = 0, seeds 0..2) for the selected configurations
# -----------------------------------------------------------------------------
rows = []
for r_ in SEL.itertuples():
    ms, effs, st = [], [], []
    for s in range(CFG.n_train_seeds):
        p = P2_EXPL / str(r_.did) / f"{r_.config_id}_s{s}_r0.npz"
        if not p.exists():
            st.append("missing"); ms.append(None); continue
        z = np.load(p)
        st.append(str(z["status"]))
        if str(z["status"]) == "ok":
            ms.append(z["m"]); effs.append(float(z["efficiency_error"]))
        else:
            ms.append(None)
    rows.append(dict(did=r_.did, config_id=r_.config_id, classifier=r_.classifier,
                     preprocessor=r_.preprocessor, balancing=r_.balancing, pair_idx=r_.pair_idx,
                     role=r_.role, in_R_1se=bool(r_.in_R_1se), in_R_2se=bool(r_.in_R_2se),
                     logloss=r_.logloss, accuracy=r_.accuracy,
                     n_seeds_ok=sum(x == "ok" for x in st),
                     max_eff_err=max(effs) if effs else np.nan,
                     M=np.vstack(ms) if all(m is not None for m in ms) else None))
P2 = pd.DataFrame(rows)
assert P2.max_eff_err.max() < 1e-6, "efficiency violated in a saved explanation"
print(f"\nexplanation files: {int(P2.n_seeds_ok.sum())} ok of {len(P2) * CFG.n_train_seeds} "
      f"(previous notebook reported 3,999 of 4,020)")

P2 = P2[P2.M.notna()].reset_index(drop=True)                       # all 3 seeds ok
P2["degenerate"] = P2.M.apply(lambda M: any(normalise(m)["degenerate"] for m in M))
print(f"configurations with all 3 seeds explained: {len(P2)}; degenerate (constant output): "
      f"{int(P2.degenerate.sum())} -> excluded")
P2 = P2[~P2.degenerate].reset_index(drop=True)

print("\nExplained configurations per dataset and role:")
print(P2.groupby(["did", "role"]).size().unstack(fill_value=0).to_string())

# -----------------------------------------------------------------------------
# Coverage of the factorial cells (what item 8's interaction term needs)
# -----------------------------------------------------------------------------
print("\nCell coverage of the explained configurations (cell = classifier x preprocessor x balancing):")
cov = []
for did in CFG.dataset_ids:
    for set_name, mask in (("U (all explained)", P2.did == did),
                           ("R_2se", (P2.did == did) & P2.in_R_2se),
                           ("R_1se", (P2.did == did) & P2.in_R_1se)):
        sub = P2[mask]
        per_cell = sub.groupby(["classifier", "preprocessor", "balancing"]).size()
        cp = sub.groupby(["classifier", "preprocessor"]).size()
        cov.append(dict(did=did, set=set_name, configs=len(sub),
                        classifiers=sub.classifier.nunique(), preprocessors=sub.preprocessor.nunique(),
                        cells_present=len(per_cell), cells_with_2plus=int((per_cell >= 2).sum()),
                        clf_x_pp_present=len(cp), clf_x_pp_with_2plus=int((cp >= 2).sum())))
COV = pd.DataFrame(cov).set_index(["did", "set"])
print(COV.to_string())
print("\nfor reference: 330 sampled cells and 165 allowed classifier x preprocessor pairs per dataset")

P2.drop(columns="M").to_csv(DIRS["data"] / "phase2_population.csv", index=False)

Items 6-7 (saved):
                  name  cells_sampled  holes_forbidden  holes_incompatible  configs  cv_ok  cv_failed  R_1se  R_2se
did                                                                                                                
37            diabetes            330               54                  32     1640   1628         12     55    161
31            credit-g            330               54                  32     1642   1600         42     50    141
44            spambase            330               54                  32     1634   1608         26      4     18
4534  PhishingWebsites            330               54                  32     1640   1574         66      9     12
1461    bank-marketing            330               54                  32     1636   1406        230      5      8

explanation files: 3999 ok of 4020 (previous notebook reported 3,999 of 4,020)
configurations with all 3 seeds explained: 1333; degenerate (constant output): 12 -> excl

In [16]:
# =============================================================================
# CELL 10 — Phase 2, option B: explain the full factorial (every CV-ok
#           configuration), seed 0, run r = 0, with the identical protocol of the
#           existing Phase-2 explanations (same worker module, inputs and seeds)
# =============================================================================
import importlib
from ConfigSpace import Configuration
from ConfigSpace.hyperparameters import (CategoricalHyperparameter, Constant, OrdinalHyperparameter,
                                         UniformIntegerHyperparameter, UniformFloatHyperparameter)
from autosklearn.pipeline.classification import SimpleClassificationPipeline

P2_TIME_S = 6 * 3600                   # same per-job limit as the existing run
P2_MEM_MB = 6144
DP2 = {"multiclass": False, "multilabel": False, "sparse": False}   # as in the previous notebook

assert (WORKDIR / "phase1_explain.py").exists(), "worker module from the previous notebook not found"
import phase1_explain
importlib.reload(phase1_explain)

# ---- Phase-2 inputs (saved row sets, configs, seeds) --------------------------------
with open(P2_DATA / "configs.json") as f:
    CONF_JSON = json.load(f)
with open(P2_DATA / "train_seeds.json") as f:
    SEEDS2 = json.load(f)
ROWS2 = {did: np.load(P2_DATA / f"rowsets_{did}.npz", allow_pickle=True) for did in CFG.dataset_ids}
FEAT2 = {did: {i: ("categorical" if c else "numerical") for i, c in enumerate(DATA[did].cat_mask)}
         for did in CFG.dataset_ids}
SPACE2 = {did: SimpleClassificationPipeline(feat_type=FEAT2[did], dataset_properties=DP2)
                .get_hyperparameter_search_space() for did in CFG.dataset_ids}


def rebuild_config(did, cid):
    """configs.json entry -> Configuration with the exact types of auto-sklearn's space,
    checked value by value against the saved record."""
    raw = CONF_JSON[f"{did}|{cid}"]
    cs = SPACE2[did]
    vals = {}
    for name, v in raw.items():
        hp = cs.get_hyperparameter(name)
        if isinstance(hp, UniformIntegerHyperparameter):
            vals[name] = int(v)
        elif isinstance(hp, UniformFloatHyperparameter):
            vals[name] = float(v)
        elif isinstance(hp, (CategoricalHyperparameter, OrdinalHyperparameter)):
            choices = hp.choices if isinstance(hp, CategoricalHyperparameter) else hp.sequence
            match = [c for c in choices if str(c) == str(v)]
            assert len(match) == 1, f"{name}: {v!r} not among {choices}"
            vals[name] = match[0]
        elif isinstance(hp, Constant):
            assert str(hp.value) == str(v), f"{name}: {v!r} != constant {hp.value!r}"
            vals[name] = hp.value
        else:
            raise TypeError(f"unhandled hyperparameter type {type(hp).__name__} for {name}")
    conf = Configuration(cs, values=vals)
    back = conf.get_dictionary()
    assert set(back) == set(raw) and all(str(back[k]) == str(raw[k]) for k in raw), \
        f"{did}|{cid}: rebuilt configuration differs from the saved record"
    return conf


def p2_path(did, cid, s=0, r=0):
    return P2_EXPL / str(did) / f"{cid}_s{s}_r{r}.npz"


def p2_args(did, cid, out_path=None):
    rs, D = ROWS2[did], DATA[did]
    return dict(conf=rebuild_config(did, cid).get_dictionary(), feat_type=FEAT2[did], dp=DP2,
                X_train=D.X[rs["train_idx"]], y_train=D.y_event[rs["train_idx"]],
                seed=SEEDS2[f"{did}|{cid}"][0],
                B=D.X[rs["bg_idx"][0]], XE=D.X[rs["E_idx"]],
                max_evals=CFG.max_evals(D.d), perm_seed=derive_seed("perm", did, 0),
                chunk=CFG.pred_chunk, mem_limit_mb=P2_MEM_MB,
                out_path=str(out_path or p2_path(did, cid)), tmp_dir=str(DIRS["tmp"]))

# ---- 1. bit-for-bit check: recompute one existing explanation from the rebuilt config ----
chk = P2[(P2.did == 37) & (P2.classifier == "lda")].iloc[0]
chk_path = DIRS["tmp"] / "p2_check.npz"
if chk_path.exists():
    chk_path.unlink()
print("check job:", phase1_explain.run_isolated(p2_args(37, chk.config_id, chk_path), 3600, str(WORKDIR)))
new, old = np.load(chk_path), np.load(p2_path(37, chk.config_id))
assert str(new["status"]) == "ok", str(new["err"])
assert np.array_equal(new["m"], old["m"]) and np.array_equal(new["pE"], old["pE"]), \
    "recomputed explanation differs from the saved one: protocol not identical, stopping"
chk_path.unlink()
print("rebuilt configuration + saved inputs reproduce an existing explanation bit-for-bit")

# ---- 2. jobs: every CV-ok configuration without an explanation at (seed 0, run 0) -------
SLOW = {"k_nearest_neighbors": 3, "libsvm_svc": 3, "random_forest": 2, "extra_trees": 2,
        "gradient_boosting": 2, "mlp": 2, "adaboost": 1}
jobs = []
for did in CFG.dataset_ids:
    cv = CV2[did]
    for r_ in cv[cv.status == "ok"].itertuples():
        if not p2_path(did, r_.config_id).exists():
            cost = (CFG.max_evals(DATA[did].d) * len(ROWS2[did]["train_idx"]), SLOW.get(r_.classifier, 0))
            jobs.append((cost, did, r_.config_id))
jobs.sort(key=lambda j: j[0], reverse=True)
n_ok = sum(int((CV2[d].status == "ok").sum()) for d in CFG.dataset_ids)
n_workers = workers_for(P2_MEM_MB)
print(f"\n{n_ok} CV-ok configurations; {n_ok - len(jobs)} already explained at (seed 0, run 0); "
      f"{len(jobs)} to run on {n_workers} workers ({P2_MEM_MB} MB, {P2_TIME_S // 3600} h limit per job)")

for did in CFG.dataset_ids:                      # validate every configuration before launching
    for _, d_, cid in [j for j in jobs if j[1] == did]:
        rebuild_config(did, cid)
print("all configurations to run rebuilt and validated against configs.json")

if jobs:
    for p in DIRS["tmp"].glob("*"):
        p.unlink()
    fresh_pool()
    t0 = time.time()
    Parallel(n_jobs=n_workers, verbose=5)(
        delayed(phase1_explain.run_isolated)(p2_args(did, cid), P2_TIME_S, str(WORKDIR))
        for _, did, cid in jobs)
    print(f"finished in {(time.time() - t0) / 60:.1f} min")

# ---- 3. status --------------------------------------------------------------------------
st = []
for did in CFG.dataset_ids:
    cv = CV2[did]
    for r_ in cv[cv.status == "ok"].itertuples():
        z = np.load(p2_path(did, r_.config_id))
        st.append(dict(did=did, classifier=r_.classifier, preprocessor=r_.preprocessor,
                       status=str(z["status"])))
ST = pd.DataFrame(st)
print("\nFull-factorial explanation status (seed 0, run 0):")
print(ST.groupby(["did", "status"]).size().unstack(fill_value=0).to_string())
bad = ST[ST.status != "ok"]
if len(bad):
    print("\nnot explained, by classifier x preprocessor:")
    print(bad.groupby(["did", "classifier", "preprocessor", "status"]).size().to_string())

check job: done
rebuilt configuration + saved inputs reproduce an existing explanation bit-for-bit

7816 CV-ok configurations; 1340 already explained at (seed 0, run 0); 6476 to run on 174 workers (6144 MB, 6 h limit per job)
all configurations to run rebuilt and validated against configs.json


[Parallel(n_jobs=174)]: Using backend LokyBackend with 174 concurrent workers.
[Parallel(n_jobs=174)]: Done 102 tasks      | elapsed: 15.9min
[Parallel(n_jobs=174)]: Done 300 tasks      | elapsed: 30.6min
[Parallel(n_jobs=174)]: Done 534 tasks      | elapsed: 36.8min
[Parallel(n_jobs=174)]: Done 804 tasks      | elapsed: 38.9min
[Parallel(n_jobs=174)]: Done 1110 tasks      | elapsed: 42.4min
[Parallel(n_jobs=174)]: Done 1452 tasks      | elapsed: 65.7min
[Parallel(n_jobs=174)]: Done 1830 tasks      | elapsed: 78.8min
[Parallel(n_jobs=174)]: Done 2244 tasks      | elapsed: 91.7min
[Parallel(n_jobs=174)]: Done 2694 tasks      | elapsed: 159.4min
[Parallel(n_jobs=174)]: Done 3180 tasks      | elapsed: 183.0min
[Parallel(n_jobs=174)]: Done 3702 tasks      | elapsed: 237.3min
[Parallel(n_jobs=174)]: Done 4260 tasks      | elapsed: 288.2min
[Parallel(n_jobs=174)]: Done 4854 tasks      | elapsed: 304.5min
[Parallel(n_jobs=174)]: Done 5484 tasks      | elapsed: 327.1min
[Parallel(n_jobs=174)]:

finished in 646.5 min

Full-factorial explanation status (seed 0, run 0):
status  error    ok  timeout
did                         
31          0  1592        8
37          0  1628        0
44          0  1576       32
1461        1  1391       14
4534        0  1563       11

not explained, by classifier x preprocessor:
did   classifier           preprocessor                            status 
31    adaboost             polynomial                              timeout     2
      k_nearest_neighbors  polynomial                              timeout     2
      libsvm_svc           polynomial                              timeout     4
44    adaboost             polynomial                              timeout     2
      k_nearest_neighbors  polynomial                              timeout    10
      libsvm_svc           polynomial                              timeout    10
      qda                  polynomial                              timeout    10
1461  k_nearest_neighbors  feature_

In [17]:
# =============================================================================
# CELL 11 — Item 4 (Phase 2): explainer replicates for the delta_0 split
#   Third compute lever: explainer runs r = 1..4 (background B_r + permutation
#   seed r, shared by all pipelines) for all 3 training seeds, on ~50
#   configurations per dataset stratified by classifier.
# =============================================================================
N_SUB2      = 50
SUB_TIME_CAP = 3600          # only configs whose seed-0 explanation took <= 1 h (recorded)

def p2_rep_args(did, cid, s, r):
    rs, D = ROWS2[did], DATA[did]
    return dict(conf=rebuild_config(did, cid).get_dictionary(), feat_type=FEAT2[did], dp=DP2,
                X_train=D.X[rs["train_idx"]], y_train=D.y_event[rs["train_idx"]],
                seed=SEEDS2[f"{did}|{cid}"][s],
                B=D.X[rs["bg_idx"][r]], XE=D.X[rs["E_idx"]],
                max_evals=CFG.max_evals(D.d), perm_seed=derive_seed("perm", did, r),
                chunk=CFG.pred_chunk, mem_limit_mb=P2_MEM_MB,
                out_path=str(p2_path(did, cid, s, r)), tmp_dir=str(DIRS["tmp"]))

# ---- subset: configurations with all 3 seeds explained, stratified by classifier ------
P2["seed0_secs"] = [float(np.load(p2_path(r_.did, r_.config_id))["explain_secs"]) for r_ in P2.itertuples()]
SUB2 = {}
for did in CFG.dataset_ids:
    pool = P2[(P2.did == did) & (P2.seed0_secs <= SUB_TIME_CAP)]
    rng = np.random.default_rng(derive_seed("delta0-subset-p2", did))
    quota = equal_allocation(pool.classifier.value_counts().to_dict(), N_SUB2)
    chosen = []
    for c, q in quota.items():
        idx = pool.index[pool.classifier == c].to_numpy()
        chosen += list(rng.choice(idx, size=q, replace=False))
    SUB2[did] = P2.loc[sorted(chosen)]
excluded_slow = P2[P2.seed0_secs > SUB_TIME_CAP].groupby(["did", "classifier"]).size()
print("Item-4 subset per classifier x dataset:")
print(pd.concat(SUB2.values()).pivot_table(index="classifier", columns="did", values="config_id",
                                           aggfunc="count", fill_value=0).to_string())
print("\nnot eligible for the subset (seed-0 explanation > 1 h):")
print(excluded_slow.to_string() if len(excluded_slow) else "  none")
pd.concat(SUB2.values()).drop(columns="M").to_csv(DIRS["data"] / "phase2_delta0_subset.csv", index=False)

# ---- jobs: s = 0..2, r = 1..4 (r = 0 already exists for every seed) -------------------
jobs = [(did, cid, s, r) for did in CFG.dataset_ids for cid in SUB2[did].config_id
        for s in range(CFG.n_train_seeds) for r in range(1, CFG.n_explainer_reps)]
todo = [j for j in jobs if not p2_path(*j).exists()]
todo.sort(key=lambda j: -float(P2.loc[(P2.did == j[0]) & (P2.config_id == j[1]), "seed0_secs"].iloc[0]))
n_workers = workers_for(P2_MEM_MB)
print(f"\n{len(jobs)} replicate jobs, {len(todo)} to run on {n_workers} workers")

if todo:
    for p in DIRS["tmp"].glob("*"):
        p.unlink()
    fresh_pool()
    t0 = time.time()
    Parallel(n_jobs=n_workers, verbose=5)(
        delayed(phase1_explain.run_isolated)(p2_rep_args(*j), P2_TIME_S, str(WORKDIR)) for j in todo)
    print(f"finished in {(time.time() - t0) / 60:.1f} min")

status = pd.Series([str(np.load(p2_path(*j))["status"]) for j in jobs])
print("\nreplicate job status:", status.value_counts().to_dict())

Item-4 subset per classifier x dataset:
did                  31    37    44    1461  4534
classifier                                       
adaboost                4     4     4     4     4
bernoulli_nb            4     4     4     4     4
decision_tree           3     3     3     3     3
extra_trees             3     3     3     3     3
gaussian_nb             3     3     3     3     3
gradient_boosting       3     3     3     3     3
k_nearest_neighbors     3     3     3     3     3
lda                     3     3     3     3     3
liblinear_svc           3     3     3     3     3
libsvm_svc              3     3     3     3     3
mlp                     3     3     3     3     3
multinomial_nb          3     3     3     3     3
passive_aggressive      3     3     3     3     3
qda                     3     3     3     3     3
random_forest           3     3     3     3     3
sgd                     3     3     3     3     3

not eligible for the subset (seed-0 explanation > 1 h):
did

[Parallel(n_jobs=171)]: Using backend LokyBackend with 171 concurrent workers.
[Parallel(n_jobs=171)]: Done 108 tasks      | elapsed: 37.4min
[Parallel(n_jobs=171)]: Done 306 tasks      | elapsed: 51.6min
[Parallel(n_jobs=171)]: Done 540 tasks      | elapsed: 58.0min
[Parallel(n_jobs=171)]: Done 810 tasks      | elapsed: 63.2min
[Parallel(n_jobs=171)]: Done 1116 tasks      | elapsed: 66.3min
[Parallel(n_jobs=171)]: Done 1458 tasks      | elapsed: 68.6min
[Parallel(n_jobs=171)]: Done 1836 tasks      | elapsed: 70.4min
[Parallel(n_jobs=171)]: Done 2250 tasks      | elapsed: 71.4min
[Parallel(n_jobs=171)]: Done 3000 out of 3000 | elapsed: 72.5min finished


finished in 72.5 min

replicate job status: {'ok': 3000}


In [18]:
# =============================================================================
# CELL 12 — Phase 2: item 4 (delta_0 and its split) and item 3 (D, D_bar)
#   Measurement: PermutationExplainer, run r = 0 shared by all pipelines.
#   delta_0     = E d(rep(s, r=0), rep(s', r=0)), s != s'   (analysis protocol)
#   delta_expl  = E d(rep(s, r), rep(s, r')),     r != r'   (same trained model)
#   delta_seed  = E d(mean_r rep(s, .), mean_r rep(s', .))  (explainer noise averaged out)
#   delta_indep = E d(rep(s, r), rep(s', r')),    s != s', r != r'
# =============================================================================
from itertools import combinations

# ---------------------------------------------------------------------------
# Full factorial population: one explanation per configuration (seed 0, run 0)
# ---------------------------------------------------------------------------
rows = []
for did in CFG.dataset_ids:
    cv = CV2[did]
    for r_ in cv[cv.status == "ok"].itertuples():
        z = np.load(p2_path(did, r_.config_id))
        if str(z["status"]) != "ok":
            continue
        rows.append(dict(did=did, config_id=r_.config_id, classifier=r_.classifier,
                         preprocessor=r_.preprocessor, balancing=r_.balancing, pair_idx=r_.pair_idx,
                         in_R_1se=bool(r_.in_R_1se), in_R_2se=bool(r_.in_R_2se),
                         logloss=r_.logloss, accuracy=r_.accuracy,
                         eff=float(z["efficiency_error"]), m=z["m"]))
F2 = pd.DataFrame(rows)
assert F2.eff.max() < 1e-6, "efficiency violated in a saved explanation"
F2["degenerate"] = F2.m.apply(lambda m: normalise(m)["degenerate"])
print(f"full factorial: {len(F2)} explained configurations; degenerate (constant output) "
      f"excluded: {int(F2.degenerate.sum())}")
F2 = F2[~F2.degenerate].reset_index(drop=True)
print(F2.groupby("did").agg(configs=("config_id", "size"), R_2se=("in_R_2se", "sum"),
                            R_1se=("in_R_1se", "sum")).to_string())

# ---------------------------------------------------------------------------
# Item 4a: delta_0 from training seeds (r = 0) on every configuration with 3 seeds
# ---------------------------------------------------------------------------
def seed_m(did, cid, r=0):
    return [np.load(p2_path(did, cid, s, r))["m"] for s in range(CFG.n_train_seeds)]

d0_rows = []
for r_ in P2.itertuples():
    N = [normalise(m) for m in seed_m(r_.did, r_.config_id)]
    if any(n["degenerate"] for n in N):
        continue
    dd = [explanation_distances(N[i], N[j]) for i, j in combinations(range(len(N)), 2)]
    d0_rows.append(dict(did=r_.did, config_id=r_.config_id, classifier=r_.classifier,
                        **{k: float(np.mean([d[k] for d in dd])) for k in DIST_NAMES}))
D0T = pd.DataFrame(d0_rows)
D0T["deterministic"] = (D0T[DIST_NAMES] == 0).all(axis=1)
DELTA0_2 = D0T.groupby("did")[DIST_NAMES].mean()
print(f"\nItem 4 — delta_0 (seed replicates, shared run r = 0; {len(D0T)} configurations):")
print(DELTA0_2.round(4).to_string())
print("\nstochastic configurations only:")
print(D0T[~D0T.deterministic].groupby("did")[DIST_NAMES].mean().round(4).to_string())
print("\nshare of deterministic configurations:", D0T.groupby("did").deterministic.mean().round(2).to_dict())
print("\ndelta_0 (TV) per classifier x dataset:")
print(D0T.pivot_table(index="classifier", columns="did", values="tv", aggfunc="mean").round(4).to_string())

# ---------------------------------------------------------------------------
# Item 4b: the split, from the explainer-replicate subset (Cell 11)
# ---------------------------------------------------------------------------
split_rows, incomplete = [], 0
for did in CFG.dataset_ids:
    for cid in SUB2[did].config_id:
        paths = [[p2_path(did, cid, s, r) for r in range(CFG.n_explainer_reps)] for s in range(CFG.n_train_seeds)]
        Z = [[np.load(p) for p in row] for row in paths]
        if any(str(z["status"]) != "ok" for row in Z for z in row):
            incomplete += 1
            continue
        for row in Z:                                   # same seed -> same model in every run
            assert all(np.array_equal(row[0]["pE"], z["pE"]) for z in row[1:]), f"{did} {cid}: refit not reproducible"
        N = [[normalise(z["m"]) for z in row] for row in Z]
        Nbar = [normalise(np.mean([z["m"] for z in row], axis=0)) for row in Z]
        S_, R_ = CFG.n_train_seeds, CFG.n_explainer_reps
        groups = {
            "delta_0":     [(N[s][0], N[t][0]) for s, t in combinations(range(S_), 2)],
            "delta_expl":  [(N[s][r], N[s][q]) for s in range(S_) for r, q in combinations(range(R_), 2)],
            "delta_seed":  [(Nbar[s], Nbar[t]) for s, t in combinations(range(S_), 2)],
            "delta_indep": [(N[s][r], N[t][q]) for s, t in combinations(range(S_), 2)
                            for r in range(R_) for q in range(R_) if r != q],
        }
        rec = dict(did=did, config_id=cid)
        for g, pp in groups.items():
            dd = [explanation_distances(a, b) for a, b in pp]
            for k in DIST_NAMES:
                rec[f"{g}|{k}"] = float(np.mean([d[k] for d in dd]))
        split_rows.append(rec)
SPLIT = pd.DataFrame(split_rows)
print(f"\nItem 4 split (subset: {len(SPLIT)} configurations; incomplete replicate sets: {incomplete})")
for g in ("delta_0", "delta_expl", "delta_seed", "delta_indep"):
    t = SPLIT.groupby("did")[[f"{g}|{k}" for k in DIST_NAMES]].mean()
    t.columns = DIST_NAMES
    print(f"\n{g}:")
    print(t.round(4).to_string())
D0T.to_csv(DIRS["metrics"] / "p2_delta0_seeds.csv", index=False)
SPLIT.to_csv(DIRS["metrics"] / "p2_delta0_split.csv", index=False)
DELTA0_2.to_csv(DIRS["metrics"] / "p2_delta0.csv")

# ---------------------------------------------------------------------------
# Item 3: D and D_bar on the full factorial (one explanation per configuration)
# ---------------------------------------------------------------------------
D2_DIR = DIRS["metrics"] / "D_phase2"
D2_DIR.mkdir(exist_ok=True)
F2_FACTORS = ["classifier", "preprocessor", "balancing"]
ITEM3_2 = []
for did in CFG.dataset_ids:
    P = F2[F2.did == did].reset_index(drop=True)
    N = [normalise(m) for m in P.m]
    D = pairwise_D(np.vstack([x["shares"] for x in N]), np.vstack([x["ranks"] for x in N]))
    for k in DIST_NAMES:
        np.save(D2_DIR / f"D_{did}_{k}.npy", D[k].astype(np.float32))
    P.drop(columns="m").to_csv(D2_DIR / f"D_{did}_index.csv", index=False)
    for set_name, mask in (("all pipelines", np.ones(len(P), bool)),
                           ("R_2se", P.in_R_2se.to_numpy()), ("R_1se", P.in_R_1se.to_numpy())):
        idx = np.where(mask)[0]
        for k in DIST_NAMES:
            d0 = DELTA0_2.loc[did, k]
            row = dict(did=did, set=set_name, distance=k, n=len(idx), delta0=d0,
                       D_bar=mean_pairwise(D[k], idx) if len(idx) > 1 else np.nan)
            row["D_bar/delta0"] = row["D_bar"] / d0 if d0 > 0 else np.inf
            if len(idx) > 2:
                sub = D[k][np.ix_(idx, idx)]
                for f in F2_FACTORS:
                    lab = P.loc[idx, f].astype(str).to_numpy()
                    if len(np.unique(lab)) > 1:
                        w, b = within_between(sub, lab)
                        row[f"{f}:between/within"] = b / w if w > 0 else np.inf
            ITEM3_2.append(row)
    del D
ITEM3_2 = pd.DataFrame(ITEM3_2).set_index(["did", "set", "distance"])
ITEM3_2.to_csv(DIRS["metrics"] / "p2_item3_dispersion.csv")

pd.set_option("display.width", 220)
print("\nItem 3 — D_bar in delta_0 units (TV), all pipelines vs Rashomon sets:")
print(ITEM3_2.xs("tv", level="distance")[["n", "delta0", "D_bar", "D_bar/delta0"]].round(4).to_string())
print("\nbetween/within mean distance per component (TV; 1.0 = separates nothing):")
print(ITEM3_2.xs("tv", level="distance")[[f"{f}:between/within" for f in F2_FACTORS]].round(3).to_string())

full factorial: 7750 explained configurations; degenerate (constant output) excluded: 94
      configs  R_2se  R_1se
did                        
31       1577    141     50
37       1602    161     55
44       1562     18      4
1461     1354      8      5
4534     1561     12      9

Item 4 — delta_0 (seed replicates, shared run r = 0; 1321 configurations):
          tv    jac3    jac5     rbo
did                                 
31    0.0879  0.2048  0.1977  0.0871
37    0.0520  0.1013  0.0567  0.0287
44    0.1362  0.2033  0.2263  0.1467
1461  0.0656  0.1279  0.1243  0.0619
4534  0.0456  0.0843  0.0933  0.0494

stochastic configurations only:
          tv    jac3    jac5     rbo
did                                 
31    0.0879  0.2048  0.1977  0.0871
37    0.0765  0.1488  0.0833  0.0422
44    0.1362  0.2033  0.2263  0.1467
1461  0.0656  0.1279  0.1243  0.0619
4534  0.0456  0.0843  0.0933  0.0494

share of deterministic configurations: {31: 0.0, 37: 0.32, 44: 0.0, 1461: 0.0, 4534: 0.

In [21]:
# =============================================================================
# CELL 13 — Item 8: PERMANOVA on the pipeline-level distance matrix
#   Python implementation of vegan::adonis2 (McArdle & Anderson 2001):
#   G = centred(-1/2 D^2); SS(model) = tr(H G); sequential ("terms") and marginal
#   ("margin") sums of squares; F = (SS_t/df_t)/(SS_res/df_res); 9,999 permutations
#   of observations.  m_int: classifier*preprocessor + balancing (by terms,
#   interaction last).  m_main: classifier + preprocessor + balancing (by margin).
# =============================================================================
from scipy.linalg import orth

N_PERM_8 = 9999

# ---------------------------------------------------------------- 0. determinism check
DET_CLF = {"lda", "qda", "gaussian_nb", "bernoulli_nb", "k_nearest_neighbors"}
DET_PP = {"no_preprocessing", "select_percentile_classification", "select_rates_classification",
          "feature_agglomeration", "polynomial"}
chk = []
for r_ in P2[P2.classifier.isin(DET_CLF) & P2.preprocessor.isin(DET_PP)].itertuples():
    z0, z1 = np.load(p2_path(r_.did, r_.config_id, 0)), np.load(p2_path(r_.did, r_.config_id, 1))
    same_model = np.array_equal(z0["pE"], z1["pE"])
    chk.append(dict(did=r_.did, same_model=same_model,
                    same_expl_given_same_model=(np.array_equal(z0["m"], z1["m"]) if same_model else np.nan)))
chk = pd.DataFrame(chk)
print("Determinism check (deterministic learner + deterministic preprocessor, seed 0 vs seed 1):")
print(chk.groupby("did").agg(n=("same_model", "size"), identical_models=("same_model", "mean"),
                             identical_expl_when_identical_model=("same_expl_given_same_model", "mean"))
      .round(2).to_string())

# ---------------------------------------------------------------- 1. PERMANOVA core
def gower_centre(D):
    A = -0.5 * np.asarray(D, dtype=np.float64) ** 2
    return A - A.mean(0, keepdims=True) - A.mean(1, keepdims=True) + A.mean()


def dummies(values):
    codes, uniq = pd.factorize(pd.Series(values).astype(str), sort=True)
    M = np.zeros((len(codes), len(uniq)))
    M[np.arange(len(codes)), codes] = 1.0
    return M[:, 1:]                                      # treatment contrasts, as R


def build_blocks(meta, with_interaction):
    blocks = []
    for f in ("classifier", "preprocessor", "balancing"):
        B = dummies(meta[f].to_numpy())
        if B.shape[1] > 0:
            blocks.append((f, B))
    names = [b[0] for b in blocks]
    if with_interaction and "classifier" in names and "preprocessor" in names:
        Bc, Bp = dict(blocks)["classifier"], dict(blocks)["preprocessor"]
        blocks.append(("classifier:preprocessor",
                       np.einsum("ni,nj->nij", Bc, Bp).reshape(len(meta), -1)))
    return blocks


def hat(X):
    Q = orth(X)
    return Q @ Q.T, Q.shape[1]


def permanova(D, meta, by, n_perm=N_PERM_8, seed=0):
    n = len(meta)
    G = gower_centre(D)
    ss_tot = float(np.trace(G))
    if ss_tot <= 1e-15:                       # every explanation identical under this distance
        return pd.DataFrame([dict(term="(no dispersion: all identical)", df=np.nan, SS=0.0,
                                  R2=np.nan, F=np.nan, p=np.nan)])
    one = np.ones((n, 1))
    blocks = build_blocks(meta, with_interaction=(by == "terms"))
    H_full, r_full = hat(np.hstack([one] + [b for _, b in blocks]))
    df_res = n - r_full
    if by == "terms":
        Hs, ranks, cols = [], [1], [one]
        for _, B in blocks:
            cols.append(B)
            H, r = hat(np.hstack(cols))
            Hs.append(H); ranks.append(r)
        terms = [(name, Hs[i], Hs[i - 1] if i else None, ranks[i + 1] - ranks[i])
                 for i, (name, _) in enumerate(blocks)]
    else:
        terms = []
        for j, (name, _) in enumerate(blocks):
            H_red, r_red = hat(np.hstack([one] + [b for i, (_, b) in enumerate(blocks) if i != j]))
            terms.append((name, H_full, H_red, r_full - r_red))
    terms = [t for t in terms if t[3] > 0]

    def stats(Gm):
        tr_full = float(np.vdot(H_full, Gm))
        ss_res = ss_tot - tr_full
        out = []
        for name, Hhi, Hlo, df in terms:
            ss = float(np.vdot(Hhi, Gm)) - (float(np.vdot(Hlo, Gm)) if Hlo is not None else 0.0)
            out.append((ss, (ss / df) / (ss_res / df_res) if df_res > 0 and ss_res > 0 else np.nan))
        return out, ss_res

    obs, ss_res = stats(G)
    rng = np.random.default_rng(seed)
    exceed = np.zeros(len(terms))
    for _ in range(n_perm):
        p = rng.permutation(n)
        res, _ = stats(G[np.ix_(p, p)])
        exceed += np.array([r[1] >= o[1] - 1e-12 for r, o in zip(res, obs)])
    rows = [dict(term=name, df=df, SS=ss, R2=ss / ss_tot, F=F, p=(1 + e) / (n_perm + 1))
            for (name, _, _, df), (ss, F), e in zip(terms, obs, exceed)]
    rows.append(dict(term="Residual", df=df_res, SS=ss_res, R2=ss_res / ss_tot, F=np.nan, p=np.nan))
    rows.append(dict(term="Total", df=n - 1, SS=ss_tot, R2=1.0, F=np.nan, p=np.nan))
    return pd.DataFrame(rows)

# ---------------------------------------------------------------- 2. validation
rng = np.random.default_rng(derive_seed("permanova-tests"))
n_t = 120
meta_t = pd.DataFrame(dict(classifier=rng.choice(list("abcd"), n_t), preprocessor=rng.choice(list("xyz"), n_t),
                           balancing=rng.choice(["none", "weighting"], n_t)))
Y = rng.normal(size=(n_t, 5)) + (meta_t.classifier == "a").to_numpy()[:, None] * 1.5
D_t = cdist(Y, Y)
tab = permanova(D_t, meta_t, "terms", n_perm=99)
assert abs(tab.SS.iloc[:-1].sum() - tab.SS.iloc[-1]) < 1e-8, "sequential SS + residual != total"
Yc = Y - Y.mean(0)                                          # Euclidean: equals the MANOVA SS
grp = meta_t.classifier.to_numpy()
ss_between = sum((grp == g).sum() * np.sum((Y[grp == g].mean(0) - Y.mean(0)) ** 2) for g in np.unique(grp))
assert abs(tab.loc[tab.term == "classifier", "SS"].item() - ss_between) < 1e-8, "MANOVA identity failed"
assert abs(tab.loc[tab.term == "Total", "SS"].item() - np.sum(Yc ** 2)) < 1e-8
perm = rng.permutation(n_t)
tab_p = permanova(D_t[np.ix_(perm, perm)], meta_t.iloc[perm].reset_index(drop=True), "terms", n_perm=99)
assert np.allclose(tab.SS.to_numpy(), tab_p.SS.to_numpy()), "not invariant to row order"
null_p = [permanova(cdist(Z := rng.normal(size=(60, 4)), Z),
                    meta_t.iloc[:60].reset_index(drop=True), "margin", n_perm=99, seed=i).p.iloc[0]
          for i in range(40)]
print(f"\nPERMANOVA validation passed: SS add up, MANOVA identity, row-order invariance; "
      f"mean p under the null = {np.mean(null_p):.2f} (expected ~0.5)")

# ---------------------------------------------------------------- 3. analysis sets
def rashomon_model_level(did, col):
    """Rashomon members x 3 seeds (seeds as replicates, item 8)."""
    mem = P2[(P2.did == did) & P2[col]]
    S, R, meta = [], [], []
    for r_ in mem.itertuples():
        for s, m in enumerate(seed_m(did, r_.config_id)):
            x = normalise(m)
            S.append(x["shares"]); R.append(x["ranks"])
            meta.append(dict(config_id=r_.config_id, seed=s, classifier=r_.classifier,
                             preprocessor=r_.preprocessor, balancing=r_.balancing))
    return np.vstack(S), np.vstack(R), pd.DataFrame(meta)

RUN_DIR = DIRS["metrics"] / "permanova_inputs"
RUN_DIR.mkdir(exist_ok=True)
RUNS = []
for did in CFG.dataset_ids:
    idx_all = pd.read_csv(D2_DIR / f"D_{did}_index.csv")
    for k in DIST_NAMES:
        RUNS.append(dict(did=did, set="all pipelines", distance=k,
                         D_path=str(D2_DIR / f"D_{did}_{k}.npy"), meta=idx_all))
    for col, set_name in (("in_R_2se", "R_2se"), ("in_R_1se", "R_1se")):
        S, R, meta = rashomon_model_level(did, col)
        if len(meta) < 4:
            continue
        Dm = pairwise_D(S, R)
        for k in DIST_NAMES:
            p = RUN_DIR / f"D_{did}_{set_name}_{k}.npy"
            np.save(p, Dm[k])
            RUNS.append(dict(did=did, set=set_name, distance=k, D_path=str(p), meta=meta))


def _run(job, by):
    D = np.load(job["D_path"]).astype(np.float64)
    t = permanova(D, job["meta"], by, seed=derive_seed("permanova", job["did"], job["set"], job["distance"], by))
    return t.assign(did=job["did"], set=job["set"], distance=job["distance"], model=by, n=len(job["meta"]))

fresh_pool()
t0 = time.time()
res = Parallel(n_jobs=min(os.cpu_count() - 1, 2 * len(RUNS)), verbose=0)(
    delayed(_run)(j, by) for j in RUNS for by in ("terms", "margin"))
PERM8 = pd.concat(res, ignore_index=True)
PERM8.to_csv(DIRS["metrics"] / "item8_permanova.csv", index=False)
print(f"\n{len(res)} PERMANOVA runs x {N_PERM_8} permutations in {(time.time() - t0) / 60:.1f} min")

# ---------------------------------------------------------------- 4. report
def show(model, distance="tv"):
    t = PERM8[(PERM8.model == model) & (PERM8.distance == distance) & (~PERM8.term.isin(["Total"]))]
    t = t.assign(cell=t.R2.map("{:.3f}".format) + np.where(t.p.isna(), "", " (p=" + t.p.map("{:.4f}".format) + ")"))
    return t.pivot_table(index=["did", "set", "n"], columns="term", values="cell", aggfunc="first")

pd.set_option("display.width", 250)
print("\nItem 8 — m_int (by terms, interaction last), TV: R^2 (permutation p)")
print(show("terms").to_string())
print("\nItem 8 — m_main (by margin), TV: R^2 (permutation p)")
print(show("margin").to_string())
print("\n(all four distances are saved in item8_permanova.csv)")

Determinism check (deterministic learner + deterministic preprocessor, seed 0 vs seed 1):
       n  identical_models identical_expl_when_identical_model
did                                                           
31    27              0.89                                 1.0
37    31              0.97                                 1.0
44    21              0.95                                 1.0
1461  27              0.78                                 1.0
4534  23              0.87                                 1.0

PERMANOVA validation passed: SS add up, MANOVA identity, row-order invariance; mean p under the null = 0.49 (expected ~0.5)

120 PERMANOVA runs x 9999 permutations in 4.5 min

Item 8 — m_int (by terms, interaction last), TV: R^2 (permutation p)
term                    Residual         balancing        classifier classifier:preprocessor      preprocessor
did  set           n                                                                                          
3

In [22]:
# =============================================================================
# CELL 14 — Item 9: Shapley (LMG) variance shares over {classifier, preprocessor,
#           balancing}; interaction reported separately; paired effect sizes in
#           delta_0 units; seed-0 vs 3-seed check of the shares
# =============================================================================
from math import factorial

MAIN = ["classifier", "preprocessor", "balancing"]


def r2_of(G, tr_G, meta, subset, interaction=False):
    n = len(meta)
    cols = [np.ones((n, 1))]
    blocks = {f: dummies(meta[f].to_numpy()) for f in MAIN}
    cols += [blocks[f] for f in subset if blocks[f].shape[1] > 0]
    if interaction and blocks["classifier"].shape[1] and blocks["preprocessor"].shape[1]:
        cols.append(np.einsum("ni,nj->nij", blocks["classifier"], blocks["preprocessor"]).reshape(n, -1))
    H, _ = hat(np.hstack(cols))
    return float(np.vdot(H, G)) / tr_G


def shapley_shares(D, meta):
    G = gower_centre(D)
    tr_G = float(np.trace(G))
    if tr_G <= 1e-15:
        return None
    r2 = {frozenset(S): r2_of(G, tr_G, meta, S) for k in range(len(MAIN) + 1)
          for S in combinations(MAIN, k)}
    F = len(MAIN)
    phi = {}
    for c in MAIN:
        others = [f for f in MAIN if f != c]
        phi[c] = sum(factorial(len(S)) * factorial(F - len(S) - 1) / factorial(F)
                     * (r2[frozenset(S) | {c}] - r2[frozenset(S)])
                     for k in range(len(others) + 1) for S in combinations(others, k))
    total = r2[frozenset(MAIN)]
    assert abs(sum(phi.values()) - total) < 1e-9, "Shapley shares do not sum to R^2(F)"
    inter = r2_of(G, tr_G, meta, MAIN, interaction=True) - total
    return dict(**{f"phi_{c}": v for c, v in phi.items()}, R2_main=total, interaction=inter)


def _shap_run(job):
    D = np.load(job["D_path"]).astype(np.float64)
    out = shapley_shares(D, job["meta"])
    base = dict(did=job["did"], set=job["set"], distance=job["distance"], n=len(job["meta"]))
    return {**base, **(out or {"note": "no dispersion: all identical"})}

fresh_pool()
SHAP9 = pd.DataFrame(Parallel(n_jobs=min(os.cpu_count() - 1, len(RUNS)))(delayed(_shap_run)(j) for j in RUNS))
SHAP9["D_bar/delta0"] = [ITEM3_2.loc[(r.did, r.set, r.distance), "D_bar/delta0"]
                         if (r.did, r.set, r.distance) in ITEM3_2.index else np.nan for r in SHAP9.itertuples()]
SHAP9.to_csv(DIRS["metrics"] / "item9_shapley_shares.csv", index=False)

def winner(r):
    vals = {c: r[f"phi_{c}"] for c in MAIN if pd.notna(r.get(f"phi_{c}"))}
    return max(vals, key=vals.get) if vals else "-"

t = SHAP9[SHAP9.distance == "tv"].copy()
t["largest"] = t.apply(winner, axis=1)
for c in MAIN:
    t[f"{c} %"] = 100 * t[f"phi_{c}"] / t.R2_main
pd.set_option("display.width", 250)
print("Item 9 — Shapley shares (TV). Shares sum to R2_main; interaction reported separately.")
print(t.set_index(["did", "set", "n"])[[f"phi_{c}" for c in MAIN] + ["R2_main", "interaction",
      "largest", "D_bar/delta0"] + [f"{c} %" for c in MAIN]].round(3).to_string())

print("\nLargest component, every distance:")
w = SHAP9.copy()
w["largest"] = w.apply(winner, axis=1)
print(w.pivot_table(index=["did", "set"], columns="distance", values="largest", aggfunc="first")[DIST_NAMES].to_string())

# ---------------------------------------------------------------------------
# Paired effect sizes in delta_0 units (all pipelines; paired design, item 6)
#   effect of component c = mean distance between pipelines identical in every
#   other design element (same pair index -> same hyperparameter draws) but c
# ---------------------------------------------------------------------------
eff_rows = []
for did in CFG.dataset_ids:
    meta = pd.read_csv(D2_DIR / f"D_{did}_index.csv")
    Dtv = np.load(D2_DIR / f"D_{did}_tv.npy").astype(np.float64)
    for c in MAIN:
        keys = [f for f in MAIN if f != c] + ["pair_idx"]
        dd = []
        for _, g in meta.groupby(keys):
            ix = g.index.to_numpy()
            if len(ix) > 1:
                sub = Dtv[np.ix_(ix, ix)]
                dd.extend(sub[np.triu_indices(len(ix), 1)])
        d0 = DELTA0_2.loc[did, "tv"]
        eff_rows.append(dict(did=did, component=c, n_pairs=len(dd), mean_TV=np.mean(dd),
                             in_delta0_units=np.mean(dd) / d0))
EFF = pd.DataFrame(eff_rows)
EFF.to_csv(DIRS["metrics"] / "item4_8_paired_effects_delta0.csv", index=False)
print("\nPaired effect sizes (TV) in delta_0 units — pipelines differing ONLY in the component:")
print(EFF.pivot(index="did", columns="component", values="in_delta0_units")[MAIN].round(2).to_string())
print("(1 = indistinguishable from training-seed noise)")

# ---------------------------------------------------------------------------
# Seed check: shares with seed 0 only vs 3 seeds as replicates (same configurations)
# ---------------------------------------------------------------------------
chk = []
for did in CFG.dataset_ids:
    cfgs = P2[P2.did == did]
    S0, R0, S3, R3, m0, m3 = [], [], [], [], [], []
    for r_ in cfgs.itertuples():
        N = [normalise(m) for m in seed_m(did, r_.config_id)]
        if any(x["degenerate"] for x in N):
            continue
        base = dict(classifier=r_.classifier, preprocessor=r_.preprocessor, balancing=r_.balancing)
        S0.append(N[0]["shares"]); R0.append(N[0]["ranks"]); m0.append(base)
        for x in N:
            S3.append(x["shares"]); R3.append(x["ranks"]); m3.append(base)
    a = shapley_shares(pairwise_D(np.vstack(S0), np.vstack(R0))["tv"], pd.DataFrame(m0))
    b = shapley_shares(pairwise_D(np.vstack(S3), np.vstack(R3))["tv"], pd.DataFrame(m3))
    for lab, res in (("seed 0 only", a), ("3 seeds", b)):
        chk.append(dict(did=did, version=lab, n=len(m0) if lab == "seed 0 only" else len(m3),
                        **{c: res[f"phi_{c}"] for c in MAIN},
                        largest=max(MAIN, key=lambda c: res[f"phi_{c}"])))
CHK = pd.DataFrame(chk)
CHK.to_csv(DIRS["metrics"] / "item9_seed_check.csv", index=False)
print("\nSeed check (TV Shapley shares on the 1,321 configurations with 3 seeds):")
print(CHK.set_index(["did", "version"]).round(3).to_string())
same = CHK.groupby("did").largest.nunique().eq(1).all()
print(f"\nlargest component identical with seed 0 only and with 3 seeds on every dataset: {same}")

Item 9 — Shapley shares (TV). Shares sum to R2_main; interaction reported separately.
                         phi_classifier  phi_preprocessor  phi_balancing  R2_main  interaction       largest  D_bar/delta0  classifier %  preprocessor %  balancing %
did  set           n                                                                                                                                                 
37   all pipelines 1602           0.055             0.461          0.002    0.518        0.059  preprocessor         6.370        10.631          89.020        0.349
     R_2se         483            0.212             0.525          0.006    0.743        0.080  preprocessor         4.674        28.552          70.633        0.815
     R_1se         165            0.254             0.483          0.008    0.745        0.071  preprocessor         3.034        34.075          64.801        1.124
31   all pipelines 1577           0.113             0.394          0.001    0.508   

In [23]:
# =============================================================================
# CELL 15 — Item 7 deliverable: Shapley shares, unconditional vs Rashomon
#           (epsilon = 1 SE and 2 SE), with resolvability against delta_0
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RESOLVE = 1.2                      # same threshold as Phase 1 (our choice; stated)
SETS = ["all pipelines", "R_2se", "R_1se"]

F15 = SHAP9[SHAP9.distance == "tv"].copy()
F15["n_classifiers"] = [
    (pd.read_csv(D2_DIR / f"D_{r.did}_index.csv") if r.set == "all pipelines" else
     P2[(P2.did == r.did) & P2["in_" + r.set]]).classifier.nunique() for r in F15.itertuples()]
F15["resolvable"] = F15["D_bar/delta0"] >= RESOLVE
for c in MAIN:
    F15[f"{c} %"] = 100 * F15[f"phi_{c}"] / F15.R2_main
F15["largest"] = F15.apply(winner, axis=1)

def verdict(r):
    if not r.resolvable:
        return "explanations agree within noise: no component materially changes them"
    if r.n_classifiers < 2:
        return f"{r.largest} largest (single classifier: classifier effect not measurable)"
    return f"{r.largest} contributes most"
F15["verdict"] = F15.apply(verdict, axis=1)

FINAL = F15.set_index(["did", "set"]).loc[[(d, s) for d in CFG.dataset_ids for s in SETS]]
FINAL[["n", "n_classifiers", "D_bar/delta0", "resolvable", "classifier %", "preprocessor %",
       "balancing %", "R2_main", "interaction", "verdict"]].to_csv(DIRS["metrics"] / "FINAL_item7_9_shares.csv")
pd.set_option("display.width", 250, "display.max_colwidth", 90)
print(FINAL[["n", "n_classifiers", "D_bar/delta0", "classifier %", "preprocessor %", "balancing %",
             "interaction", "verdict"]].round(2).to_string())

# ---- figure: stacked shares per dataset and set -----------------------------------
colors = {"classifier": "#3b6ea8", "preprocessor": "#d9822b", "balancing": "#6aa84f"}
fig, axes = plt.subplots(1, len(CFG.dataset_ids), figsize=(4 * len(CFG.dataset_ids), 4.2), sharey=True)
for ax, did in zip(axes, CFG.dataset_ids):
    for i, s in enumerate(SETS):
        r = FINAL.loc[(did, s)]
        bottom, alpha = 0.0, (1.0 if r.resolvable else 0.3)
        for c in MAIN:
            v = r[f"phi_{c}"]
            ax.bar(i, v, bottom=bottom, color=colors[c], alpha=alpha, label=c if (did == CFG.dataset_ids[0] and i == 0) else None)
            bottom += v
        ax.bar(i, r.interaction, bottom=bottom, color="lightgrey", hatch="//", alpha=alpha,
               label="classifier x preprocessor" if (did == CFG.dataset_ids[0] and i == 0) else None)
        if not r.resolvable:
            ax.text(i, bottom + r.interaction + 0.02, "≈ noise", ha="center", fontsize=8)
    ax.set_xticks(range(len(SETS)))
    ax.set_xticklabels(["all", "R 2SE", "R 1SE"])
    ax.set_title(f"{DATA[did].name} ({did})")
axes[0].set_ylabel("R² (Shapley shares, TV)")
fig.legend(loc="upper center", ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
for ext in ("png", "pdf"):
    fig.savefig(DIRS["figures"] / f"item7_9_shapley_shares_uncond_vs_rashomon.{ext}", dpi=200)
print(f"\nfigure saved to {DIRS['figures']}")

                       n  n_classifiers  D_bar/delta0  classifier %  preprocessor %  balancing %  interaction                                                                verdict
did  set                                                                                                                                                                            
37   all pipelines  1602             16          6.37         10.63           89.02         0.35         0.06                                          preprocessor contributes most
     R_2se           483              9          4.67         28.55           70.63         0.82         0.08                                          preprocessor contributes most
     R_1se           165              7          3.03         34.07           64.80         1.12         0.07                                          preprocessor contributes most
31   all pipelines  1577             16          4.43         22.31           77.55         0.1

In [24]:
# =============================================================================
# CELL 16 — Item 10: model family vs hyperparameters
#   Nested PERMANOVA: family, then classifier configuration within family
#   (classifier configuration = classifier + paired draw index, item 6).
#   Hypothesis: within-family dispersion large for flexible learners
#   (gradient_boosting, mlp, libsvm_svc), small for low-capacity ones (lda, NB).
# =============================================================================
N_PERM_10 = 9999
FLEXIBLE = {"gradient_boosting", "mlp", "libsvm_svc"}
LOW_CAP = {"lda", "qda", "gaussian_nb", "bernoulli_nb", "multinomial_nb"}


def permanova_seq(D, blocks, n_perm, seed):
    """Sequential PERMANOVA for an arbitrary ordered list of (name, design block)."""
    n = len(D)
    G = gower_centre(D)
    ss_tot = float(np.trace(G))
    if ss_tot <= 1e-15:
        return None
    cols, Hs, ranks = [np.ones((n, 1))], [], [1]
    for _, B in blocks:
        cols.append(B)
        H, r = hat(np.hstack(cols))
        Hs.append(H); ranks.append(r)
    df_res = n - ranks[-1]
    H_full = Hs[-1]

    def stats(Gm):
        ss_res = ss_tot - float(np.vdot(H_full, Gm))
        prev, out = 0.0, []
        for i, H in enumerate(Hs):
            cur = float(np.vdot(H, Gm))
            df = ranks[i + 1] - ranks[i]
            ss = cur - prev
            prev = cur
            out.append((ss, df, (ss / df) / (ss_res / df_res) if df > 0 and df_res > 0 and ss_res > 0 else np.nan))
        return out, ss_res

    obs, ss_res = stats(G)
    rng = np.random.default_rng(seed)
    exceed = np.zeros(len(obs))
    for _ in range(n_perm):
        p = rng.permutation(n)
        res, _ = stats(G[np.ix_(p, p)])
        exceed += np.array([(r[2] >= o[2] - 1e-12) if np.isfinite(o[2]) else 0 for r, o in zip(res, obs)])
    rows = [dict(term=name, df=df, R2=ss / ss_tot, F=F, p=(1 + e) / (n_perm + 1) if df > 0 else np.nan)
            for (name, _), (ss, df, F), e in zip(blocks, obs, exceed)]
    rows.append(dict(term="Residual", df=df_res, R2=ss_res / ss_tot, F=np.nan, p=np.nan))
    return pd.DataFrame(rows)


def nested_blocks(meta, order):
    m = meta.assign(clf_cfg=meta.classifier.astype(str) + "#" + meta.pair_idx.astype(int).astype(str))
    b = {"family": dummies(m.classifier.to_numpy()),
         "configuration within family": dummies(m.clf_cfg.to_numpy()),
         "preprocessor": dummies(m.preprocessor.to_numpy()),
         "balancing": dummies(m.balancing.to_numpy())}
    names = (["family", "configuration within family", "preprocessor", "balancing"] if order == "family first"
             else ["preprocessor", "balancing", "family", "configuration within family"])
    return [(nm, b[nm]) for nm in names if b[nm].shape[1] > 0]


# ---- analysis sets: all pipelines (seed 0) and resolvable Rashomon sets (3 seeds) ---
pair_of = {(r.did, r.config_id): r.pair_idx for r in P2.itertuples()}
JOBS10 = []
for did in CFG.dataset_ids:
    meta = pd.read_csv(D2_DIR / f"D_{did}_index.csv")
    JOBS10.append(dict(did=did, set="all pipelines", D_path=str(D2_DIR / f"D_{did}_tv.npy"), meta=meta))
    for s in ("R_2se", "R_1se"):
        if not FINAL.loc[(did, s), "resolvable"] or FINAL.loc[(did, s), "n_classifiers"] < 2:
            continue
        S, R, mm = rashomon_model_level(did, "in_" + s)
        mm["pair_idx"] = [pair_of[(did, c)] for c in mm.config_id]
        p = RUN_DIR / f"D_{did}_{s}_tv.npy"          # saved by Cell 13
        JOBS10.append(dict(did=did, set=s, D_path=str(p), meta=mm))


def _run10(job, order):
    D = np.load(job["D_path"]).astype(np.float64)
    t = permanova_seq(D, nested_blocks(job["meta"], order), N_PERM_10,
                      derive_seed("item10", job["did"], job["set"], order))
    return t.assign(did=job["did"], set=job["set"], order=order, n=len(job["meta"]))

fresh_pool()
t0 = time.time()
NEST = pd.concat(Parallel(n_jobs=min(os.cpu_count() - 1, 2 * len(JOBS10)))(
    delayed(_run10)(j, o) for j in JOBS10 for o in ("family first", "preprocessing first")), ignore_index=True)
NEST.to_csv(DIRS["metrics"] / "item10_nested_permanova.csv", index=False)
print(f"{2 * len(JOBS10)} nested PERMANOVA runs in {(time.time() - t0) / 60:.1f} min\n")

piv = NEST.assign(cell=NEST.R2.map("{:.3f}".format) + np.where(NEST.p.isna(), "", " (p=" + NEST.p.map("{:.4f}".format) + ")"))
pd.set_option("display.width", 250)
for order in ("family first", "preprocessing first"):
    print(f"Item 10 — nested PERMANOVA (TV), order: {order}")
    print(piv[piv.order == order].pivot_table(index=["did", "set", "n"], columns="term", values="cell",
                                              aggfunc="first").to_string(), "\n")

ff = NEST[NEST.order == "family first"].pivot_table(index=["did", "set"], columns="term", values="R2")
ff["hyperparameter % of classifier part"] = 100 * ff["configuration within family"] / (
    ff["family"] + ff["configuration within family"])
print("Classifier-attributable dispersion: family vs hyperparameters (family-first order)")
print(ff[["family", "configuration within family", "hyperparameter % of classifier part"]].round(3).to_string())

# ---- hypothesis: within-family dispersion per family, hyperparameters ONLY differ ----
rows = []
for did in CFG.dataset_ids:
    meta = pd.read_csv(D2_DIR / f"D_{did}_index.csv")
    Dtv = np.load(D2_DIR / f"D_{did}_tv.npy").astype(np.float64)
    sub = meta[meta.preprocessor == "no_preprocessing"]          # no preprocessor hyperparameters
    for (clf, bal), g in sub.groupby(["classifier", "balancing"]):
        ix = g.index.to_numpy()
        if len(ix) > 1:
            d = Dtv[np.ix_(ix, ix)][np.triu_indices(len(ix), 1)]
            rows.append(dict(did=did, classifier=clf, balancing=bal, mean_TV=d.mean()))
WF = pd.DataFrame(rows).groupby(["did", "classifier"]).mean_TV.mean().reset_index()
WF["in_delta0_units"] = WF.mean_TV / WF.did.map(DELTA0_2["tv"])
WF["group"] = np.where(WF.classifier.isin(FLEXIBLE), "flexible",
                       np.where(WF.classifier.isin(LOW_CAP), "low-capacity", "other"))
WF.to_csv(DIRS["metrics"] / "item10_within_family_dispersion.csv", index=False)
print("\nWithin-family dispersion from hyperparameters only (no_preprocessing, same balancing), delta_0 units:")
print(WF.pivot_table(index=["group", "classifier"], columns="did", values="in_delta0_units").round(2).to_string())
grp = WF.groupby(["did", "group"]).in_delta0_units.mean().unstack()
print("\nGroup means (delta_0 units):")
print(grp.round(2).to_string())
print("\nhypothesis 'flexible > low-capacity' holds on datasets:",
      [d for d in grp.index if grp.loc[d, "flexible"] > grp.loc[d, "low-capacity"]])

18 nested PERMANOVA runs in 2.5 min

Item 10 — nested PERMANOVA (TV), order: family first
term                    Residual         balancing configuration within family            family      preprocessor
did  set           n                                                                                              
31   R_1se         150     0.190  0.006 (p=0.0099)            0.147 (p=0.0001)  0.330 (p=0.0001)  0.328 (p=0.0001)
     R_2se         423     0.271  0.000 (p=0.8111)            0.130 (p=0.0001)  0.209 (p=0.0001)  0.390 (p=0.0001)
     all pipelines 1577    0.393  0.001 (p=0.0303)            0.099 (p=0.0001)  0.118 (p=0.0001)  0.388 (p=0.0001)
37   R_1se         165     0.158  0.003 (p=0.0371)            0.210 (p=0.0001)  0.285 (p=0.0001)  0.343 (p=0.0001)
     R_2se         483     0.199  0.006 (p=0.0001)            0.104 (p=0.0001)  0.332 (p=0.0001)  0.359 (p=0.0001)
     all pipelines 1602    0.401  0.002 (p=0.0013)            0.082 (p=0.0001)  0.056 (p=0.0001)  0.459 (

In [25]:
# =============================================================================
# CELL 16b — Item 10 corrected: exact configuration identifiers
#   classifier hyperparameters  = clf_config_id  (nested in family)
#   preprocessor hyperparameters = fp_config_id  (nested in preprocessor type)
#   + Shapley shares with whole components as players:
#     classifier (family + hyperparameters), preprocessor (type + hyperparameters), balancing
# =============================================================================
IDS = DESIGN[DESIGN.status == "sampled"].set_index(["did", "config_id"])[["clf_config_id", "fp_config_id"]]

def add_ids(meta, did):
    ids = IDS.loc[[(did, c) for c in meta.config_id]]
    return meta.assign(clf_cfg=ids.clf_config_id.to_numpy(), pp_cfg=ids.fp_config_id.to_numpy())

def blocks_full(meta):
    return {"classifier family": dummies(meta.classifier.to_numpy()),
            "classifier hyperparameters (within family)": dummies(meta.clf_cfg.to_numpy()),
            "preprocessor type": dummies(meta.preprocessor.to_numpy()),
            "preprocessor hyperparameters (within type)": dummies(meta.pp_cfg.to_numpy()),
            "balancing": dummies(meta.balancing.to_numpy())}

ORDERS = {
    "classifier first": ["classifier family", "classifier hyperparameters (within family)",
                         "preprocessor type", "preprocessor hyperparameters (within type)", "balancing"],
    "preprocessor first": ["preprocessor type", "preprocessor hyperparameters (within type)", "balancing",
                           "classifier family", "classifier hyperparameters (within family)"],
}
PLAYERS = {"classifier (family + hyperparameters)": ["classifier family", "classifier hyperparameters (within family)"],
           "preprocessor (type + hyperparameters)": ["preprocessor type", "preprocessor hyperparameters (within type)"],
           "balancing": ["balancing"]}


def component_shapley(D, b):
    n = len(D)
    G = gower_centre(D)
    trG = float(np.trace(G))
    if trG <= 1e-15:
        return None
    def r2(S):
        cols = [np.ones((n, 1))] + [b[t] for p in S for t in PLAYERS[p] if b[t].shape[1] > 0]
        H, _ = hat(np.hstack(cols))
        return float(np.vdot(H, G)) / trG
    P = list(PLAYERS)
    cache = {frozenset(S): r2(S) for k in range(len(P) + 1) for S in combinations(P, k)}
    phi = {}
    for c in P:
        others = [p for p in P if p != c]
        phi[c] = sum(factorial(len(S)) * factorial(len(P) - len(S) - 1) / factorial(len(P))
                     * (cache[frozenset(S) | {c}] - cache[frozenset(S)])
                     for k in range(len(others) + 1) for S in combinations(others, k))
    tot = cache[frozenset(P)]
    assert abs(sum(phi.values()) - tot) < 1e-9
    return dict(**phi, R2_total=tot)


def _run16b(job, order):
    D = np.load(job["D_path"]).astype(np.float64)
    meta = add_ids(job["meta"], job["did"])
    b = blocks_full(meta)
    blocks = [(nm, b[nm]) for nm in ORDERS[order] if b[nm].shape[1] > 0]
    t = permanova_seq(D, blocks, N_PERM_10, derive_seed("item10b", job["did"], job["set"], order))
    out = t.assign(did=job["did"], set=job["set"], order=order, n=len(meta))
    sh = component_shapley(D, b) if order == "classifier first" else None
    return out, (dict(did=job["did"], set=job["set"], n=len(meta), **sh) if sh else None)

fresh_pool()
t0 = time.time()
res = Parallel(n_jobs=min(os.cpu_count() - 1, 2 * len(JOBS10)))(
    delayed(_run16b)(j, o) for j in JOBS10 for o in ORDERS)
NEST2 = pd.concat([r[0] for r in res], ignore_index=True)
COMP = pd.DataFrame([r[1] for r in res if r[1] is not None])
NEST2.to_csv(DIRS["metrics"] / "item10_nested_permanova_corrected.csv", index=False)
COMP.to_csv(DIRS["metrics"] / "item9_10_component_shapley.csv", index=False)
print(f"{len(res)} runs in {(time.time() - t0) / 60:.1f} min\n")

cell = NEST2.assign(c=NEST2.R2.map("{:.3f}".format) + np.where(NEST2.p.isna(), "", " (p=" + NEST2.p.map("{:.4f}".format) + ")"))
pd.set_option("display.width", 260, "display.max_colwidth", 40)
for order in ORDERS:
    print(f"Item 10 (corrected) — sequential PERMANOVA, TV, {order}:")
    print(cell[cell.order == order].pivot_table(index=["did", "set", "n"], columns="term", values="c",
                                                aggfunc="first")[ORDERS[order] + ["Residual"]].to_string(), "\n")

cf = NEST2[NEST2.order == "classifier first"].pivot_table(index=["did", "set"], columns="term", values="R2")
cf["hyperparameter % of classifier part"] = 100 * cf["classifier hyperparameters (within family)"] / (
    cf["classifier family"] + cf["classifier hyperparameters (within family)"])
print("Family vs hyperparameters (classifier-first order):")
print(cf[["classifier family", "classifier hyperparameters (within family)",
          "hyperparameter % of classifier part"]].round(3).to_string())

COMP["largest"] = COMP[list(PLAYERS)].idxmax(axis=1)
for p in PLAYERS:
    COMP[f"{p.split(' ')[0]} %"] = 100 * COMP[p] / COMP.R2_total
print("\nShapley shares with WHOLE components (hyperparameters included on both sides), TV:")
print(COMP.set_index(["did", "set", "n"])[list(PLAYERS) + ["R2_total", "largest"] +
      [f"{p.split(' ')[0]} %" for p in PLAYERS]].round(3).to_string())

18 runs in 2.7 min

Item 10 (corrected) — sequential PERMANOVA, TV, classifier first:
term                    classifier family classifier hyperparameters (within family) preprocessor type preprocessor hyperparameters (within type)         balancing Residual
did  set           n                                                                                                                                                        
31   R_1se         150   0.330 (p=0.0001)                           0.147 (p=0.0001)  0.328 (p=0.0001)                           0.088 (p=0.0001)  0.002 (p=0.1174)    0.106
     R_2se         423   0.209 (p=0.0001)                           0.126 (p=0.0001)  0.390 (p=0.0001)                           0.105 (p=0.0001)  0.001 (p=0.3448)    0.170
     all pipelines 1577  0.118 (p=0.0001)                           0.098 (p=0.0001)  0.388 (p=0.0001)                           0.100 (p=0.0001)  0.001 (p=0.0118)    0.295
37   R_1se         165   0.285 (p=0.0001)        

In [26]:
# =============================================================================
# CELL 17 — Item 11: learner invariances predict where preprocessing matters
#   Predicted-invariant cells should sit at ~delta_0 from the same classifier
#   configuration with no_preprocessing; predicted-changing cells well above.
#   Per-configuration classification (document: "classify each cell per
#   configuration, not per learner name"). Columns "per-feature affine" and
#   "per-feature monotone" are rescalers (pinned) -> not testable here.
# =============================================================================
from scipy.stats import mannwhitneyu

TREES = {"decision_tree", "random_forest", "extra_trees", "gradient_boosting", "adaboost"}
NONLIN = {"kernel_pca", "nystroem_sampler", "kitchen_sinks", "polynomial", "random_trees_embedding"}
SELECTORS = {"feature_agglomeration", "select_percentile_classification", "select_rates_classification",
             "extra_trees_preproc_for_classification", "liblinear_svc_preprocessor"}
TYPES = ["orthogonal rotation", "invertible linear", "information-removing", "nonlinear expansion", "identity"]

# ---------------------------------------------------------------------------
# 1. Transform type of every preprocessor configuration, measured by refitting
# ---------------------------------------------------------------------------
def _measure_fp(did, cid, conf_dict, feat_type, dp, X, y, seed):
    import warnings
    warnings.filterwarnings("ignore")
    from autosklearn.pipeline.classification import SimpleClassificationPipeline
    pipe = SimpleClassificationPipeline(config=conf_dict, feat_type=feat_type, dataset_properties=dp,
                                        random_state=seed)
    Xt, _ = pipe.fit_transformer(X, y)
    d_in = pipe.named_steps["data_preprocessor"].transform(X).shape[1]
    return dict(did=did, config_id=cid, d_in=int(d_in), d_out=int(Xt.shape[1]))

fp_rows = DESIGN[(DESIGN.status == "sampled") & DESIGN.preprocessor.isin({"pca", "fast_ica"} | SELECTORS)]
fp_rep = fp_rows.drop_duplicates(["did", "fp_config_id"])            # one representative per fp config
fresh_pool()
meas = Parallel(n_jobs=min(os.cpu_count() - 1, len(fp_rep)))(
    delayed(_measure_fp)(r.did, r.config_id, rebuild_config(r.did, r.config_id).get_dictionary(),
                         FEAT2[r.did], DP2, DATA[r.did].X[ROWS2[r.did]["train_idx"]],
                         DATA[r.did].y_event[ROWS2[r.did]["train_idx"]], SEEDS2[f"{r.did}|{r.config_id}"][0])
    for r in fp_rep.itertuples())
MEAS = pd.DataFrame(meas).merge(fp_rep[["did", "config_id", "preprocessor", "fp_config_id"]], on=["did", "config_id"])


def fp_type(row):
    name = row.preprocessor
    if name == "no_preprocessing":
        return "identity"
    if name in NONLIN:
        return "nonlinear expansion"
    if name == "pca":
        if row.d_out < row.d_in:
            return "information-removing"
        whiten = str(CONF_JSON[f"{row.did}|{row.config_id}"].get("feature_preprocessor:pca:whiten")) == "True"
        return "invertible linear" if whiten else "orthogonal rotation"
    if name == "fast_ica":
        return "invertible linear" if row.d_out >= row.d_in else "information-removing"
    return "information-removing" if row.d_out < row.d_in else "identity"

MEAS["type"] = MEAS.apply(fp_type, axis=1)
FPTYPE = {(r.did, r.fp_config_id): r.type for r in MEAS.itertuples()}
print("Measured transform types of preprocessor configurations:")
print(MEAS.groupby(["preprocessor", "type"]).size().unstack(fill_value=0).to_string())

# ---------------------------------------------------------------------------
# 2. Learner invariance per classifier configuration (document's table)
# ---------------------------------------------------------------------------
def invariant_to(clf, cd):
    hp = lambda k: str(cd.get(f"classifier:{clf}:{k}"))
    inv = {"identity"}
    if clf == "lda" and hp("shrinkage") == "None":
        inv |= {"orthogonal rotation", "invertible linear"}
    elif clf == "qda" and float(cd.get("classifier:qda:reg_param", 1)) == 0.0:
        inv |= {"orthogonal rotation", "invertible linear"}
    elif clf in ("liblinear_svc", "sgd") and hp("penalty") == "l2":
        inv |= {"orthogonal rotation"}
    elif clf == "passive_aggressive":
        inv |= {"orthogonal rotation"}
    elif clf == "libsvm_svc" and hp("kernel") in ("rbf", "poly", "sigmoid"):
        inv |= {"orthogonal rotation"}                 # inner-product kernels are rotation-invariant
    elif clf == "k_nearest_neighbors" and hp("p") == "2":
        inv |= {"orthogonal rotation"}
    return inv                                          # trees, gaussian_nb: only rescaler columns (untested)

def learner_group(clf):
    if clf in TREES: return "trees / ensembles"
    if clf == "gaussian_nb": return "gaussian NB"
    if clf in ("lda", "qda"): return "LDA / QDA"
    if clf in ("liblinear_svc", "sgd", "passive_aggressive"): return "linear (l2)"
    if clf in ("libsvm_svc", "k_nearest_neighbors"): return "SVM / kNN"
    if clf == "mlp": return "MLP"
    return "Bernoulli / multinomial NB"

# ---------------------------------------------------------------------------
# 3. Paired distances to the same classifier configuration with no_preprocessing
# ---------------------------------------------------------------------------
d0_clf = D0T.groupby(["did", "classifier"]).tv.mean()
pairs = []
for did in CFG.dataset_ids:
    meta = add_ids(pd.read_csv(D2_DIR / f"D_{did}_index.csv"), did)
    Dtv = np.load(D2_DIR / f"D_{did}_tv.npy").astype(np.float64)
    ref = {(r.clf_cfg, r.balancing): i for i, r in meta[meta.preprocessor == "no_preprocessing"].iterrows()}
    for i, r in meta[meta.preprocessor != "no_preprocessing"].iterrows():
        j = ref.get((r.clf_cfg, r.balancing))
        if j is None:
            continue
        t = "nonlinear expansion" if r.preprocessor in NONLIN else FPTYPE.get((did, r.pp_cfg))
        if t is None:
            continue
        cd = CONF_JSON[f"{did}|{r.config_id}"]
        d0 = d0_clf.get((did, r.classifier), DELTA0_2.loc[did, "tv"])
        pairs.append(dict(did=did, classifier=r.classifier, group=learner_group(r.classifier),
                          preprocessor=r.preprocessor, type=t,
                          predicted="invariant" if t in invariant_to(r.classifier, cd) else "changes",
                          tv=Dtv[i, j], in_delta0=Dtv[i, j] / d0 if d0 > 0 else np.nan, row=i))
PAIRS = pd.DataFrame(pairs)
PAIRS.to_csv(DIRS["metrics"] / "item11_paired_invariance.csv", index=False)

print("\nPaired cells by prediction (distance to the same classifier configuration with no_preprocessing):")
summ = PAIRS.groupby(["did", "predicted"]).in_delta0.agg(["size", "median", "mean"]).unstack()
print(summ.round(2).to_string())
for did in CFG.dataset_ids:
    a = PAIRS[(PAIRS.did == did) & (PAIRS.predicted == "invariant")].in_delta0.dropna()
    b = PAIRS[(PAIRS.did == did) & (PAIRS.predicted == "changes")].in_delta0.dropna()
    if len(a) and len(b):
        p = mannwhitneyu(a, b, alternative="less").pvalue
        print(f"  {did}: invariant median {a.median():.2f} delta_0 (n={len(a)}) vs changing "
              f"median {b.median():.2f} delta_0 (n={len(b)}); one-sided Mann-Whitney p = {p:.2e}")

# ---------------------------------------------------------------------------
# 4. Predicted vs observed: learner group x transform type (all datasets)
# ---------------------------------------------------------------------------
obs = PAIRS.pivot_table(index="group", columns="type", values="in_delta0", aggfunc="median")
pred = PAIRS.pivot_table(index="group", columns="type", values="predicted",
                         aggfunc=lambda s: f"I {100 * (s == 'invariant').mean():.0f}%")
cnt = PAIRS.pivot_table(index="group", columns="type", values="tv", aggfunc="size")
cols = [t for t in TYPES if t in obs.columns]
print("\nObserved median distance (delta_0 units), learner group x transform type:")
print(obs[cols].round(2).to_string())
print("\nShare of cells predicted invariant:")
print(pred.reindex(columns=cols).to_string())
print("\nNumber of paired cells:")
print(cnt.reindex(columns=cols).fillna(0).astype(int).to_string())

fig, ax = plt.subplots(figsize=(1.9 * len(cols) + 2, 0.6 * len(obs) + 1.5))
im = ax.imshow(obs[cols].to_numpy(dtype=float), cmap="viridis", aspect="auto")
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=25, ha="right")
ax.set_yticks(range(len(obs))); ax.set_yticklabels(obs.index)
for y_, g in enumerate(obs.index):
    for x_, t in enumerate(cols):
        v = obs.loc[g, t]
        if pd.notna(v):
            ax.text(x_, y_, f"{v:.1f}\n{pred.loc[g, t] if t in pred.columns else ''}", ha="center",
                    va="center", fontsize=7, color="white")
fig.colorbar(im, ax=ax, label="median distance to no_preprocessing (delta_0 units)")
ax.set_title("Item 11: observed effect vs predicted invariance (I % = share predicted invariant)")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(DIRS["figures"] / f"item11_invariance_heatmap.{ext}", dpi=200)

# ---------------------------------------------------------------------------
# 5. Preprocessor's Shapley share over predicted-changing cells only
# ---------------------------------------------------------------------------
rows = []
for did in CFG.dataset_ids:
    meta = pd.read_csv(D2_DIR / f"D_{did}_index.csv")
    Dtv = np.load(D2_DIR / f"D_{did}_tv.npy").astype(np.float64)
    changing = set(PAIRS[(PAIRS.did == did) & (PAIRS.predicted == "changes")].row)
    for lab, idx in (("all cells", np.arange(len(meta))), ("predicted-changing cells only", np.array(sorted(changing)))):
        sh = shapley_shares(Dtv[np.ix_(idx, idx)], meta.iloc[idx].reset_index(drop=True))
        rows.append(dict(did=did, cells=lab, n=len(idx), **{c: sh[f"phi_{c}"] for c in MAIN},
                         **{f"{c} %": 100 * sh[f"phi_{c}"] / sh["R2_main"] for c in MAIN}))
SH11 = pd.DataFrame(rows)
SH11.to_csv(DIRS["metrics"] / "item11_shapley_changing_cells.csv", index=False)
print("\nShapley shares (TV): all cells vs predicted-changing cells only")
print(SH11.set_index(["did", "cells", "n"]).round(3).to_string())
print(f"\nheatmap saved to {DIRS['figures']}")

Measured transform types of preprocessor configurations:
type                                    identity  information-removing  invertible linear
preprocessor                                                                             
extra_trees_preproc_for_classification         0                    25                  0
fast_ica                                       0                     0                 23
feature_agglomeration                         24                     1                  0
liblinear_svc_preprocessor                     0                    25                  0
pca                                            0                    25                  0
select_percentile_classification               0                    25                  0
select_rates_classification                    2                    23                  0

Paired cells by prediction (distance to the same classifier configuration with no_preprocessing):
             size            media

In [27]:
# =============================================================================
# CELL 17b — Item 11 with corrected invariance rules, reported NEXT TO the
#            pre-registered rules (both are kept; corrections are data-triggered)
#   (1) fast_ica with whiten=False uses an orthonormal unmixing matrix -> orthogonal rotation
#   (2) LDA shrinkage / QDA reg_param shrink toward c*I, which commutes with rotations
#       -> still rotation-invariant (not invariant to general invertible linear maps)
# =============================================================================
def fp_type_v2(row):
    t = fp_type(row)
    if row.preprocessor == "fast_ica" and t == "invertible linear":
        if str(CONF_JSON[f"{row.did}|{row.config_id}"].get("feature_preprocessor:fast_ica:whiten")) == "False":
            return "orthogonal rotation"
    return t

def invariant_to_v2(clf, cd):
    inv = set(invariant_to(clf, cd))
    if clf in ("lda", "qda"):
        inv.add("orthogonal rotation")
    return inv

MEAS["type_v2"] = MEAS.apply(fp_type_v2, axis=1)
FPTYPE2 = {(r.did, r.fp_config_id): r.type_v2 for r in MEAS.itertuples()}
print("fast_ica configurations reclassified:")
print(MEAS[MEAS.preprocessor == "fast_ica"].groupby(["type", "type_v2"]).size().to_string())

rows = []
for did in CFG.dataset_ids:
    meta = add_ids(pd.read_csv(D2_DIR / f"D_{did}_index.csv"), did)
    Dtv = np.load(D2_DIR / f"D_{did}_tv.npy").astype(np.float64)
    ref = {(r.clf_cfg, r.balancing): i for i, r in meta[meta.preprocessor == "no_preprocessing"].iterrows()}
    for i, r in meta[meta.preprocessor != "no_preprocessing"].iterrows():
        j = ref.get((r.clf_cfg, r.balancing))
        if j is None:
            continue
        if r.preprocessor in NONLIN:
            t1 = t2 = "nonlinear expansion"
        else:
            t1, t2 = FPTYPE.get((did, r.pp_cfg)), FPTYPE2.get((did, r.pp_cfg))
            if t1 is None:
                continue
        cd = CONF_JSON[f"{did}|{r.config_id}"]
        d0 = d0_clf.get((did, r.classifier), DELTA0_2.loc[did, "tv"])
        rows.append(dict(did=did, classifier=r.classifier, group=learner_group(r.classifier),
                         preprocessor=r.preprocessor, type_v1=t1, type_v2=t2,
                         pred_v1="invariant" if t1 in invariant_to(r.classifier, cd) else "changes",
                         pred_v2="invariant" if t2 in invariant_to_v2(r.classifier, cd) else "changes",
                         in_delta0=Dtv[i, j] / d0 if d0 > 0 else np.nan, row=i))
P11 = pd.DataFrame(rows)
P11.to_csv(DIRS["metrics"] / "item11_paired_invariance_v1_v2.csv", index=False)

def test(df, pred_col, label):
    print(f"\n{label}")
    for did in CFG.dataset_ids:
        d = df[df.did == did]
        a, b = d[d[pred_col] == "invariant"].in_delta0.dropna(), d[d[pred_col] == "changes"].in_delta0.dropna()
        nt = d[(d[pred_col] == "invariant") & (d[pred_col.replace("pred", "type")] != "identity")].in_delta0.dropna()
        p = mannwhitneyu(a, b, alternative="less").pvalue
        print(f"  {did:>5}: invariant median {a.median():.2f} (n={len(a)}; non-identity n={len(nt)}, "
              f"median {nt.median() if len(nt) else float('nan'):.2f}) | changes median {b.median():.2f} "
              f"(n={len(b)}) | p = {p:.1e}")

test(P11, "pred_v1", "PRE-REGISTERED rules (document table as first implemented):")
test(P11, "pred_v2", "CORRECTED rules (fast_ica whiten=False = rotation; LDA/QDA rotation-invariant):")

flip = P11[P11.pred_v1 != P11.pred_v2]
print(f"\ncells whose prediction changed: {len(flip)}; their median distance: "
      f"{flip.in_delta0.median():.2f} delta_0 (all moved changes -> invariant)")
print(flip.groupby(["group", "type_v2"]).in_delta0.agg(["size", "median"]).round(2).to_string())

# ---- heatmap with corrected types --------------------------------------------------
obs2 = P11.pivot_table(index="group", columns="type_v2", values="in_delta0", aggfunc="median")
pred2 = P11.pivot_table(index="group", columns="type_v2", values="pred_v2",
                        aggfunc=lambda s: f"I {100 * (s == 'invariant').mean():.0f}%")
cols2 = [t for t in TYPES if t in obs2.columns]
print("\nObserved median (delta_0 units), corrected transform types:")
print(obs2[cols2].round(2).to_string())
print("\nShare predicted invariant (corrected rules):")
print(pred2.reindex(columns=cols2).to_string())

fig, ax = plt.subplots(figsize=(1.9 * len(cols2) + 2, 0.6 * len(obs2) + 1.5))
im = ax.imshow(obs2[cols2].to_numpy(dtype=float), cmap="viridis", aspect="auto")
ax.set_xticks(range(len(cols2))); ax.set_xticklabels(cols2, rotation=25, ha="right")
ax.set_yticks(range(len(obs2))); ax.set_yticklabels(obs2.index)
for y_, g in enumerate(obs2.index):
    for x_, t in enumerate(cols2):
        v = obs2.loc[g, t]
        if pd.notna(v):
            ax.text(x_, y_, f"{v:.1f}\n{pred2.loc[g, t]}", ha="center", va="center", fontsize=7, color="white")
fig.colorbar(im, ax=ax, label="median distance to no_preprocessing (delta_0 units)")
ax.set_title("Item 11 (corrected rules): observed effect vs predicted invariance")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(DIRS["figures"] / f"item11_invariance_heatmap_corrected.{ext}", dpi=200)
print(f"\nfigures saved to {DIRS['figures']} (pre-registered and corrected versions both kept)")

fast_ica configurations reclassified:
type               type_v2            
invertible linear  invertible linear       9
                   orthogonal rotation    14

PRE-REGISTERED rules (document table as first implemented):
     37: invariant median 0.07 (n=180; non-identity n=0, median nan) | changes median 5.29 (n=1270) | p = 9.0e-59
     31: invariant median 0.00 (n=120; non-identity n=0, median nan) | changes median 3.47 (n=1305) | p = 5.2e-61
     44: invariant median 0.02 (n=184; non-identity n=4, median 0.00) | changes median 3.34 (n=1228) | p = 7.1e-71
   4534: invariant median 0.00 (n=150; non-identity n=0, median nan) | changes median 5.26 (n=1259) | p = 2.9e-73
   1461: invariant median 0.00 (n=146; non-identity n=2, median 0.00) | changes median 4.14 (n=1050) | p = 1.6e-66

CORRECTED rules (fast_ica whiten=False = rotation; LDA/QDA rotation-invariant):
     37: invariant median 0.00 (n=226; non-identity n=46, median 0.00) | changes median 5.45 (n=1224) | p = 7.7e-81
   

In [30]:
# =============================================================================
# CELL 18 — Final results package: robustness (bootstrap CIs, leave-one-
#           preprocessor-out), remaining deliverable figures, FINAL_SUMMARY.md
#           (re-runnable: saved bootstrap / leave-one-out results are reused)
# =============================================================================
N_BOOT = 200
BOOT_CSV = DIRS["metrics"] / "robust_bootstrap_shares.csv"
LOO_CSV = DIRS["metrics"] / "robust_leave_one_preprocessor_out.csv"

# ---------------------------------------------------------------------------
# 1. Bootstrap 95% CIs for the item-9 Shapley shares (all pipelines, TV)
# ---------------------------------------------------------------------------
def _boot(did, b):
    meta = pd.read_csv(D2_DIR / f"D_{did}_index.csv")
    D = np.load(D2_DIR / f"D_{did}_tv.npy", mmap_mode="r")
    idx = np.random.default_rng(derive_seed("boot", did, b)).integers(0, len(meta), len(meta))
    sh = shapley_shares(np.asarray(D[np.ix_(idx, idx)], dtype=np.float64), meta.iloc[idx].reset_index(drop=True))
    return dict(did=did, b=b, **{c: sh[f"phi_{c}"] for c in MAIN})

if BOOT_CSV.exists() and len(pd.read_csv(BOOT_CSV)) == N_BOOT * len(CFG.dataset_ids):
    BOOT = pd.read_csv(BOOT_CSV)
    print(f"bootstrap: loaded {len(BOOT)} saved resamples")
else:
    fresh_pool()
    t0 = time.time()
    BOOT = pd.DataFrame(Parallel(n_jobs=min(os.cpu_count() - 1, 200))(
        delayed(_boot)(did, b) for did in CFG.dataset_ids for b in range(N_BOOT)))
    BOOT["pp_minus_clf"] = BOOT.preprocessor - BOOT.classifier
    BOOT.to_csv(BOOT_CSV, index=False)
    print(f"bootstrap: {len(BOOT)} resamples in {(time.time() - t0) / 60:.1f} min")
if "pp_minus_clf" not in BOOT:
    BOOT["pp_minus_clf"] = BOOT.preprocessor - BOOT.classifier

CI = BOOT.groupby("did").agg(**{f"{c}_{q}": (c, (lambda x, q=q: x.quantile(q)))
                                for c in MAIN + ["pp_minus_clf"] for q in (0.025, 0.975)})
CI["P(preprocessor > classifier)"] = BOOT.groupby("did").pp_minus_clf.apply(lambda x: (x > 0).mean())
print(f"\nBootstrap ({N_BOOT} resamples per dataset): 95% CIs of Shapley shares, TV, all pipelines")
print(CI.round(3).to_string())

# ---------------------------------------------------------------------------
# 2. Leave-one-preprocessor-out
# ---------------------------------------------------------------------------
if LOO_CSV.exists():
    LOO = pd.read_csv(LOO_CSV)
    print(f"\nleave-one-preprocessor-out: loaded {len(LOO)} saved fits")
else:
    loo = []
    for did in CFG.dataset_ids:
        meta = pd.read_csv(D2_DIR / f"D_{did}_index.csv")
        D = np.load(D2_DIR / f"D_{did}_tv.npy").astype(np.float64)
        for p in sorted(meta.preprocessor.unique()):
            keep = np.where(meta.preprocessor != p)[0]
            sh = shapley_shares(D[np.ix_(keep, keep)], meta.iloc[keep].reset_index(drop=True))
            loo.append(dict(did=did, dropped=p, **{c: sh[f"phi_{c}"] for c in MAIN},
                            largest=max(MAIN, key=lambda c: sh[f"phi_{c}"])))
    LOO = pd.DataFrame(loo)
    LOO.to_csv(LOO_CSV, index=False)
print("\nLeave-one-preprocessor-out: largest component when each preprocessor is removed")
print(LOO.pivot(index="dropped", columns="did", values="largest").to_string())
print("\nsmallest preprocessor share across all removals:", LOO.groupby("did").preprocessor.min().round(3).to_dict())
print("largest classifier share across all removals:  ", LOO.groupby("did").classifier.max().round(3).to_dict())

# ---------------------------------------------------------------------------
# 3. Remaining deliverable figures
# ---------------------------------------------------------------------------
names = [DATA[d].name for d in CFG.dataset_ids]
eff_tab = EFF.set_index(["did", "component"]).in_delta0_units.unstack().loc[list(CFG.dataset_ids)]
fig, ax = plt.subplots(figsize=(8, 4))
w = 0.27
for k, c in enumerate(MAIN):
    ax.bar(np.arange(len(names)) + (k - 1) * w, eff_tab[c].to_numpy(), w, label=c, color=colors[c])
ax.axhline(1, ls="--", c="k", lw=1)
ax.text(len(names) - 0.5, 1.05, "noise floor (1 δ₀)", ha="right", fontsize=8)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names)
ax.set_ylabel("paired effect (δ₀ units)"); ax.legend(frameon=False)
ax.set_title("Effect of changing only one component, in noise-floor units (items 4, 8)")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(DIRS["figures"] / f"item4_8_effect_sizes_delta0.{ext}", dpi=200)
plt.close(fig)

fh = cf.reset_index()
fig, ax = plt.subplots(figsize=(9, 4))
labels = [f"{DATA[row['did']].name}\n{row['set']}" for _, row in fh.iterrows()]
ax.bar(range(len(fh)), fh["classifier family"], label="family", color="#3b6ea8")
ax.bar(range(len(fh)), fh["classifier hyperparameters (within family)"], bottom=fh["classifier family"],
       label="hyperparameters within family", color="#9fc5e8")
ax.set_xticks(range(len(fh))); ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("R² (sequential, classifier first)"); ax.legend(frameon=False)
ax.set_title("Item 10: classifier dispersion split into family vs hyperparameters")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(DIRS["figures"] / f"item10_family_vs_hyperparameters.{ext}", dpi=200)
plt.close(fig)

# ---------------------------------------------------------------------------
# 4. FINAL_SUMMARY.md (every number computed from the results)
# ---------------------------------------------------------------------------
un = FINAL.xs("all pipelines", level="set")
v2 = {did: (P11[(P11.did == did) & (P11.pred_v2 == "invariant") & (P11.type_v2 != "identity")].in_delta0.median(),
            P11[(P11.did == did) & (P11.pred_v2 == "changes")].in_delta0.median()) for did in CFG.dataset_ids}
noise_sets = FINAL[(FINAL.index.get_level_values("set") != "all pipelines") & (~FINAL.resolvable)]
noise_lo, noise_hi = noise_sets["D_bar/delta0"].min(), noise_sets["D_bar/delta0"].max()
seed_diff = CHK.pivot(index="did", columns="version", values=MAIN)
seed_maxdiff = max((seed_diff[c]["seed 0 only"] - seed_diff[c]["3 seeds"]).abs().max() for c in MAIN)
seed_same = CHK.groupby("did").largest.nunique().eq(1).all()
dist_win = w_ = SHAP9.copy()
dist_win["largest"] = dist_win.apply(winner, axis=1)
clf_lead = dist_win[dist_win.largest == "classifier"][["did", "set", "distance"]]
clf_lead_txt = "; ".join(f"{DATA[r['did']].name} {r['set']} ({r['distance']})" for _, r in clf_lead.iterrows())

lines = ["# Final results: which AutoML pipeline component drives SHAP explanation variation", "",
         "## Phase 1 — re-measurement of the original pipelines (items 1-5)",
         "- Classifier dominance as originally reported (0.18 vs 0.0075) does not survive matched measurement.",
         "- On credit-g, spambase, Phishing and bank-marketing the original KernelExplainer values are at the noise floor "
         "(D_bar/delta_0 = " + f"{VERDICT.loc[[31, 44, 4534, 1461], 'D_bar_over_delta0'].min():.2f}-"
         f"{VERDICT.loc[[31, 44, 4534, 1461], 'D_bar_over_delta0'].max():.2f}); on diabetes (exact Kernel SHAP) one-way R^2: "
         f"classifier {VERDICT.loc[37, 'R2_classifier']:.2f}, rescaling {VERDICT.loc[37, 'R2_cfg_rescaling']:.2f}, "
         f"preprocessor {VERDICT.loc[37, 'R2_preprocessor']:.2f}.",
         "- The original code mislabels features on credit-g and bank-marketing (verified by recomputation; corrected).", "",
         "## Phase 2 — balanced factorial (items 6-9), TV, all pipelines",
         "| dataset | classifier share | preprocessor share | balancing share | interaction (added R^2) | P(preprocessor > classifier), bootstrap |",
         "|---|---|---|---|---|---|"]
for did in CFG.dataset_ids:
    r = un.loc[did]
    lines.append(f"| {DATA[did].name} | {r['classifier %']:.0f}% | {r['preprocessor %']:.0f}% | {r['balancing %']:.1f}% | "
                 f"{r['interaction']:.3f} | {CI.loc[did, 'P(preprocessor > classifier)']:.2f} |")
lines += [
    "",
    "Paired effects (changing only one component), delta_0 units: " +
    "; ".join(f"{DATA[d].name}: " + ", ".join(f"{c} {eff_tab.loc[d, c]:.1f}" for c in MAIN) for d in CFG.dataset_ids),
    "",
    "Rashomon sets: diabetes and credit-g — preprocessor largest; classifier share rises with the accuracy constraint (" +
    "; ".join(f"{DATA[d].name}: {FINAL.loc[(d, 'all pipelines'), 'classifier %']:.0f}% -> "
              f"{FINAL.loc[(d, 'R_2se'), 'classifier %']:.0f}% -> {FINAL.loc[(d, 'R_1se'), 'classifier %']:.0f}%"
              for d in (37, 31)) +
    f"). spambase, Phishing, bank-marketing — near-optimal explanations agree within noise "
    f"(D_bar/delta_0 = {noise_lo:.2f}-{noise_hi:.2f}).",
    "",
    "Whole components (hyperparameters on both sides), preprocessor share: " +
    ", ".join(f"{DATA[r['did']].name} {r['preprocessor %']:.0f}%" for _, r in COMP[COMP.set == "all pipelines"].iterrows()),
    "",
    "## Phase 3 — mechanism (items 10-11)",
    "- Item 10: hyperparameters carry " +
    ", ".join(f"{DATA[i[0]].name} {i[1]} {v:.0f}%" for i, v in cf["hyperparameter % of classifier part"].items()) +
    " of the classifier part. Within-family dispersion is larger for flexible than for low-capacity learners on datasets " +
    ", ".join(DATA[d].name for d in grp.index if grp.loc[d, "flexible"] > grp.loc[d, "low-capacity"]) + ".",
    "- Item 11 (corrected rules): non-identity predicted-invariant cells, median distance " +
    ", ".join(f"{DATA[d].name} {v2[d][0]:.2f}" for d in CFG.dataset_ids) + " delta_0 vs predicted-changing " +
    ", ".join(f"{DATA[d].name} {v2[d][1]:.2f}" for d in CFG.dataset_ids) +
    " delta_0. The pre-registered rules are also significant (p <= 1e-59).",
    "",
    "## Robustness",
    f"- Bootstrap ({N_BOOT}x): P(preprocessor > classifier) = " +
    ", ".join(f"{DATA[d].name} {CI.loc[d, 'P(preprocessor > classifier)']:.2f}" for d in CFG.dataset_ids) + ".",
    f"- Leave-one-preprocessor-out: the preprocessor is the largest component in "
    f"{int((LOO.largest == 'preprocessor').sum())} of {len(LOO)} removals.",
    f"- Seed check: shares with seed 0 vs 3 seeds differ by at most {seed_maxdiff:.3f}; "
    f"same largest component on every dataset: {seed_same}.",
    "- Distance choice: cases where the classifier is the largest component: " + (clf_lead_txt or "none") + ".",
    "",
    "## Deviations, choices and corrections",
    "- Phase 1 reuses the original KernelExplainer SHAP (option C); deviations from item 1 in item1_deviations_reused_shap.csv.",
    "- Feature-label bug of the original code corrected for credit-g and bank-marketing.",
    "- LDA/QDA predict_proba kept as auto-sklearn outputs it (sigma(2P-1)); recorded, not corrected.",
    "- Item 5 not available on diabetes (no matching RF pipeline in the filtered set).",
    "- Phase-1 seed refits use the full training set (auto-sklearn run models used an internal subset).",
    "- Rescaling pinned to standardize (plus mean imputation, one-hot, no coalescence); item 11's affine and "
    "monotone columns therefore untested.",
    "- k = 5 configurations per cell, paired draws across cells; E = 200 held-out rows, B = 100 stratified training rows.",
    "- Full factorial explained at seed 0; seeds as replicates only for Rashomon sets and delta_0 (seed check above).",
    "- One explainer run (r = 0) for every pipeline; extra runs only on a 50-configuration subset (lever 3), "
    "restricted to configurations explained in <= 1 h.",
    "- CV: seed 0, SE of L(p*) from per-row out-of-fold losses; 1 h per CV fold, 6 h per explanation, 3/6 GB memory limits.",
    "- PERMANOVA implemented in Python (validated: SS identity, MANOVA identity, null p ~ 0.5); vegan not used.",
    "- RBO_ext with p = 0.9; resolvability threshold D_bar/delta_0 >= 1.2 (our choice).",
    "- Item 10 nested terms use exact configuration ids (correction of a first version that used draw indices).",
    "- Item 11 rule corrections (fast_ica whiten=False = rotation; LDA/QDA rotation-invariant) were data-triggered; "
    "both pre-registered and corrected results are reported.",
    "- auto-sklearn PCA always truncates (keep_variance <= 0.9999), so rotation is tested via fast_ica only.",
    "- Items 12-15 out of scope.",
]
(OUT / "FINAL_SUMMARY.md").write_text("\n".join(lines))
print(f"\nwrote {OUT / 'FINAL_SUMMARY.md'}")
print("figures:", sorted(p.name for p in DIRS["figures"].glob("*.png")))

bootstrap: loaded 1000 saved resamples

Bootstrap (200 resamples per dataset): 95% CIs of Shapley shares, TV, all pipelines
      classifier_0.025  classifier_0.975  preprocessor_0.025  preprocessor_0.975  balancing_0.025  balancing_0.975  pp_minus_clf_0.025  pp_minus_clf_0.975  P(preprocessor > classifier)
did                                                                                                                                                                                     
31               0.100             0.143               0.377               0.415            0.000            0.002               0.234               0.309                           1.0
37               0.044             0.080               0.434               0.492            0.001            0.005               0.359               0.444                           1.0
44               0.118             0.164               0.329               0.395            0.001            0.003               0.175  

In [32]:
# =============================================================================
# CELL 19 — Extension: PFI and LIME on the Phase-2 factorial (PILOT)
#   Same pipelines (refit with saved seed -> identical to the SHAP models),
#   same raw features end-to-end, shared randomness across pipelines.
#   PFI: log-loss increase, 5 repeats, <=2,000 fixed held-out rows, negatives clipped later.
#   LIME: LimeTabularExplainer defaults, same 200 rows E, global = mean |weight|.
#   Jobs are launched from THREADS (each job runs in its own subprocess), so no
#   joblib process pool is involved and the pool hang cannot occur.
# =============================================================================
import importlib

XAI_DIR = OUT / "xai"
XAI_DIR.mkdir(exist_ok=True)
XAI_MEM_MB, XAI_TIME_S = 6144, 6 * 3600
PFI_ROWS, PFI_REPEATS, LIME_SAMPLES = 2000, 5, 5000

XAI_SRC = r'''
# phase2_xai.py -- refit one Phase-2 pipeline and compute PFI and LIME.
# Run as: python -m phase2_xai <args.pkl>
import os, sys, pickle, time, warnings, resource
import numpy as np


def _vmsize():
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmSize:"):
                return int(line.split()[1]) * 1024


def _logloss(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def pfi(proba, X, y, n_repeats, seed):
    """Mean log-loss increase when one raw column is permuted; permutations shared
    across pipelines (seeded by feature and repeat only)."""
    base = _logloss(y, proba(X)[:, 1])
    imp = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        incr = []
        for r in range(n_repeats):
            perm = np.random.RandomState((seed + 1000003 * j + 7919 * r) % (2**31 - 1)).permutation(len(X))
            Xp = X.copy()
            Xp[:, j] = X[perm, j]
            incr.append(_logloss(y, proba(Xp)[:, 1]) - base)
        imp[j] = np.mean(incr)
    return imp


def lime_global(proba, X_train, XE, cat_idx, n_samples, seed):
    from lime.lime_tabular import LimeTabularExplainer
    d = X_train.shape[1]
    ex = LimeTabularExplainer(X_train, mode="classification", categorical_features=cat_idx,
                              feature_names=[f"f{j}" for j in range(d)],
                              discretize_continuous=True, random_state=seed)
    W = np.zeros((len(XE), d))
    for i, x in enumerate(XE):
        e = ex.explain_instance(x, proba, labels=(1,), num_features=d, num_samples=n_samples)
        for j, w in e.as_map()[1]:
            W[i, j] = w
    return np.abs(W).mean(0)


def job(a):
    warnings.filterwarnings("ignore")
    soft, hard = resource.getrlimit(resource.RLIMIT_AS)
    limit = _vmsize() + a["mem_limit_mb"] * 1024 ** 2
    if hard != resource.RLIM_INFINITY:
        limit = min(limit, hard)
    resource.setrlimit(resource.RLIMIT_AS, (limit, hard))
    out = dict(status="ok", err="")
    try:
        from autosklearn.pipeline.classification import SimpleClassificationPipeline
        t0 = time.time()
        pipe = SimpleClassificationPipeline(config=a["conf"], feat_type=a["feat_type"],
                                            dataset_properties=a["dp"], random_state=a["seed"])
        pipe.fit(a["X_train"], a["y_train"])
        out["fit_secs"] = time.time() - t0
        chunk = a["chunk"]

        def proba(X):
            X = np.asarray(X, dtype=np.float64)
            if len(X) <= chunk:
                return pipe.predict_proba(X)
            return np.vstack([pipe.predict_proba(X[i:i + chunk]) for i in range(0, len(X), chunk)])

        out["pE"] = proba(a["XE"])[:, 1]
        t1 = time.time()
        out["m_pfi"] = pfi(proba, a["X_pfi"], a["y_pfi"], a["n_repeats"], a["pfi_seed"])
        out["pfi_secs"] = time.time() - t1
        t2 = time.time()
        out["m_lime"] = lime_global(proba, a["X_train"], a["XE"], a["cat_idx"], a["lime_samples"], a["lime_seed"])
        out["lime_secs"] = time.time() - t2
    except MemoryError:
        out.update(status="memout")
    except Exception as e:
        out.update(status="error", err=f"{type(e).__name__}: {str(e)[:300]}")
    tmp = a["out_path"] + ".tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, a["out_path"])


def run_isolated(a, time_limit_s, workdir):
    import subprocess, tempfile
    fd, apath = tempfile.mkstemp(suffix=".pkl", dir=a["tmp_dir"])
    os.close(fd)
    with open(apath, "wb") as f:
        pickle.dump(a, f, protocol=4)
    t0 = time.time()
    status, err = None, ""
    try:
        r = subprocess.run([sys.executable, "-m", "phase2_xai", apath], cwd=workdir,
                           timeout=time_limit_s, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        if r.returncode != 0:
            status = "crashed"
            err = f"exit code {r.returncode}; " + r.stderr.decode(errors="ignore")[-300:]
    except subprocess.TimeoutExpired:
        status, err = "timeout", f"exceeded {time_limit_s}s"
    finally:
        os.remove(apath)
    if os.path.exists(a["out_path"]):
        return "done"
    np.savez(a["out_path"], status=status or "crashed", err=err, total_secs=time.time() - t0)
    return status or "crashed"


if __name__ == "__main__":
    with open(sys.argv[1], "rb") as f:
        job(pickle.load(f))
'''
(WORKDIR / "phase2_xai.py").write_text(XAI_SRC)
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
import phase2_xai
importlib.reload(phase2_xai)

# ---- fixed PFI evaluation rows per dataset: <= 2,000 held-out rows, stratified -------
PFI_IDX = {}
for did in CFG.dataset_ids:
    te = ROWS2[did]["test_idx"]
    y = DATA[did].y_event
    if len(te) <= PFI_ROWS:
        PFI_IDX[did] = np.sort(te)
    else:
        sub, _ = train_test_split(te, train_size=PFI_ROWS, stratify=y[te],
                                  random_state=derive_seed("pfi_rows", did))
        PFI_IDX[did] = np.sort(sub)
print("PFI evaluation rows per dataset:", {d: len(v) for d, v in PFI_IDX.items()}, flush=True)


def xai_path(did, cid, s):
    d = XAI_DIR / str(did)
    d.mkdir(exist_ok=True)
    return d / f"{cid}_s{s}.npz"


def xai_args(did, cid, s, out_path=None):
    rs, D = ROWS2[did], DATA[did]
    return dict(conf=rebuild_config(did, cid).get_dictionary(), feat_type=FEAT2[did], dp=DP2,
                X_train=D.X[rs["train_idx"]], y_train=D.y_event[rs["train_idx"]],
                seed=SEEDS2[f"{did}|{cid}"][s], XE=D.X[rs["E_idx"]],
                X_pfi=D.X[PFI_IDX[did]], y_pfi=D.y_event[PFI_IDX[did]],
                cat_idx=[int(j) for j in np.where(D.cat_mask)[0]],
                n_repeats=PFI_REPEATS, pfi_seed=derive_seed("pfi", did),
                lime_samples=LIME_SAMPLES, lime_seed=derive_seed("lime", did),
                chunk=CFG.pred_chunk, mem_limit_mb=XAI_MEM_MB,
                out_path=str(out_path or xai_path(did, cid, s)), tmp_dir=str(DIRS["tmp"]))


def run_threaded(jobs_with_paths, label):
    """jobs_with_paths: list of (did, cid, s, out_path or None). Threads only launch
    subprocesses and wait, so no process pool is involved."""
    if not jobs_with_paths:
        print(f"{label}: nothing to run", flush=True)
        return
    n_thr = min(workers_for(XAI_MEM_MB), len(jobs_with_paths))
    print(f"{label}: {len(jobs_with_paths)} jobs on {n_thr} threads (each job in its own process)", flush=True)
    t0 = time.time()
    Parallel(n_jobs=n_thr, backend="threading", verbose=5)(
        delayed(phase2_xai.run_isolated)(xai_args(did, cid, s, out_path=op), XAI_TIME_S, str(WORKDIR))
        for did, cid, s, op in jobs_with_paths)
    print(f"{label}: finished in {(time.time() - t0) / 60:.1f} min", flush=True)

# ---- pilot: one random configuration per classifier per dataset ------------------------
IDX2 = {did: pd.read_csv(D2_DIR / f"D_{did}_index.csv") for did in CFG.dataset_ids}
rng = np.random.default_rng(derive_seed("xai-pilot"))
pilot = []
for did in CFG.dataset_ids:
    for clf, g in IDX2[did].groupby("classifier"):
        pilot.append((did, g.config_id.iloc[rng.integers(len(g))], 0))

# one determinism re-run per dataset (first pilot job of each dataset), written to tmp
dups = [next(j for j in pilot if j[0] == did) for did in CFG.dataset_ids]
dup_paths = {j: DIRS["tmp"] / f"dup_{j[0]}_{j[1]}.npz" for j in dups}
for p in list(DIRS["tmp"].glob("*.pkl")) + list(dup_paths.values()):
    if p.exists():
        p.unlink()

work = [(did, cid, s, None) for did, cid, s in pilot if not xai_path(did, cid, s).exists()]
work += [(did, cid, s, dup_paths[(did, cid, s)]) for did, cid, s in dups]
run_threaded(work, "pilot")

# ---- checks ---------------------------------------------------------------------------
rows = []
for did, cid, s in pilot:
    z = np.load(xai_path(did, cid, s))
    st = str(z["status"])
    rec = dict(did=did, config_id=cid, classifier=IDX2[did].set_index("config_id").loc[cid, "classifier"], status=st)
    if st == "ok":
        saved = np.load(p2_path(did, cid, 0))
        rec.update(identical_to_shap_model=bool(np.array_equal(z["pE"], saved["pE"])),
                   fit_secs=float(z["fit_secs"]), pfi_secs=float(z["pfi_secs"]), lime_secs=float(z["lime_secs"]),
                   pfi_all_nonpositive=bool((z["m_pfi"] <= 0).all()))
    else:
        rec["err"] = str(z["err"])[:120]
    rows.append(rec)
PILOT = pd.DataFrame(rows)
print("\nstatus:", PILOT.status.value_counts().to_dict())
ok = PILOT[PILOT.status == "ok"].copy()
print(f"refit identical to the SHAP-explained model: {int(ok.identical_to_shap_model.sum())} of {len(ok)}")
print(f"PFI all <= 0 (degenerate) pipelines: {int(ok.pfi_all_nonpositive.sum())}")
for j, p in dup_paths.items():
    a, b = np.load(xai_path(*j)), np.load(p)
    if str(a["status"]) == "ok" and str(b["status"]) == "ok":
        same = all(np.array_equal(a[k], b[k]) for k in ("m_pfi", "m_lime", "pE"))
        print(f"determinism {j[0]}: identical PFI, LIME and predictions on re-run -> {same}")
    else:
        print(f"determinism {j[0]}: re-run not comparable (status {a['status']} / {b['status']})")
if (PILOT.status != "ok").any():
    print(PILOT[PILOT.status != "ok"].to_string())

ok["total"] = ok.fit_secs + ok.pfi_secs + ok.lime_secs
print("\nmedian seconds per job (fit + PFI + LIME), classifier x dataset:")
print(ok.pivot_table(index="classifier", columns="did", values="total", aggfunc="median").round(0).to_string())

# ---- projection for the full run -----------------------------------------------------------
n_full = sum(len(IDX2[d]) for d in CFG.dataset_ids) + 2 * len(P2)
cpu_h = 0.0
for did in CFG.dataset_ids:
    med = ok[ok.did == did].groupby("classifier").total.median()
    cnt = IDX2[did].classifier.value_counts().add(
        2 * P2[P2.did == did].classifier.value_counts(), fill_value=0)
    cpu_h += float((cnt * med.reindex(cnt.index).fillna(med.median())).sum()) / 3600
nw = workers_for(XAI_MEM_MB)
print(f"\nfull run: ~{n_full} jobs, ~{cpu_h:.0f} CPU-hours -> roughly {cpu_h / nw:.1f} h on {nw} parallel jobs "
      f"(plus a tail of slow jobs)")

PFI evaluation rows per dataset: {37: 231, 31: 300, 44: 1381, 4534: 2000, 1461: 2000}
pilot: 85 jobs on 85 threads (each job in its own process)


[Parallel(n_jobs=85)]: Using backend ThreadingBackend with 85 concurrent workers.
[Parallel(n_jobs=85)]: Done   6 out of  85 | elapsed:   17.8s remaining:  3.9min
[Parallel(n_jobs=85)]: Done  24 out of  85 | elapsed:  3.2min remaining:  8.1min
[Parallel(n_jobs=85)]: Done  42 out of  85 | elapsed:  4.0min remaining:  4.1min
[Parallel(n_jobs=85)]: Done  60 out of  85 | elapsed:  4.4min remaining:  1.8min
[Parallel(n_jobs=85)]: Done  78 out of  85 | elapsed:  8.3min remaining:   44.9s


pilot: finished in 15.5 min

status: {'ok': 80}
refit identical to the SHAP-explained model: 80 of 80
PFI all <= 0 (degenerate) pipelines: 0
determinism 37: identical PFI, LIME and predictions on re-run -> True
determinism 31: identical PFI, LIME and predictions on re-run -> True
determinism 44: identical PFI, LIME and predictions on re-run -> True
determinism 4534: identical PFI, LIME and predictions on re-run -> True
determinism 1461: identical PFI, LIME and predictions on re-run -> True

median seconds per job (fit + PFI + LIME), classifier x dataset:
did                   31     37     44     1461   4534
classifier                                            
adaboost             245.0  272.0  479.0  236.0   76.0
bernoulli_nb         198.0  264.0  927.0  183.0   10.0
decision_tree        196.0  254.0  446.0  179.0    7.0
extra_trees          236.0  280.0  553.0  276.0   79.0
gaussian_nb          199.0  251.0  454.0  177.0   10.0
gradient_boosting    204.0  263.0  532.0  273.0   10.0

[Parallel(n_jobs=85)]: Done  85 out of  85 | elapsed: 15.5min finished


In [33]:
# =============================================================================
# CELL 20 — Extension: full PFI + LIME run on the Phase-2 factorial
#   seed 0 for every explained configuration (decomposition)
#   seeds 1-2 for the configurations with SHAP seed replicates (delta_0 per method)
#   Threaded launcher (each job in its own process), slowest first, resumable.
# =============================================================================
med_cost = ok.groupby(["did", "classifier"]).total.median()            # from the Cell-19 pilot

JOBS_X = []
for did in CFG.dataset_ids:
    for cid, clf in IDX2[did][["config_id", "classifier"]].itertuples(index=False):
        JOBS_X.append((did, cid, 0, clf))
for r_ in P2.itertuples():
    for s in (1, 2):
        JOBS_X.append((r_.did, r_.config_id, s, r_.classifier))

todo = [j for j in JOBS_X if not xai_path(j[0], j[1], j[2]).exists()]
todo.sort(key=lambda j: -float(med_cost.get((j[0], j[3]), med_cost.median())))
print(f"{len(JOBS_X)} PFI+LIME jobs in total; {len(JOBS_X) - len(todo)} already done; "
      f"{len(todo)} to run", flush=True)

for p in DIRS["tmp"].glob("*.pkl"):
    p.unlink()
run_threaded([(did, cid, s, None) for did, cid, s, _ in todo], "full PFI+LIME")

# ---- status and checks ---------------------------------------------------------------------
rows = []
for did, cid, s, clf in JOBS_X:
    z = np.load(xai_path(did, cid, s))
    st = str(z["status"])
    rec = dict(did=did, config_id=cid, seed=s, classifier=clf, status=st)
    if st == "ok" and s == 0:
        rec["identical_to_shap_model"] = bool(np.array_equal(z["pE"], np.load(p2_path(did, cid, 0))["pE"]))
    if st != "ok":
        rec["err"] = str(z["err"])[:120]
    rows.append(rec)
XST = pd.DataFrame(rows)
XST.to_csv(DIRS["metrics"] / "xai_status.csv", index=False)

print("\nstatus per dataset and seed:")
print(XST.groupby(["did", "seed", "status"]).size().unstack(fill_value=0).to_string())
s0 = XST[(XST.seed == 0) & (XST.status == "ok")]
print(f"\nseed-0 refits identical to the SHAP-explained models: "
      f"{int(s0.identical_to_shap_model.sum())} of {len(s0)}")
bad = XST[XST.status != "ok"]
if len(bad):
    print("\nnot completed, by classifier:")
    print(bad.groupby(["did", "classifier", "status"]).size().to_string())
    print(bad.drop_duplicates("status")[["did", "classifier", "status", "err"]].to_string())

10298 PFI+LIME jobs in total; 80 already done; 10218 to run
full PFI+LIME: 10218 jobs on 173 threads (each job in its own process)


[Parallel(n_jobs=173)]: Using backend ThreadingBackend with 173 concurrent workers.
[Parallel(n_jobs=173)]: Done 104 tasks      | elapsed: 10.6min
[Parallel(n_jobs=173)]: Done 302 tasks      | elapsed: 27.0min
[Parallel(n_jobs=173)]: Done 536 tasks      | elapsed: 44.1min
[Parallel(n_jobs=173)]: Done 806 tasks      | elapsed: 62.4min
[Parallel(n_jobs=173)]: Done 1112 tasks      | elapsed: 82.7min
[Parallel(n_jobs=173)]: Done 1454 tasks      | elapsed: 105.6min
[Parallel(n_jobs=173)]: Done 1832 tasks      | elapsed: 130.6min
[Parallel(n_jobs=173)]: Done 2246 tasks      | elapsed: 152.6min
[Parallel(n_jobs=173)]: Done 2696 tasks      | elapsed: 169.6min
[Parallel(n_jobs=173)]: Done 3182 tasks      | elapsed: 187.9min
[Parallel(n_jobs=173)]: Done 3704 tasks      | elapsed: 206.4min
[Parallel(n_jobs=173)]: Done 4262 tasks      | elapsed: 223.8min
[Parallel(n_jobs=173)]: Done 4856 tasks      | elapsed: 242.2min
[Parallel(n_jobs=173)]: Done 5486 tasks      | elapsed: 262.4min
[Parallel(n_job

full PFI+LIME: finished in 435.1 min


[Parallel(n_jobs=173)]: Done 10218 out of 10218 | elapsed: 435.1min finished



status per dataset and seed:
status       ok
did  seed      
31   0     1577
     1      340
     2      340
37   0     1602
     1      357
     2      357
44   0     1562
     1      212
     2      212
1461 0     1354
     1      202
     2      202
4534 0     1561
     1      210
     2      210

seed-0 refits identical to the SHAP-explained models: 7656 of 7656


In [34]:
# =============================================================================
# CELL 21 — Extension analysis: is preprocessor dominance explainer-agnostic?
#   PFI and LIME analysed exactly like SHAP: shares, delta_0, D_bar, Shapley shares
#   (all pipelines + Rashomon), paired effects, invariance test (corrected rules),
#   bootstrap, and agreement between methods.
# =============================================================================
from scipy.stats import mannwhitneyu, spearmanr

needed = ["IDX2", "xai_path", "P2", "add_ids", "FPTYPE2", "invariant_to_v2", "NONLIN", "CONF_JSON",
          "learner_group", "FINAL", "CI", "colors", "DELTA0_2", "D2_DIR"]
missing = [v for v in needed if v not in globals()]
assert not missing, f"kernel lost these variables (re-run the cells that define them): {missing}"

METHODS = {"PFI": "m_pfi", "LIME": "m_lime"}
prep = lambda key, v: np.clip(np.asarray(v, float), 0, None) if key == "m_pfi" else np.asarray(v, float)
N_BOOT_X = 200
XA = {}                     # (method, did) -> results
T0 = time.time()

for did in CFG.dataset_ids:
    meta_all = IDX2[did].reset_index(drop=True)
    Z0 = [np.load(xai_path(did, cid, 0)) for cid in meta_all.config_id]
    p2 = P2[P2.did == did]
    Zs = {cid: [np.load(xai_path(did, cid, s)) for s in range(CFG.n_train_seeds)] for cid in p2.config_id}
    D_shap = np.load(D2_DIR / f"D_{did}_tv.npy").astype(np.float64)

    for meth, key in METHODS.items():
        Nall = [normalise(prep(key, z[key])) for z in Z0]
        keep = np.array([not x["degenerate"] for x in Nall])
        mk = meta_all[keep].reset_index(drop=True)
        S = np.vstack([Nall[i]["shares"] for i in np.where(keep)[0]])
        R = np.vstack([Nall[i]["ranks"] for i in np.where(keep)[0]])
        D = pairwise_D(S, R)

        # ---- delta_0 from seed replicates (shared randomness, as for SHAP) ----------
        d0rows, seedN = [], {}
        for r_ in p2.itertuples():
            Ns = [normalise(prep(key, z[key])) for z in Zs[r_.config_id]]
            if any(x["degenerate"] for x in Ns):
                continue
            seedN[r_.config_id] = Ns
            Dd = pairwise_D(np.vstack([x["shares"] for x in Ns]), np.vstack([x["ranks"] for x in Ns]))
            iu = np.triu_indices(len(Ns), 1)
            d0rows.append(dict(classifier=r_.classifier, **{k: Dd[k][iu].mean() for k in DIST_NAMES}))
        d0t = pd.DataFrame(d0rows)
        d0 = d0t[DIST_NAMES].mean()
        d0_clf = d0t.groupby("classifier").tv.mean()

        res = dict(n=len(mk), degenerate=int((~keep).sum()), delta0=d0,
                   Dbar_over_d0=mean_pairwise(D["tv"]) / d0["tv"])

        # ---- Shapley shares, all pipelines, every distance ---------------------------
        res["shares"] = {k: shapley_shares(D[k], mk) for k in DIST_NAMES}

        # ---- Rashomon sets (model level, 3 seeds as replicates) ----------------------
        res["rashomon"] = {}
        for col, sname in (("in_R_2se", "R_2se"), ("in_R_1se", "R_1se")):
            mem = p2[p2[col] & p2.config_id.isin(seedN)]
            if len(mem) < 2:
                continue
            Ss, Rs, mm = [], [], []
            for r_ in mem.itertuples():
                for x in seedN[r_.config_id]:
                    Ss.append(x["shares"]); Rs.append(x["ranks"])
                    mm.append(dict(classifier=r_.classifier, preprocessor=r_.preprocessor, balancing=r_.balancing))
            Dr = pairwise_D(np.vstack(Ss), np.vstack(Rs))["tv"]
            mm = pd.DataFrame(mm)
            res["rashomon"][sname] = dict(n=len(mm), n_classifiers=mm.classifier.nunique(),
                                          Dbar_over_d0=mean_pairwise(Dr) / d0["tv"],
                                          shares=shapley_shares(Dr, mm))

        # ---- paired effects in delta_0 units (TV) ---------------------------------------
        eff = {}
        for c in MAIN:
            keys = [f for f in MAIN if f != c] + ["pair_idx"]
            dd = []
            for _, g in mk.groupby(keys):
                ix = g.index.to_numpy()
                if len(ix) > 1:
                    dd.extend(D["tv"][np.ix_(ix, ix)][np.triu_indices(len(ix), 1)])
            eff[c] = np.mean(dd) / d0["tv"]
        res["paired"] = eff

        # ---- invariance test (corrected rules) --------------------------------------------
        mki = add_ids(mk, did)
        ref = {(r_.clf_cfg, r_.balancing): i for i, r_ in mki[mki.preprocessor == "no_preprocessing"].iterrows()}
        inv = []
        for i, r_ in mki[mki.preprocessor != "no_preprocessing"].iterrows():
            j = ref.get((r_.clf_cfg, r_.balancing))
            if j is None:
                continue
            t = "nonlinear expansion" if r_.preprocessor in NONLIN else FPTYPE2.get((did, r_.pp_cfg))
            if t is None:
                continue
            dd0 = d0_clf.get(r_.classifier, d0["tv"])
            inv.append(dict(type=t, pred="invariant" if t in invariant_to_v2(r_.classifier, CONF_JSON[f"{did}|{r_.config_id}"])
                            else "changes", d=D["tv"][i, j] / dd0 if dd0 > 0 else np.nan))
        inv = pd.DataFrame(inv)
        a = inv[(inv.pred == "invariant") & (inv.type != "identity")].d.dropna()
        b = inv[inv.pred == "changes"].d.dropna()
        res["invariance"] = dict(nonidentity_invariant_median=a.median(), n_inv=len(a),
                                 changes_median=b.median(), n_chg=len(b),
                                 p=mannwhitneyu(a, b, alternative="less").pvalue if len(a) and len(b) else np.nan)

        # ---- bootstrap P(preprocessor > classifier), TV -----------------------------------
        def _bx(b_, Dtv=D["tv"], mk=mk):
            idx = np.random.default_rng(derive_seed("boot-x", meth, did, b_)).integers(0, len(mk), len(mk))
            sh = shapley_shares(Dtv[np.ix_(idx, idx)], mk.iloc[idx].reset_index(drop=True))
            return sh["phi_preprocessor"] - sh["phi_classifier"]
        diffs = np.array(Parallel(n_jobs=64, backend="threading")(delayed(_bx)(b_) for b_ in range(N_BOOT_X)))
        res["P_pp_gt_clf"] = float((diffs > 0).mean())
        res["gap_CI"] = (float(np.quantile(diffs, 0.025)), float(np.quantile(diffs, 0.975)))

        # ---- agreement with SHAP: correlation of distance matrices (same pipelines) -------
        iu = np.triu_indices(keep.sum(), 1)
        Ds = D_shap[np.ix_(keep, keep)][iu]
        res["corr_with_SHAP_D"] = spearmanr(Ds, D["tv"][iu]).correlation
        res["D_tv"], res["keep"] = D["tv"], keep
        XA[(meth, did)] = res
    print(f"  {did} {DATA[did].name} done ({(time.time() - T0) / 60:.1f} min)", flush=True)

# PFI vs LIME agreement on common pipelines
for did in CFG.dataset_ids:
    kp, kl = XA[("PFI", did)]["keep"], XA[("LIME", did)]["keep"]
    common = kp & kl
    ip, il = np.cumsum(kp)[common] - 1, np.cumsum(kl)[common] - 1
    iu = np.triu_indices(common.sum(), 1)
    XA[("PFI", did)]["corr_with_LIME_D"] = spearmanr(XA[("PFI", did)]["D_tv"][np.ix_(ip, ip)][iu],
                                                     XA[("LIME", did)]["D_tv"][np.ix_(il, il)][iu]).correlation

# =============================================================================
# REPORT
# =============================================================================
pd.set_option("display.width", 250)
rows = []
for did in CFG.dataset_ids:
    f = FINAL.loc[(did, "all pipelines")]
    rows.append(dict(did=did, method="SHAP", n=int(f.n), degenerate=0, D_bar_over_delta0=f["D_bar/delta0"],
                     **{f"{c} %": f[f"{c} %"] for c in MAIN}, interaction=f.interaction,
                     P_pp_gt_clf=CI.loc[did, "P(preprocessor > classifier)"]))
    for meth in METHODS:
        r = XA[(meth, did)]
        sh = r["shares"]["tv"]
        rows.append(dict(did=did, method=meth, n=r["n"], degenerate=r["degenerate"],
                         D_bar_over_delta0=r["Dbar_over_d0"],
                         **{f"{c} %": 100 * sh[f"phi_{c}"] / sh["R2_main"] for c in MAIN},
                         interaction=sh["interaction"], P_pp_gt_clf=r["P_pp_gt_clf"]))
CMP = pd.DataFrame(rows).set_index(["did", "method"])
CMP.to_csv(DIRS["metrics"] / "xai_comparison_all_pipelines.csv")
print("\nAll pipelines, TV: Shapley shares by explanation method")
print(CMP.round(3).to_string())

print("\nLargest component per distance (all pipelines):")
lg = pd.DataFrame({(meth, did): {k: max(MAIN, key=lambda c: XA[(meth, did)]["shares"][k][f"phi_{c}"])
                                 for k in DIST_NAMES} for meth in METHODS for did in CFG.dataset_ids}).T
print(lg.to_string())

print("\nNoise floor delta_0 (TV) per method:")
print(pd.DataFrame({meth: {did: XA[(meth, did)]["delta0"]["tv"] for did in CFG.dataset_ids} for meth in METHODS})
      .assign(SHAP=DELTA0_2["tv"]).round(4).to_string())

print("\nPaired effects (TV, delta_0 units): changing only one component")
print(pd.DataFrame({(meth, did): XA[(meth, did)]["paired"] for meth in METHODS for did in CFG.dataset_ids}).T.round(2).to_string())

print("\nRashomon sets (TV): shares and resolvability")
rr = []
for meth in METHODS:
    for did in CFG.dataset_ids:
        for sname, v in XA[(meth, did)]["rashomon"].items():
            sh = v["shares"]
            row = dict(method=meth, did=did, set=sname, n=v["n"], n_classifiers=v["n_classifiers"],
                       D_bar_over_delta0=v["Dbar_over_d0"])
            if sh:
                row.update({f"{c} %": 100 * sh[f"phi_{c}"] / sh["R2_main"] for c in MAIN})
            rr.append(row)
RR = pd.DataFrame(rr).set_index(["method", "did", "set"])
RR.to_csv(DIRS["metrics"] / "xai_comparison_rashomon.csv")
print(RR.round(2).to_string())

print("\nInvariance test (corrected rules): non-identity predicted-invariant vs predicted-changing (delta_0 units)")
print(pd.DataFrame({(meth, did): XA[(meth, did)]["invariance"] for meth in METHODS for did in CFG.dataset_ids})
      .T.round(3).to_string())

print("\nAgreement on which pipelines differ (Spearman correlation of TV distance matrices):")
print(pd.DataFrame({did: {"SHAP vs PFI": XA[("PFI", did)]["corr_with_SHAP_D"],
                          "SHAP vs LIME": XA[("LIME", did)]["corr_with_SHAP_D"],
                          "PFI vs LIME": XA[("PFI", did)]["corr_with_LIME_D"]} for did in CFG.dataset_ids}).round(3).to_string())

# ---- figure: shares by method --------------------------------------------------------------
fig, axes = plt.subplots(1, len(CFG.dataset_ids), figsize=(4 * len(CFG.dataset_ids), 4), sharey=True)
for ax, did in zip(axes, CFG.dataset_ids):
    for i, meth in enumerate(["SHAP", "PFI", "LIME"]):
        bottom = 0
        for c in MAIN:
            v = CMP.loc[(did, meth), f"{c} %"]
            ax.bar(i, v, bottom=bottom, color=colors[c], label=c if (did == CFG.dataset_ids[0] and i == 0) else None)
            bottom += v
    ax.set_xticks(range(3)); ax.set_xticklabels(["SHAP", "PFI", "LIME"])
    ax.set_title(f"{DATA[did].name} ({did})")
axes[0].set_ylabel("share of explained variance (%)")
fig.legend(loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
for ext in ("png", "pdf"):
    fig.savefig(DIRS["figures"] / f"extension_shares_SHAP_PFI_LIME.{ext}", dpi=200)
plt.close(fig)
print(f"\nfigure saved; total {(time.time() - T0) / 60:.1f} min")

  37 diabetes done (0.6 min)
  31 credit-g done (1.1 min)
  44 spambase done (1.7 min)
  4534 PhishingWebsites done (2.2 min)
  1461 bank-marketing done (2.7 min)

All pipelines, TV: Shapley shares by explanation method
                n  degenerate  D_bar_over_delta0  classifier %  preprocessor %  balancing %  interaction  P_pp_gt_clf
did  method                                                                                                          
37   SHAP    1602           0              6.370        10.631          89.020        0.349        0.059         1.00
     PFI     1595           7              4.606        31.453          68.427        0.120        0.119         1.00
     LIME    1602           0              5.582        14.515          85.026        0.459        0.070         1.00
31   SHAP    1577           0              4.426        22.306          77.549        0.144        0.090         1.00
     PFI     1567          10              3.412        39.365          

In [35]:
# =============================================================================
# CELL 22 — Manipulation analysis (item 15, worst-case reach; X-hacking hook)
#   Which single pipeline choice can change an explanation without losing accuracy?
#   Paired design: alternatives differ from the base pipeline in ONE component and
#   have CV accuracy >= base accuracy - 0.01 (the original study's tolerance).
#   Noise baseline: switching only the training seed of the same configuration.
# =============================================================================
ACC_TOL = 0.01
KS = (1, 3)
LEVERS = {"preprocessor": ["clf_cfg", "balancing"],     # same classifier config + balancing
          "classifier":   ["pp_cfg", "balancing"],      # same preprocessor config + balancing
          "balancing":    ["clf_cfg", "pp_cfg"]}        # same classifier + preprocessor config
ALL_METHODS = ["SHAP", "PFI", "LIME"]


def vectors(meth, did, cids, s):
    if meth == "SHAP":
        return [np.load(p2_path(did, c, s))["m"] for c in cids]
    key = METHODS[meth]
    return [prep(key, np.load(xai_path(did, c, s))[key]) for c in cids]


def ranks_of(vecs):
    N = [normalise(v) for v in vecs]
    ok = np.array([not x["degenerate"] for x in N])
    d = len(vecs[0])
    R = np.vstack([x["ranks"] if not x["degenerate"] else np.full(d, -1) for x in N])
    return R, ok


def summarise_events(hide, sw, drop, nalt):
    out = {f"hide@{k}": float(np.mean(hide[k])) if hide[k] else np.nan for k in KS}
    out.update({f"per_switch@{k}": float(np.mean(sw[k])) if sw[k] else np.nan for k in KS})
    out.update(max_rank_drop_top3=float(np.mean(drop)) if drop else np.nan,
               mean_alternatives=float(np.mean(nalt)) if nalt else np.nan, n_bases=len(nalt))
    return out


def lever_stats(R, ok, acc, meta, keys, base_mask):
    hide, sw, drop, nalt = {k: [] for k in KS}, {k: [] for k in KS}, [], []
    for _, g in meta.groupby(keys):
        ix = g.index.to_numpy()
        ix = ix[ok[ix]]
        for b in ix:
            if not base_mask[b]:
                continue
            alts = [a for a in ix if a != b and acc[a] >= acc[b] - ACC_TOL]
            if not alts:
                continue
            nalt.append(len(alts))
            Ra = R[alts]
            for k in KS:
                top = np.where(R[b] <= k)[0]
                out = Ra[:, top] > k
                hide[k].extend(out.any(0)); sw[k].extend(out.mean(0))
            top3 = np.where(R[b] <= 3)[0]
            drop.extend((Ra[:, top3] - R[b, top3]).max(0))
    return summarise_events(hide, sw, drop, nalt)


def noise_stats(meth, did, cids, base_ok):
    """Same measures when the only change is the training seed (seed 0 -> seeds 1, 2)."""
    hide, sw, drop, nalt = {k: [] for k in KS}, {k: [] for k in KS}, [], []
    for cid in cids:
        if not base_ok.get(cid, False):
            continue
        Rs, oks = ranks_of(vectors(meth, did, [cid] * 0 or [cid], 0) +
                           [vectors(meth, did, [cid], s)[0] for s in (1, 2)])
        if not oks.all():
            continue
        Rb, Ra = Rs[0], Rs[1:]
        nalt.append(len(Ra))
        for k in KS:
            top = np.where(Rb <= k)[0]
            out = Ra[:, top] > k
            hide[k].extend(out.any(0)); sw[k].extend(out.mean(0))
        top3 = np.where(Rb <= 3)[0]
        drop.extend((Ra[:, top3] - Rb[top3]).max(0))
    return summarise_events(hide, sw, drop, nalt)


rows = []
t0 = time.time()
for did in CFG.dataset_ids:
    meta = add_ids(IDX2[did].reset_index(drop=True), did)
    acc = meta.accuracy.to_numpy()
    p2 = P2[P2.did == did]
    for meth in ALL_METHODS:
        R, ok = ranks_of(vectors(meth, did, meta.config_id, 0))
        for bases, mask in (("all pipelines", np.ones(len(meta), bool)),
                            ("near-optimal (R_2se)", meta.in_R_2se.to_numpy())):
            for lever, keys in LEVERS.items():
                rows.append(dict(method=meth, did=did, bases=bases, lever=lever,
                                 **lever_stats(R, ok, acc, meta, keys, mask)))
            cids = p2.config_id if bases == "all pipelines" else p2[p2.in_R_2se].config_id
            base_ok = dict(zip(meta.config_id, ok))
            rows.append(dict(method=meth, did=did, bases=bases, lever="seed only (noise)",
                             **noise_stats(meth, did, cids, base_ok)))
    print(f"  {did} {DATA[did].name} done ({(time.time() - t0) / 60:.1f} min)", flush=True)

MAN = pd.DataFrame(rows)
MAN.to_csv(DIRS["metrics"] / "manipulation_analysis.csv", index=False)

pd.set_option("display.width", 250)
order = ["preprocessor", "classifier", "balancing", "seed only (noise)"]
for metric, label in (("per_switch@3", "share of single switches that push a top-3 feature out of the top 3"),
                      ("hide@3", "share of top-3 features that SOME allowed switch pushes out of the top 3"),
                      ("hide@1", "share of pipelines whose top feature SOME allowed switch replaces"),
                      ("max_rank_drop_top3", "worst-case rank drop of a top-3 feature (positions)")):
    print(f"\n{label}  [{metric}]")
    t = MAN.pivot_table(index=["bases", "method", "did"], columns="lever", values=metric)[order]
    print(t.round(3).to_string())

print("\nmean number of allowed alternatives per base pipeline:")
print(MAN.pivot_table(index=["bases", "did"], columns="lever", values="mean_alternatives")[order]
      .round(1).to_string())

# ---- figure: per-switch manipulation rate by lever (SHAP, all pipelines) -----------------------
t = MAN[(MAN.method == "SHAP") & (MAN.bases == "all pipelines")].pivot(index="did", columns="lever",
                                                                      values="per_switch@3").loc[list(CFG.dataset_ids)]
fig, ax = plt.subplots(figsize=(8, 4))
w = 0.2
lev_colors = {"preprocessor": colors["preprocessor"], "classifier": colors["classifier"],
              "balancing": colors["balancing"], "seed only (noise)": "lightgrey"}
for k, lev in enumerate(order):
    ax.bar(np.arange(len(t)) + (k - 1.5) * w, t[lev], w, label=lev, color=lev_colors[lev])
ax.set_xticks(range(len(t))); ax.set_xticklabels([DATA[d].name for d in t.index])
ax.set_ylabel("share of switches removing a top-3 feature")
ax.set_title("Which single choice changes the explanation without losing accuracy (SHAP)")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(DIRS["figures"] / f"manipulation_by_lever.{ext}", dpi=200)
plt.close(fig)
print(f"\nfigure saved; total {(time.time() - t0) / 60:.1f} min")

  37 diabetes done (0.1 min)
  31 credit-g done (0.1 min)
  44 spambase done (0.1 min)
  4534 PhishingWebsites done (0.2 min)
  1461 bank-marketing done (0.2 min)

share of single switches that push a top-3 feature out of the top 3  [per_switch@3]
lever                             preprocessor  classifier  balancing  seed only (noise)
bases                method did                                                         
all pipelines        LIME   31           0.458       0.437      0.200              0.155
                            37           0.263       0.217      0.102              0.079
                            44           0.570       0.626      0.195              0.204
                            1461         0.369       0.367      0.169              0.096
                            4534         0.148       0.115      0.036              0.040
                     PFI    31           0.420       0.440      0.226              0.183
                            37          

In [36]:
# =============================================================================
# CELL 23 — Append the extension (PFI, LIME) and the manipulation analysis to
#           FINAL_SUMMARY.md (all numbers computed from the results)
# =============================================================================
def rng_txt(s, fmt="{:.0f}"):
    return f"{fmt.format(s.min())}-{fmt.format(s.max())}"

ext = ["", "## Extension — other explanation methods (PFI, LIME) on the same factorial pipelines",
       "Same ~7,750 pipelines (all 7,656 seed-0 refits identical to the SHAP-explained models), same raw features, "
       "same decomposition. PFI = log-loss increase (5 repeats, <=2,000 held-out rows, shared permutations, negatives "
       "clipped); LIME = LimeTabularExplainer defaults on the same 200 rows E, global = mean |weight|.", "",
       "| dataset | method | classifier share | preprocessor share | balancing share | P(preprocessor > classifier) |",
       "|---|---|---|---|---|---|"]
for (did, meth), r in CMP.iterrows():
    ext.append(f"| {DATA[did].name} | {meth} | {r['classifier %']:.0f}% | {r['preprocessor %']:.0f}% | "
               f"{r['balancing %']:.1f}% | {r['P_pp_gt_clf']:.2f} |")
lime = CMP.xs("LIME", level="method"); pfi_ = CMP.xs("PFI", level="method")
ext += ["",
        f"- LIME: preprocessor largest on {int((lime['preprocessor %'] > lime['classifier %']).sum())} of 5 datasets "
        f"(preprocessor {rng_txt(lime['preprocessor %'])}%).",
        f"- PFI: preprocessor largest on {int((pfi_['preprocessor %'] > pfi_['classifier %']).sum())} of 5 datasets; "
        f"classifier share higher than under SHAP ({rng_txt(pfi_['classifier %'])}%), consistent with loss-based "
        "importance being sensitive to calibration, which the classifier controls.",
        "- Invariance test holds for both methods: non-identity predicted-invariant cells median " +
        ", ".join(f"{m} {rng_txt(pd.Series([XA[(m, d)]['invariance']['nonidentity_invariant_median'] for d in CFG.dataset_ids]), '{:.2f}')}"
                  for m in METHODS) + " delta_0 vs predicted-changing " +
        ", ".join(f"{m} {rng_txt(pd.Series([XA[(m, d)]['invariance']['changes_median'] for d in CFG.dataset_ids]), '{:.1f}')}"
                  for m in METHODS) + " delta_0.",
        "- Agreement on which pipelines differ (Spearman correlation of TV distance matrices): SHAP-LIME " +
        rng_txt(pd.Series([XA[("LIME", d)]["corr_with_SHAP_D"] for d in CFG.dataset_ids]), "{:.2f}") +
        ", SHAP-PFI " + rng_txt(pd.Series([XA[("PFI", d)]["corr_with_SHAP_D"] for d in CFG.dataset_ids]), "{:.2f}") +
        ", PFI-LIME " + rng_txt(pd.Series([XA[("PFI", d)]["corr_with_LIME_D"] for d in CFG.dataset_ids]), "{:.2f}") + "."]

M_all = MAN[MAN.bases == "all pipelines"]
M_opt = MAN[MAN.bases == "near-optimal (R_2se)"]
ps = lambda df, lev, col="per_switch@3": df[df.lever == lev][col]
opt_s = M_opt[M_opt.method == "SHAP"].pivot(index="did", columns="lever", values="per_switch@3")
clf_wins = int((opt_s["classifier"] > opt_s["preprocessor"]).sum())
ext += ["", "## Manipulation analysis (item 15; X-hacking)",
        "Alternatives differ from a base pipeline in one component and keep CV accuracy >= base - 0.01; "
        "noise baseline = retraining the same configuration with another seed.",
        f"- All pipelines, share of single switches pushing a top-3 feature out of the top 3 (all methods): "
        f"preprocessor {rng_txt(ps(M_all, 'preprocessor'), '{:.2f}')}, classifier {rng_txt(ps(M_all, 'classifier'), '{:.2f}')}, "
        f"balancing {rng_txt(ps(M_all, 'balancing'), '{:.2f}')}, seed only {rng_txt(ps(M_all, 'seed only (noise)'), '{:.2f}')}.",
        f"- Top feature replaced by some allowed switch: preprocessor {rng_txt(ps(M_all, 'preprocessor', 'hide@1'), '{:.2f}')}, "
        f"classifier {rng_txt(ps(M_all, 'classifier', 'hide@1'), '{:.2f}')}.",
        f"- Worst-case rank drop of a top-3 feature (positions): preprocessor {rng_txt(ps(M_all, 'preprocessor', 'max_rank_drop_top3'), '{:.1f}')}, "
        f"classifier {rng_txt(ps(M_all, 'classifier', 'max_rank_drop_top3'), '{:.1f}')}.",
        f"- Near-optimal pipelines (R_2se, SHAP): the classifier is the stronger per-switch lever on {clf_wins} of 5 datasets.",
        "- Summary: preprocessing dominates how much the overall explanation varies; the classifier more often decides "
        "which feature is on top, especially among near-optimal pipelines; balancing is about as strong as seed noise.",
        "", "## Additional choices (extension and manipulation analysis)",
        "- LIME 0.2.0.1 installed with --no-deps (scikit-image not needed for tabular LIME; environment unchanged).",
        "- PFI on log-loss with 5 repeats on <=2,000 held-out rows; negative importances clipped at 0; pipelines with "
        "all-zero importance excluded (PFI: 4-10 per dataset).",
        "- Manipulation analysis uses an accuracy tolerance of 0.01 (the original study's filter); per-switch rates "
        "correct for different numbers of alternatives; near-optimal sets on spambase, Phishing and bank-marketing are small."]

path = OUT / "FINAL_SUMMARY.md"
text = path.read_text()
marker = "\n## Extension — other explanation methods"
if marker in text:
    text = text[:text.index(marker)]
path.write_text(text.rstrip() + "\n" + "\n".join(ext) + "\n")
print(f"updated {path}")
print("figures:", sorted(p.name for p in DIRS["figures"].glob("*.png")))
print("\n".join(ext[-16:]))

updated /home/dbanerjee/example_for_debjeet/results/clean/FINAL_SUMMARY.md
figures: ['extension_shares_SHAP_PFI_LIME.png', 'item10_family_vs_hyperparameters.png', 'item11_invariance_heatmap.png', 'item11_invariance_heatmap_corrected.png', 'item4_8_effect_sizes_delta0.png', 'item7_9_shapley_shares_uncond_vs_rashomon.png', 'manipulation_by_lever.png']
- PFI: preprocessor largest on 4 of 5 datasets; classifier share higher than under SHAP (31-53%), consistent with loss-based importance being sensitive to calibration, which the classifier controls.
- Invariance test holds for both methods: non-identity predicted-invariant cells median PFI 0.00-0.00, LIME 0.00-0.00 delta_0 vs predicted-changing PFI 2.5-5.4, LIME 3.5-4.9 delta_0.
- Agreement on which pipelines differ (Spearman correlation of TV distance matrices): SHAP-LIME 0.84-0.95, SHAP-PFI 0.45-0.90, PFI-LIME 0.45-0.81.

## Manipulation analysis (item 15; X-hacking)
Alternatives differ from a base pipeline in one component and keep CV ac